In [1]:
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
import sys
import gc
import rasterio
import geopandas as gpd
import netCDF4 as nc
from matplotlib.colors import BoundaryNorm, ListedColormap
from rasterio.enums import Resampling
import numpy as np
import matplotlib.font_manager as fm
from rasterio.features import geometry_mask
from rasterio.mask import mask
import pandas as pd
from scipy.stats import pearsonr
import threading
from tqdm import tqdm
import xarray as xr
from rasterio import Affine
from rasterio.crs import CRS
from scipy.stats import linregress
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
import seaborn as sns
import statsmodels.api as sm
from scipy.signal import detrend
from statsmodels.tsa.stattools import adfuller
from osgeo import gdal
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from sklearn.preprocessing import MinMaxScaler
from matplotlib.cm import ScalarMappable
from scipy.stats import f
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
from scipy.ndimage import zoom
import os
from pyhdf.SD import SD, SDC
import numpy as np
import rioxarray
import h5py

# Set New Roman font
plt.rcParams['font.family'] = 'Times New Roman'
%matplotlib tk

导入所需的库


In [2]:
print('Import custom functions')
def generate_year_sequence(start_year, end_year):
    """Input start year, end year, generate an annual sequence."""
    year_sequence = []
    for year in range(start_year, end_year + 1):
        year_sequence.append(year)
    number_sequence = np.asarray(year_sequence)
    return number_sequence

def generate_date_sequence(start_year, start_month, end_year, end_month):
    """Input start year, start month, end year, end month, generate a monthly sequence."""
    start_date = pd.to_datetime(f"{start_year}-{start_month:02d}-01")
    end_date = pd.to_datetime(f"{end_year}-{end_month:02d}-01")
    next_month = end_date + pd.DateOffset(months=1)
    date_sequence = pd.date_range(start=start_date, end=next_month, freq='M')
    date_sequence = date_sequence.to_numpy()
    return date_sequence

def apply_boolean_mask(arr, mask):
    """Apply mask."""
    arr2 = arr.copy()  # Create a copy to prevent modification of the original array
    arr2[~mask] = np.nan
    return arr2

def save_ok(array, file_name):
    """Save function. Note: The code for reading LAI data must be run first, otherwise the spatial reference information of LAI cannot be used here."""
    base_path = r'D:\BaiduSyncdisk\Result'  # Local base path for saving
    output_path = f"{base_path}\\{file_name}.tif"  # Absolute path = base path + file name + .tiff suffix

    height, width = array.shape
    crs = LAI_year_025.rio.crs  # Spatial reference information needs to be specified. Here, the spatial reference of the subsequent LAI data is used.
    transform = LAI_year_025.rio.transform()

    meta = {
        'driver': 'GTiff',
        'dtype': 'float32',
        'count': 1,
        'width': width,
        'height': height,
        'transform': transform,
        'crs': crs
    }

    with rasterio.open(output_path, 'w', **meta) as dst:
        dst.write(array, 1)
    print(f"{file_name}.tif export completed")

def swap_360_720(latitude, longitude):
    """Input latitude, longitude, convert to grid point coordinates (360 * 720 resolution)."""
    x = 2 * (90 - latitude)
    y = 2 * (180 + longitude)
    return x, y

def swap_720_1440(latitude, longitude):
    """Input latitude, longitude, convert to grid point coordinates (720 * 1440 resolution)."""
    x = 4 * (90 - latitude)
    y = 4 * (180 + longitude)
    return x, y

def cal_MB(x):
    """Calculate the memory usage of a variable, unit is MB."""
    a = sys.getsizeof(x) / (1024 * 1024)
    return a

def moving_average(X, Y, window_size):
    """Piecewise average function, window_size is the window size."""
    # Ensure input sequences have the same length
    assert len(X) == len(Y), "X and Y sequences have inconsistent lengths"
    # Calculate moving average
    averages_X = []
    averages_Y = []
    for i in range(0, len(X) - window_size + 1, window_size):
        window_X = X[i:i + window_size]
        window_Y = Y[i:i + window_size]
        average_X = sum(window_X) / window_size
        average_Y = sum(window_Y) / window_size
        averages_X.append(average_X)
        averages_Y.append(average_Y)
    return np.array(averages_X), np.array(averages_Y)

def jinhua(array1, array2):
    """Purification function, only retains positions where both array1 and array2 have valid values. Requires array1 and array2 to be 1D sequences."""
    result_array1 = np.where(np.isnan(array1) | np.isnan(array2), np.nan, array1)
    result_array2 = np.where(np.isnan(array1) | np.isnan(array2), np.nan, array2)
    result_array11 = result_array1[~np.isnan(result_array1)]
    result_array22 = result_array2[~np.isnan(result_array2)]
    return result_array11, result_array22

def sort_by_A(A, B):
    """Sort A and B according to A's ascending order. Use the purification function beforehand!!"""
    # Combine sequences A and B into a list of tuples and sort by the value of A
    combined = sorted(zip(A, B), key=lambda x: x[0])
    # Unpack the sorted list of tuples to get sorted sequences A and B
    sorted_A, sorted_B = zip(*combined)
    return sorted_A, sorted_B

def split_array(arr):
    """Split an array into two halves according to the median."""
    mid = len(arr) // 2
    first_half = arr[:mid]
    second_half = arr[mid:]
    return first_half, second_half

def split_array_sorted(arr):
    """Sort an array first, then split it into two halves according to the median."""
    sorted_arr = sorted(arr)
    # Find the median position
    mid = len(sorted_arr) // 2
    # Split the sorted array into two halves
    first_half = sorted_arr[:mid]
    second_half = sorted_arr[mid:]
    return first_half, second_half

def split_arrays_pro(A, B, filename):
    """For single-point selection, without piecewise averaging, calculate three AI indices and plot. Includes plotting function."""
    x1, y1 = jinhua(A, B)
    X, Y = sort_by_A(x1, y1)
    mid_X = len(X) // 2
    x1 = X[:mid_X]
    x2 = X[mid_X:]
    # Find the median position of the Y array
    mid_Y = len(Y) // 2
    y1 = Y[:mid_Y]
    y2 = Y[mid_Y:]
    slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
    slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
    slope3, intercept3, r_value3, p_value3, std_err3 = linregress(X, Y)
    positive = np.max(y1) - np.mean(y1)
    negeative = np.mean(y1) - np.min(y1)

    # Ensure x1, x2, X are all NumPy arrays
    x1 = np.array(x1)
    x2 = np.array(x2)
    X = np.array(X)

    y1_new = x1 * slope1 + intercept1
    y2_new = x2 * slope2 + intercept2
    new_Y = X * slope3 + intercept3

    A1 = (slope2 - slope1) / (slope2 + slope1)
    B2 = (r_value2 - r_value1) / (r_value2 + r_value1)
    C3 = (positive - negeative) / (positive + negeative)

    fig, ax = plt.subplots()
    ax.scatter(x1, y1, s=200, alpha=0.25, color='#E64B35FF')
    ax.scatter(x2, y2, s=200, alpha=0.25, color='#4DBBD5FF')
    ax.plot(x1, y1_new, color='#E64B35FF',
            label=f'Arid slope:{slope1:.2f}   p_value: {p_value1:.2f} ')
    ax.plot(x2, y2_new, color='#4DBBD5FF',
            label=f'Humid slope:{slope2:.2f}   p_value: {p_value2:.2f}')
    ax.text(0.06, 0.7, f'AI_slope: {A1:.2f}', transform=ax.transAxes)
    ax.text(0.06, 0.6, f'AI_r: {B2:.2f}', transform=ax.transAxes)
    ax.text(0.06, 0.5, f'AI_extreme: {C3:.2f}', transform=ax.transAxes)

    ax.set_title(f"{filename}")
    ax.set_xlabel('SAI_PR')
    ax.set_ylabel('SAI_NDVI')
    ax.legend(loc='upper left')
    plt.show()
    return slope1, slope2, slope3

def split_arrays_ultra(A, B, filename):
    """Select 8-neighborhood, apply piecewise averaging, calculate three AI indices and plot. Includes plotting function."""
    x1, y1 = jinhua(A, B)
    X, Y = sort_by_A(x1, y1)
    X, Y = moving_average(X, Y, len(X) // 40)
    # Find the median position of the X array
    mid_X = len(X) // 2
    x1 = X[:mid_X]
    x2 = X[mid_X:]

    # Find the median position of the Y array
    mid_Y = len(Y) // 2
    y1 = Y[:mid_Y]
    y2 = Y[mid_Y:]

    slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
    slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
    slope3, intercept3, r_value3, p_value3, std_err3 = linregress(X, Y)

    # Ensure x1, x2, X are all NumPy arrays
    x1 = np.array(x1)
    x2 = np.array(x2)
    X = np.array(X)

    y1_new = x1 * slope1 + intercept1
    y2_new = x2 * slope2 + intercept2
    new_Y = X * slope3 + intercept3

    AI = (slope2 - slope1) / (slope2 + slope1)

    fig, ax = plt.subplots()
    ax.scatter(x1, y1, s=200, alpha=0.25, color='#E64B35FF')
    ax.scatter(x2, y2, s=200, alpha=0.25, color='#4DBBD5FF')
    ax.plot(x1, y1_new, color='#E64B35FF',
            label=f'Drought  slope:{slope1:.2f} p_value: {p_value1:.2f} ')
    ax.plot(x2, y2_new, color='#4DBBD5FF',
            label=f'Wet slope:{slope2:.2f} p_value: {p_value2:.2f}\nAI_slope:{AI:.2f}')

    ax.set_title(f"{filename}")
    ax.set_xlabel('SAI_pr', fontsize=14, fontweight='bold')
    ax.set_ylabel('SAI_veg', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left')
    plt.show()
    return slope1, slope2, slope3

def split_arrays_standard(A, B):
    """Select 8-neighborhood, apply piecewise averaging, calculate three AI indices, no plotting function."""
    x1, y1 = jinhua(A, B)
    X, Y = sort_by_A(x1, y1)
    mid_X = len(X) // 2
    x1 = X[:mid_X]
    x2 = X[mid_X:]

    # Find the median position of the Y array
    mid_Y = len(Y) // 2
    y1 = Y[:mid_Y]
    y2 = Y[mid_Y:]

    slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
    slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
    # Start conditional judgment
    if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # If both dry and wet months respond positively and linearly significantly
        AI_slope_010[i, j] = (slope2 - slope1) / (slope2 + slope1)
        AI_r_010[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
        AI_extreme_010[i, j] = (positive - negeative) / (positive + negeative)

    elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:  # If dry month responds positively and linearly significantly, but wet month shows significant negative response
        AI_slope_010[i, j] = -1
        AI_r_010[i, j] = -1
        AI_extreme_010[i, j] = -1
    elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 > 0.1:  # If dry month responds positively and linearly significantly, but wet month shows no significant linear response
        AI_slope_010[i, j] = -1
        AI_r_010[i, j] = -1
        AI_extreme_010[i, j] = -1
    elif slope1 < 0 or p_value1 > 0.1:  # If dry month does not respond, skip calculation directly
        AI_slope_010[i, j] = np.nan
        AI_r_010[i, j] = np.nan
        AI_extreme_010[i, j] = np.nan
    return AI_slope_010

def resample_numpy_array(arr, new_heightt, new_widthh):
    """Resample code, default resampling to 360 * 720, 0.5° spatial resolution."""
    """
    Resample a numpy array to a specified size.
    :param arr: Input numpy array
    :param new_height: Height after resampling
    :param new_width: Width after resampling
    :return: Resampled numpy array
    """
    new_height = new_heightt
    new_width = new_widthh
    # Get the height and width of the input array
    original_height, original_width = arr.shape

    # Calculate zoom factors
    zoom_factor_y = new_height / original_height
    zoom_factor_x = new_width / original_width

    # Use scipy.ndimage's zoom for resampling
    resampled_arr = zoom(arr, (zoom_factor_y, zoom_factor_x), order=1)  # order=1 indicates bilinear interpolation
    return resampled_arr

def calculate_averages(arr_b, arr_c, breakpoints):
    """Input sorted arrays b and C, and breakpoints array."""
    # 3 breakpoints will result in 4 intervals
    # Initialize result lists
    averages_b = []
    averages_c = []

    # Ensure breakpoints are sorted in ascending order
    breakpoints = sorted(breakpoints)
    breakpoints = [float('-inf')] + breakpoints + [float('inf')]

    # Traverse each interval and calculate the average
    start_index = 0
    for breakpoint in breakpoints[1:]:
        # Find the end index corresponding to the current breakpoint
        end_index = next((i for i, x in enumerate(arr_b) if x > breakpoint), len(arr_b))

        # Calculate the subarray of array B
        sub_array_b = arr_b[start_index:end_index]
        # If the subarray is not empty, calculate the average; otherwise, average is 0
        if sub_array_b:
            average_b = sum(sub_array_b) / len(sub_array_b)
        else:
            average_b = np.nan
        averages_b.append(average_b)

        # Calculate the subarray of array C
        sub_array_c = arr_c[start_index:end_index]
        # If the subarray is not empty, calculate the average; otherwise, average is 0
        if sub_array_c:
            average_c = sum(sub_array_c) / len(sub_array_c)
        else:
            average_c = np.nan
        averages_c.append(average_c)

        start_index = end_index

    return averages_b, averages_c

def veg_ganzaodu(n, array):
    """This function needs to be used in conjunction with the function above."""
    breakpoints = np.arange(0.05, 2, 0.4)  # These are the breakpoints, i.e., division points for aridity index, from 0.05 to 2, with an interval of 0.4
    mask_lin = (veg == n)  # Input the corresponding vegetation type number
    # Extract aridity values and convert to a 1D array
    masked_qihou_lin2 = ganshi_raw[mask_lin]
    masked_qihou_lin1 = masked_qihou_lin2.flatten()
    # Extract AI values and convert to a 1D array
    masked_AI_slope_lin2 = array[mask_lin]
    masked_AI_slope_lin1 = masked_AI_slope_lin2.flatten()
    # Purify first, then sort
    qihou_lin_jinhua, AI_slope_lin_jinhua = jinhua(masked_qihou_lin1, masked_AI_slope_lin1)
    masked_qihou_lin_sort, masked_veg_lin_sort = sort_by_A(qihou_lin_jinhua, AI_slope_lin_jinhua)
    masked_qihou_lin_sort = np.array(masked_qihou_lin_sort)
    masked_veg_lin_sort = np.array(masked_veg_lin_sort)

    masked_qihou_lin_sort_list = masked_qihou_lin_sort.tolist()
    masked_veg_lin_sort_list = masked_veg_lin_sort.tolist()
    lin_qihou, lin_veg = calculate_averages(masked_qihou_lin_sort_list, masked_veg_lin_sort_list, breakpoints)
    lin = np.array(lin_veg)
    return lin  # Returns the AI index

def calculate_non_nan_percentage(data_array):
    """Used to count the percentage of non-NaN values in an array. It is recommended to first calculate for the climate zone, then for the region."""
    # For example, calculate_non_nan_percentage(ganshi_025), calculate_non_nan_percentage(AI_slope_CRU_025_1). Then divide the two.
    # Convert the list to a NumPy array
    np_array = np.array(data_array, dtype=float)  # Ensure the array data type can accommodate NaN values
    non_nan_count = np.count_nonzero(~np.isnan(np_array))
    total_count = np_array.size
    percentage = (non_nan_count / total_count) * 100
    # Return the result
    return percentage

def filter_nan_values(matrix1, matrix2):
    """
    Filter NaN values at corresponding positions in two matrices, retaining only values where neither position is NaN.

    Parameters:
        matrix1 (numpy.ndarray): First matrix
        matrix2 (numpy.ndarray): Second matrix

    Returns:
        filtered_matrix1 (numpy.ndarray): Processed first matrix
        filtered_matrix2 (numpy.ndarray): Processed second matrix
    """
    # Ensure input matrices are NumPy arrays
    matrix1 = np.array(matrix1)
    matrix2 = np.array(matrix2)

    # Find boolean mask where corresponding positions in both matrices are not NaN
    mask = ~np.isnan(matrix1) & ~np.isnan(matrix2)

    # Use boolean mask to filter matrices, keeping only non-NaN positions, other positions filled with NaN
    filtered_matrix1 = np.where(mask, matrix1, np.nan)
    filtered_matrix2 = np.where(mask, matrix2, np.nan)

    return filtered_matrix1, filtered_matrix2

导入自定义函数


In [3]:
# NDVI data, 24 data points per year, from 1982-2022, a total of 41 years, 984 data points in total
inpath = r'D:/JYQ_data/JYQ/long_data/NDVI/'  # Change to your own absolute path
start_year = 1982
end_year = 2022
start_month = 1
end_month = 12
all_data_ndvi_050 = []
all_data_ndvi_025 = []

for year in range(start_year, end_year + 1):
    for month in range(start_month, end_month + 1):
        for half_month in range(1, 3):  # Half-month loop, as there are two data points per month
            # Construct file name
            prefix = 'PKU_GIMMS_NDVI_V1.2_'
            inname = f'{inpath}{prefix}{year}{month:02d}{half_month:02d}.tif'

            # Read data
            data = rioxarray.open_rasterio(inname)
            data_ndvi = data[0, :, :].astype(float)  # Extract NDVI, setting 1 would extract the quality control band
            data_ndvi = np.where(data_ndvi == 65535, np.nan, data_ndvi)  # Remove invalid values
            data_ndvi *= 0.001  # Apply scaling factor
            resampled_array_050 = resample_numpy_array(data_ndvi, 360, 720)  # Resample to 0.5° spatial resolution, size 360 * 720
            resampled_array_025 = resample_numpy_array(data_ndvi, 720, 1440)  # Resample to 0.25° spatial resolution, size 720 * 1440
            all_data_ndvi_050.append(resampled_array_050)
            all_data_ndvi_025.append(resampled_array_025)

final_array_025 = np.array(all_data_ndvi_025)  # Convert list to array
NDVI_monthly = final_array_025.copy()
final_array_050 = np.array(all_data_ndvi_050)  # Convert list to array
del all_data_ndvi_050, all_data_ndvi_025
gc.collect()  # Here, all_data_ndvi can be deleted and memory released

NDVI_year_025 = np.nanmean(final_array_025.reshape(41, 24, 720, 1440), axis=1)
NDVI_year_050 = np.nanmean(final_array_050.reshape(41, 24, 360, 720), axis=1)

del final_array_025, final_array_050  # Memory can be released
gc.collect()
print('Shape of NDVI_year_025:', NDVI_year_025.shape)
print('Shape of NDVI_year_050:', NDVI_year_050.shape)
print('Shape of NDVI_monthly:', NDVI_monthly.shape)
#--------------------------------------------------------------------------------------------------------------------------------------
# LAI data, time span from 1982-2021, annual scale data
LAI = r'D:\JYQ_data\JYQ\long_data\lai\LAI2.tif'  # Change to your own absolute path
LAI_year_025 = rioxarray.open_rasterio(LAI, default_name='LAI', lock=threading.Lock())
print('Shape of LAI_year_025:', LAI_year_025.shape)

C:\Users\JYQ\AppData\Local\Temp\ipykernel_212376\4075614791.py:33: RuntimeWarning: Mean of empty slice
  NDVI_year_025=np.nanmean(final_array_025.reshape(41,24,720,1440),axis=1)
C:\Users\JYQ\AppData\Local\Temp\ipykernel_212376\4075614791.py:34: RuntimeWarning: Mean of empty slice
  NDVI_year_050=np.nanmean(final_array_050.reshape(41,24,360,720),axis=1)


NDVI_year_025的规格： (41, 720, 1440)
NDVI_year_050的规格： (41, 360, 720)
NDVI_yue的规格： (984, 720, 1440)
LAI_year_025的规格： (40, 720, 1440)


In [5]:
pr_raw = r'D:\JYQ_data\JYQ\long_data\pr\Terra\Terra.tif'  # Remember to change to your own absolute path
pr_year = rioxarray.open_rasterio(pr_raw, default_name='pr_raw', lock=threading.Lock())
Terra_monthly = pr_year.values
Terra_year = np.sum(Terra_monthly.reshape(41, 12, 360, 720), axis=1)
Terra_year = Terra_year.astype(float)  # Filter out ocean areas
Terra_year[Terra_year == 0] = np.nan

Terra_year_025 = np.empty((41, 720, 1440), dtype=float)
for i in range(41):
    Terra_year_025[i, :, :] = resample_numpy_array(Terra_year[i], 720, 1440)
Terra_year_050 = Terra_year

Terra_monthly = Terra_monthly.astype(float)  # Filter out ocean areas
Terra_monthly[Terra_monthly == 0] = np.nan
Terra_monthly_025 = np.empty((492, 720, 1440), dtype=float)
for i in range(492):
    Terra_monthly_025[i, :, :] = resample_numpy_array(Terra_monthly[i], 720, 1440)
print('Shape of Terra_year_025:', Terra_year_025.shape)
print('Shape of Terra_year_050:', Terra_year_050.shape)
# ---------------------------------------------------------------------------------------------------------------------------------
CRU_1981_1990_raw = nc.Dataset(r'D:\JYQ_data\JYQ\long_data\pr\CRU\cru_ts4.07.1981.1990.pre.dat.nc', 'r')
CRU_1991_2000_raw = nc.Dataset(r'D:\JYQ_data\JYQ\long_data\pr\CRU\cru_ts4.07.1991.2000.pre.dat.nc', 'r')
CRU_2001_2010_raw = nc.Dataset(r'D:\JYQ_data\JYQ\long_data\pr\CRU\cru_ts4.07.2001.2010.pre.dat.nc', 'r')
CRU_2011_2020_raw = nc.Dataset(r'D:\JYQ_data\JYQ\long_data\pr\CRU\cru_ts4.07.2011.2020.pre.dat.nc', 'r')
CRU_2021_2022_raw = nc.Dataset(r'D:\JYQ_data\JYQ\long_data\pr\CRU\cru_ts4.07.2021.2022.pre.dat.nc', 'r')
CRU_1981_1990 = CRU_1981_1990_raw['pre']  # Extract precipitation variable
CRU_1991_2000 = CRU_1991_2000_raw['pre']
CRU_2001_2010 = CRU_2001_2010_raw['pre']
CRU_2011_2020 = CRU_2011_2020_raw['pre']
CRU_2021_2022 = CRU_2021_2022_raw['pre']
CRU_1981_2022 = np.concatenate((CRU_1981_1990, CRU_1991_2000, CRU_2001_2010, CRU_2011_2020, CRU_2021_2022), axis=0)  # Combine them
CRU_1982_2022 = CRU_1981_2022[12:, :, :]  # Only keep data starting from 1982
CRU_year = np.sum(CRU_1982_2022.reshape(41, 12, 360, 720), axis=1)
CRU_year_025 = np.empty((41, 720, 1440), dtype=float)
for i in range(41):  # Flip vertically
    CRU_year[i] = np.flip(CRU_year[i], axis=0)
    CRU_year_025[i, :, :] = resample_numpy_array(CRU_year[i], 720, 1440)

CRU_year_050 = CRU_year  # CRU_year is already CRU_year_050, so directly assign
CRU_year_mean = np.nanmean(CRU_year_025, axis=0)
# Resample to 720 * 1440 resolution
print("Shape of CRU_year_050:", CRU_year_050.shape)
print("Shape of CRU_year_025:", CRU_year_025.shape)


Terra_year_025的规格: (41, 720, 1440)
Terra_year_050的规格: (41, 360, 720)
CRU_year_050的规格： (41, 360, 720)
CRU_year_025的规格： (41, 720, 1440)


In [4]:
# Read in ganshi_raw, this is the data without reclassification, can be used for gradient plots. ganshi is the reclassified map.
out = r'D:\JYQ_data\JYQ\AI\Global-AI_ET0_annual_v3\Global-AI_ET0_v3_annual\ai_v3_yr.tif'
ganshi_fenqu = rioxarray.open_rasterio(out, default_name='out', lock=threading.Lock())
ganshi_fenqu2 = ganshi_fenqu[0]
ganshi_fenqu_re_050 = resample_numpy_array(ganshi_fenqu2, 360, 720)
ganshi_fenqu_re_025 = resample_numpy_array(ganshi_fenqu2, 720, 1440)
ganshi_raw_050 = ganshi_fenqu_re_050 / 10000
ganshi_raw_025 = ganshi_fenqu_re_025 / 10000

# Remove areas where the mean NDVI in the arid zone is less than 0.1, as these areas essentially have no vegetation, consisting of Gobi and deserts.
NDVI_mean_025 = np.nanmean(NDVI_year_025, axis=0)
mask_025 = NDVI_mean_025 >= 0.1
ganshi_raw_025 = np.where(mask_025, ganshi_raw_025, np.nan)

NDVI_mean_050 = np.nanmean(NDVI_year_050, axis=0)
mask_050 = NDVI_mean_050 >= 0.1
ganshi_raw_050 = np.where(mask_050, ganshi_raw_050, np.nan)

# Start reclassification
conditions_050 = [
    ganshi_raw_050 <= 0,  # Invalid value
    (ganshi_raw_050 > 0) & (ganshi_raw_050 < 0.03),  # Extremely arid
    (ganshi_raw_050 >= 0.03) & (ganshi_raw_050 <= 0.2),  # Arid
    (ganshi_raw_050 > 0.2) & (ganshi_raw_050 <= 0.5),  # Semi-arid
    (ganshi_raw_050 > 0.5) & (ganshi_raw_050 <= 0.65),  # Semi-humid
    ganshi_raw_050 > 0.65  # Humid
]

conditions_025 = [
    ganshi_raw_025 <= 0,  # Invalid value
    (ganshi_raw_025 > 0) & (ganshi_raw_025 < 0.03),  # Extremely arid
    (ganshi_raw_025 >= 0.03) & (ganshi_raw_025 <= 0.2),  # Arid
    (ganshi_raw_025 > 0.2) & (ganshi_raw_025 <= 0.5),  # Semi-arid
    (ganshi_raw_025 > 0.5) & (ganshi_raw_025 <= 0.65),  # Semi-humid
    ganshi_raw_025 > 0.65  # Humid
]

# Define values corresponding to conditions
choices = [
    0,
    1,  # Values less than 0.03 replaced with 1
    2,  # Values between 0.03 and 0.2 replaced with 2
    3,  # Values between 0.2 and 0.5 replaced with 3
    4,  # Values between 0.5 and 0.65 replaced with 4
    5  # Values greater than 0.65 replaced with 5
]
# Use np.select for conditional selection and replacement
ganshi_050 = np.select(conditions_050, choices, default=np.nan)
ganshi_025 = np.select(conditions_025, choices, default=np.nan)

print('Shape of ganshi_050:', ganshi_050.shape)
print('Shape of ganshi_raw_050:', ganshi_raw_050.shape)
print('Shape of ganshi_025:', ganshi_025.shape)
print('Shape of ganshi_raw_050:', ganshi_raw_050.shape)

ganshi_050的规格： (360, 720)
ganshi_raw_050的规格： (360, 720)
ganshi_025的规格： (720, 1440)
ganshi_raw_050的规格： (360, 720)


C:\Users\JYQ\AppData\Local\Temp\ipykernel_212376\1977937367.py:11: RuntimeWarning: Mean of empty slice
  NDVI_mean_025=np.nanmean(NDVI_year_025,axis=0)
C:\Users\JYQ\AppData\Local\Temp\ipykernel_212376\1977937367.py:15: RuntimeWarning: Mean of empty slice
  NDVI_mean_050=np.nanmean(NDVI_year_050,axis=0)


In [6]:
# The following is vegetation zone data. The label for shrubland is 6, and for grassland is 10.
out = r'D:\JYQ_data\JYQ\IGBP\IGBP.tif'
veg_re = rioxarray.open_rasterio(out, default_name='out', lock=threading.Lock())
veg_re2 = veg_re[0]
veg = resample_numpy_array(veg_re2, 720, 1440)
veg[veg == 7] = 6  # Merge open and closed shrubland types
veg[veg == 9] = 8  # Merge woody savannas and savannas
veg[veg == 14] = 12  # Merge croplands and natural vegetation mosaic
veg[veg == 17] = 0  # Remove water bodies as they are not needed
print('Shape of vegetation cover type veg:', veg.shape)
fig, ax = plt.subplots()
plt.imshow(veg)
plt.show()

植被覆盖类型veg的规格 (720, 1440)


In [7]:
print('Function 1: 3 AI indicators, single neighborhood, 40-segment piecewise averaging, adaptive spatial resolution')
print('Function 2: 3 AI indicators, 8-neighborhood, 40-segment piecewise averaging, adaptive spatial resolution')
print('Function 3: 3 AI indicators, 24-neighborhood, 40-segment piecewise averaging, adaptive spatial resolution')

def calculate_AI_1(pr_anomaly, NDVI_anomaly, n):  # Single neighborhood, input n, adaptive to three spatial resolutions
    # Define empty variables to store results
    AI_slope_050 = np.empty((360, 720), dtype=float)
    AI_r_050 = np.empty((360, 720), dtype=float)
    AI_extreme_050 = np.empty((360, 720), dtype=float)
    AI_slope_050_all = np.empty((360, 720), dtype=float)
    AI_r_050_all = np.empty((360, 720), dtype=float)
    AI_extreme_050_all = np.empty((360, 720), dtype=float)
    AI_slope_025 = np.empty((720, 1440), dtype=float)
    AI_r_025 = np.empty((720, 1440), dtype=float)
    AI_extreme_025 = np.empty((720, 1440), dtype=float)
    AI_slope_025_all = np.empty((720, 1440), dtype=float)
    AI_r_025_all = np.empty((720, 1440), dtype=float)
    AI_extreme_025_all = np.empty((720, 1440), dtype=float)
    AI_slope_010 = np.empty((1800, 3600), dtype=float)
    AI_r_010 = np.empty((1800, 3600), dtype=float)
    AI_extreme_010 = np.empty((1800, 3600), dtype=float)
    AI_slope_010_all = np.empty((1800, 3600), dtype=float)
    AI_r_010_all = np.empty((1800, 3600), dtype=float)
    AI_extreme_010_all = np.empty((1800, 3600), dtype=float)

    if n == 0.5:  # Size (360, 720)
        for i in range(360):
            for j in range(720):
                if np.all(np.isnan(pr_anomaly[:, i, j])) or np.all(np.isnan(NDVI_anomaly[:, i, j])):  # If both center pixel sequences are all NaN
                    AI_slope_050[i, j] = np.nan
                    AI_r_050[i, j] = np.nan
                    AI_extreme_050[i, j] = np.nan
                    AI_slope_050_all[i, j] = np.nan
                    AI_r_050_all[i, j] = np.nan
                    AI_extreme_050_all[i, j] = np.nan
                else:  # Ensure center pixel sequence is not all NaN, at least one valid value
                    test1 = pr_anomaly[:, i, j].flatten()
                    test2 = NDVI_anomaly[:, i, j].flatten()  # No need to extract neighborhood
                    pr_purified, NDVI_purified = jinhua(test1, test2)  # Purify two arrays
                    if np.all(np.isnan(test1)) or np.all(np.isnan(test2)):  # Must check after purification, as both might become NaN after purification
                        AI_slope_050[i, j] = np.nan
                        AI_r_050[i, j] = np.nan
                        AI_extreme_050[i, j] = np.nan
                        AI_slope_050_all[i, j] = np.nan
                        AI_r_050_all[i, j] = np.nan
                        AI_extreme_050_all[i, j] = np.nan
                    else:
                        pr_sorted, NDVI_sorted = sort_by_A(pr_purified, NDVI_purified)  # After purification, now sort by moisture from low to high
                        mid = len(pr_sorted) // 2  # Divide by median, ensuring at least 20 samples on each side
                        x1 = pr_sorted[0:mid]
                        y1 = NDVI_sorted[0:mid]
                        slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
                        x2 = pr_sorted[mid:]
                        y2 = NDVI_sorted[mid:]
                        slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
                        max_NDVI = np.mean(NDVI_sorted[-5:])  # Extract NDVI values for the highest precipitation years, last 5
                        min_NDVI = np.mean(NDVI_sorted[:5])  # Extract NDVI values for the lowest precipitation years, first 5
                        mean_NDVI = np.mean(NDVI_sorted)  # Calculate the mean of the entire sequence
                        positive = (max_NDVI - mean_NDVI)
                        negative = (mean_NDVI - min_NDVI)

                        # Direct calculation without considering significance
                        AI_slope_050_all[i, j] = (slope2 - slope1) / (slope2 + slope1)
                        AI_r_050_all[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                        AI_extreme_050_all[i, j] = (positive - negative) / (positive + negative)

                        # Start conditional judgment
                        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # If both dry and wet years respond positively and linearly significantly
                            AI_slope_050[i, j] = (slope2 - slope1) / (slope2 + slope1)
                            AI_r_050[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                            AI_extreme_050[i, j] = (positive - negative) / (positive + negative)

                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:  # If dry year responds positively and linearly significantly, but wet year responds significantly negatively
                            AI_slope_050[i, j] = -1
                            AI_r_050[i, j] = -1
                            AI_extreme_050[i, j] = -1
                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 > 0.1:  # If dry year responds positively and linearly significantly, but wet year shows no significant linear response
                            AI_slope_050[i, j] = -1
                            AI_r_050[i, j] = -1
                            AI_extreme_050[i, j] = -1
                        elif slope1 < 0 or p_value1 > 0.1:  # If dry year does not respond, skip calculation directly
                            AI_slope_050[i, j] = np.nan
                            AI_r_050[i, j] = np.nan
                            AI_extreme_050[i, j] = np.nan
        return AI_slope_050, AI_r_050, AI_extreme_050, AI_slope_050_all, AI_r_050_all, AI_extreme_050_all

    elif n == 0.25:  # Size (720, 1440)
        for i in range(720):
            for j in range(1440):
                if np.all(np.isnan(pr_anomaly[:, i, j])) or np.all(np.isnan(NDVI_anomaly[:, i, j])):
                    AI_slope_025[i, j] = np.nan
                    AI_r_025[i, j] = np.nan
                    AI_extreme_025[i, j] = np.nan
                    AI_slope_025_all[i, j] = np.nan
                    AI_r_025_all[i, j] = np.nan
                    AI_extreme_025_all[i, j] = np.nan

                else:  # Ensure center pixel sequence is not all NaN, at least one valid value
                    test1 = pr_anomaly[:, i, j].flatten()
                    test2 = NDVI_anomaly[:, i, j].flatten()  # No need to extract neighborhood
                    pr_purified, NDVI_purified = jinhua(test1, test2)  # Purify two arrays
                    if np.all(np.isnan(test1)) or np.all(np.isnan(test2)):  # Must check after purification, as both might become NaN after purification
                        AI_slope_025[i, j] = np.nan
                        AI_r_025[i, j] = np.nan
                        AI_extreme_025[i, j] = np.nan
                        AI_slope_025_all[i, j] = np.nan
                        AI_r_025_all[i, j] = np.nan
                        AI_extreme_025_all[i, j] = np.nan
                    else:
                        pr_sorted, NDVI_sorted = sort_by_A(pr_purified, NDVI_purified)  # After purification, now sort by moisture from low to high
                        mid = len(pr_sorted) // 2  # Divide by median, ensuring at least 20 samples on each side
                        x1 = pr_sorted[0:mid]
                        y1 = NDVI_sorted[0:mid]
                        slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
                        x2 = pr_sorted[mid:]
                        y2 = NDVI_sorted[mid:]
                        slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
                        max_NDVI = np.mean(NDVI_sorted[-5:])  # Extract NDVI values for the highest precipitation years, last 5
                        min_NDVI = np.mean(NDVI_sorted[:5])  # Extract NDVI values for the lowest precipitation years, first 5
                        mean_NDVI = np.mean(NDVI_sorted)  # Calculate the mean of the entire sequence
                        positive = (max_NDVI - mean_NDVI)
                        negative = (mean_NDVI - min_NDVI)

                        # Direct calculation without considering significance
                        AI_slope_025_all[i, j] = (slope2 - slope1) / (slope2 + slope1)
                        AI_r_025_all[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                        AI_extreme_025_all[i, j] = (positive - negative) / (positive + negative)

                        # Start conditional judgment
                        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # If both dry and wet years respond positively and linearly significantly
                            AI_slope_025[i, j] = (slope2 - slope1) / (slope2 + slope1)
                            AI_r_025[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                            AI_extreme_025[i, j] = (positive - negative) / (positive + negative)

                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:  # If dry year responds positively and linearly significantly, but wet year responds significantly negatively
                            AI_slope_025[i, j] = -1
                            AI_r_025[i, j] = -1
                            AI_extreme_025[i, j] = -1
                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 > 0.1:  # If dry year responds positively and linearly significantly, but wet year shows no significant linear response
                            AI_slope_025[i, j] = -1
                            AI_r_025[i, j] = -1
                            AI_extreme_025[i, j] = -1
                        elif slope1 < 0 or p_value1 > 0.1:  # If dry year does not respond, skip calculation directly
                            AI_slope_025[i, j] = np.nan
                            AI_r_025[i, j] = np.nan
                            AI_extreme_025[i, j] = np.nan
        return AI_slope_025, AI_r_025, AI_extreme_025, AI_slope_025_all, AI_r_025_all, AI_extreme_025_all

    elif n == 0.1:  # Size (1800, 3600)
        for i in range(1800):
            for j in range(3600):
                if np.all(np.isnan(pr_anomaly[:, i, j])) or np.all(np.isnan(NDVI_anomaly[:, i, j])):
                    AI_slope_010[i, j] = np.nan
                    AI_r_010[i, j] = np.nan
                    AI_extreme_010[i, j] = np.nan
                    AI_slope_010_all[i, j] = np.nan
                    AI_r_010_all[i, j] = np.nan
                    AI_extreme_010_all[i, j] = np.nan

                else:  # Ensure center pixel sequence is not all NaN, at least one valid value
                    test1 = pr_anomaly[:, i, j].flatten()
                    test2 = NDVI_anomaly[:, i, j].flatten()  # No need to extract neighborhood
                    pr_purified, NDVI_purified = jinhua(test1, test2)  # Purify two arrays
                    if np.all(np.isnan(test1)) or np.all(np.isnan(test2)):  # Must check after purification, as both might become NaN after purification
                        AI_slope_010[i, j] = np.nan
                        AI_r_010[i, j] = np.nan
                        AI_extreme_010[i, j] = np.nan
                        AI_slope_010_all[i, j] = np.nan
                        AI_r_010_all[i, j] = np.nan
                        AI_extreme_010_all[i, j] = np.nan
                    else:
                        pr_sorted, NDVI_sorted = sort_by_A(pr_purified, NDVI_purified)  # After purification, now sort by moisture from low to high
                        mid = len(pr_sorted) // 2  # Divide by median, ensuring at least 20 samples on each side
                        x1 = pr_sorted[0:mid]
                        y1 = NDVI_sorted[0:mid]
                        slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
                        x2 = pr_sorted[mid:]
                        y2 = NDVI_sorted[mid:]
                        slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
                        max_NDVI = np.mean(NDVI_sorted[-5:])  # Extract NDVI values for the highest precipitation years, last 5
                        min_NDVI = np.mean(NDVI_sorted[:5])  # Extract NDVI values for the lowest precipitation years, first 5
                        mean_NDVI = np.mean(NDVI_sorted)  # Calculate the mean of the entire sequence
                        positive = (max_NDVI - mean_NDVI)
                        negative = (mean_NDVI - min_NDVI)

                        # Direct calculation without considering significance
                        AI_slope_010_all[i, j] = (slope2 - slope1) / (slope2 + slope1)
                        AI_r_010_all[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                        AI_extreme_010_all[i, j] = (positive - negative) / (positive + negative)

                        # Start conditional judgment
                        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # If both dry and wet years respond positively and linearly significantly
                            AI_slope_010[i, j] = (slope2 - slope1) / (slope2 + slope1)
                            AI_r_010[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                            AI_extreme_010[i, j] = (positive - negative) / (positive + negative)

                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:  # If dry year responds positively and linearly significantly, but wet year responds significantly negatively
                            AI_slope_010[i, j] = -1
                            AI_r_010[i, j] = -1
                            AI_extreme_010[i, j] = -1
                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 > 0.1:  # If dry year responds positively and linearly significantly, but wet year shows no significant linear response
                            AI_slope_010[i, j] = -1
                            AI_r_010[i, j] = -1
                            AI_extreme_010[i, j] = -1
                        elif slope1 < 0 or p_value1 > 0.1:  # If dry year does not respond, skip calculation directly
                            AI_slope_010[i, j] = np.nan
                            AI_r_010[i, j] = np.nan
                            AI_extreme_010[i, j] = np.nan
        return AI_slope_010, AI_r_010, AI_extreme_010, AI_slope_010_all, AI_r_010_all, AI_extreme_010_all

def calculate_AI_8(pr_anomaly, NDVI_anomaly, n):  # 8-neighborhood, input n, adaptive to three spatial resolutions
    # Define empty variables to store results
    AI_slope_050 = np.empty((360, 720), dtype=float)
    AI_r_050 = np.empty((360, 720), dtype=float)
    AI_extreme_050 = np.empty((360, 720), dtype=float)
    AI_slope_050_all = np.empty((360, 720), dtype=float)
    AI_r_050_all = np.empty((360, 720), dtype=float)
    AI_extreme_050_all = np.empty((360, 720), dtype=float)
    AI_slope_025 = np.empty((720, 1440), dtype=float)
    AI_r_025 = np.empty((720, 1440), dtype=float)
    AI_extreme_025 = np.empty((720, 1440), dtype=float)
    AI_slope_025_all = np.empty((720, 1440), dtype=float)
    AI_r_025_all = np.empty((720, 1440), dtype=float)
    AI_extreme_025_all = np.empty((720, 1440), dtype=float)
    AI_slope_010 = np.empty((1800, 3600), dtype=float)
    AI_r_010 = np.empty((1800, 3600), dtype=float)
    AI_extreme_010 = np.empty((1800, 3600), dtype=float)
    AI_slope_010_all = np.empty((1800, 3600), dtype=float)
    AI_r_010_all = np.empty((1800, 3600), dtype=float)
    AI_extreme_010_all = np.empty((1800, 3600), dtype=float)

    if n == 0.5:  # Size (360, 720)
        for i in range(360):
            for j in range(720):
                if np.all(np.isnan(pr_anomaly[:, i, j])) or np.all(np.isnan(NDVI_anomaly[:, i, j])):  # If both center pixel sequences are all NaN
                    AI_slope_050[i, j] = np.nan
                    AI_r_050[i, j] = np.nan
                    AI_extreme_050[i, j] = np.nan
                    AI_slope_050_all[i, j] = np.nan
                    AI_r_050_all[i, j] = np.nan
                    AI_extreme_050_all[i, j] = np.nan
                else:  # Ensure center pixel sequence is not all NaN, at least one valid value
                    test1 = pr_anomaly[:, i - 1:i + 2, j - 1:j + 2].flatten()
                    test2 = NDVI_anomaly[:, i - 1:i + 2, j - 1:j + 2].flatten()  # Extract 8-neighborhood
                    pr_purified, NDVI_purified = jinhua(test1, test2)  # Purify two arrays
                    if np.all(np.isnan(test1)) or np.all(np.isnan(test2)):  # Must check after purification, as both might become NaN after purification
                        AI_slope_050[i, j] = np.nan
                        AI_r_050[i, j] = np.nan
                        AI_extreme_050[i, j] = np.nan
                        AI_slope_050_all[i, j] = np.nan
                        AI_r_050_all[i, j] = np.nan
                        AI_extreme_050_all[i, j] = np.nan
                    else:
                        pr_sorted, NDVI_sorted = sort_by_A(pr_purified, NDVI_purified)  # After purification, now sort by moisture from low to high
                        mid = len(pr_sorted) // 2  # Divide by median, ensuring at least 20 samples on each side
                        x1 = pr_sorted[0:mid]
                        y1 = NDVI_sorted[0:mid]
                        slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
                        x2 = pr_sorted[mid:]
                        y2 = NDVI_sorted[mid:]
                        slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
                        max_NDVI = np.mean(NDVI_sorted[-5:])  # Extract NDVI values for the highest precipitation years, last 5
                        min_NDVI = np.mean(NDVI_sorted[:5])  # Extract NDVI values for the lowest precipitation years, first 5
                        mean_NDVI = np.mean(NDVI_sorted)  # Calculate the mean of the entire sequence
                        positive = (max_NDVI - mean_NDVI)
                        negative = (mean_NDVI - min_NDVI)

                        # Direct calculation without considering significance
                        AI_slope_050_all[i, j] = (slope2 - slope1) / (slope2 + slope1)
                        AI_r_050_all[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                        AI_extreme_050_all[i, j] = (positive - negative) / (positive + negative)

                        # Start conditional judgment
                        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # If both dry and wet years respond positively and linearly significantly
                            AI_slope_050[i, j] = (slope2 - slope1) / (slope2 + slope1)
                            AI_r_050[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                            AI_extreme_050[i, j] = (positive - negative) / (positive + negative)

                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:  # If dry year responds positively and linearly significantly, but wet year responds significantly negatively
                            AI_slope_050[i, j] = -1
                            AI_r_050[i, j] = -1
                            AI_extreme_050[i, j] = -1
                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 > 0.1:  # If dry year responds positively and linearly significantly, but wet year shows no significant linear response
                            AI_slope_050[i, j] = -1
                            AI_r_050[i, j] = -1
                            AI_extreme_050[i, j] = -1
                        elif slope1 < 0 or p_value1 > 0.1:  # If dry year does not respond, skip calculation directly
                            AI_slope_050[i, j] = np.nan
                            AI_r_050[i, j] = np.nan
                            AI_extreme_050[i, j] = np.nan
        return AI_slope_050, AI_r_050, AI_extreme_050, AI_slope_050_all, AI_r_050_all, AI_extreme_050_all

    elif n == 0.25:  # Size (720, 1440)
        for i in range(720):
            for j in range(1440):
                if np.all(np.isnan(pr_anomaly[:, i, j])) or np.all(np.isnan(NDVI_anomaly[:, i, j])):
                    AI_slope_025[i, j] = np.nan
                    AI_r_025[i, j] = np.nan
                    AI_extreme_025[i, j] = np.nan
                    AI_slope_025_all[i, j] = np.nan
                    AI_r_025_all[i, j] = np.nan
                    AI_extreme_025_all[i, j] = np.nan

                else:  # Ensure center pixel sequence is not all NaN, at least one valid value
                    test1 = pr_anomaly[:, i - 1:i + 2, j - 1:j + 2].flatten()
                    test2 = NDVI_anomaly[:, i - 1:i + 2, j - 1:j + 2].flatten()  # Extract 8-neighborhood
                    pr_purified, NDVI_purified = jinhua(test1, test2)  # Purify two arrays
                    if np.all(np.isnan(test1)) or np.all(np.isnan(test2)):  # Must check after purification, as both might become NaN after purification
                        AI_slope_025[i, j] = np.nan
                        AI_r_025[i, j] = np.nan
                        AI_extreme_025[i, j] = np.nan
                        AI_slope_025_all[i, j] = np.nan
                        AI_r_025_all[i, j] = np.nan
                        AI_extreme_025_all[i, j] = np.nan
                    else:
                        pr_sorted, NDVI_sorted = sort_by_A(pr_purified, NDVI_purified)  # After purification, now sort by moisture from low to high
                        mid = len(pr_sorted) // 2  # Divide by median, ensuring at least 20 samples on each side
                        x1 = pr_sorted[0:mid]
                        y1 = NDVI_sorted[0:mid]
                        slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
                        x2 = pr_sorted[mid:]
                        y2 = NDVI_sorted[mid:]
                        slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
                        max_NDVI = np.mean(NDVI_sorted[-5:])  # Extract NDVI values for the highest precipitation years, last 5
                        min_NDVI = np.mean(NDVI_sorted[:5])  # Extract NDVI values for the lowest precipitation years, first 5
                        mean_NDVI = np.mean(NDVI_sorted)  # Calculate the mean of the entire sequence
                        positive = (max_NDVI - mean_NDVI)
                        negative = (mean_NDVI - min_NDVI)

                        # Direct calculation without considering significance
                        AI_slope_025_all[i, j] = (slope2 - slope1) / (slope2 + slope1)
                        AI_r_025_all[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                        AI_extreme_025_all[i, j] = (positive - negative) / (positive + negative)

                        # Start conditional judgment
                        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # If both dry and wet years respond positively and linearly significantly
                            AI_slope_025[i, j] = (slope2 - slope1) / (slope2 + slope1)
                            AI_r_025[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                            AI_extreme_025[i, j] = (positive - negative) / (positive + negative)

                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:  # If dry year responds positively and linearly significantly, but wet year responds significantly negatively
                            AI_slope_025[i, j] = -1
                            AI_r_025[i, j] = -1
                            AI_extreme_025[i, j] = -1
                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 > 0.1:  # If dry year responds positively and linearly significantly, but wet year shows no significant linear response
                            AI_slope_025[i, j] = -1
                            AI_r_025[i, j] = -1
                            AI_extreme_025[i, j] = -1
                        elif slope1 < 0 or p_value1 > 0.1:  # If dry year does not respond, skip calculation directly
                            AI_slope_025[i, j] = np.nan
                            AI_r_025[i, j] = np.nan
                            AI_extreme_025[i, j] = np.nan
        return AI_slope_025, AI_r_025, AI_extreme_025, AI_slope_025_all, AI_r_025_all, AI_extreme_025_all

    elif n == 0.1:  # Size (1800, 3600)
        for i in range(1800):
            for j in range(3600):
                if np.all(np.isnan(pr_anomaly[:, i, j])) or np.all(np.isnan(NDVI_anomaly[:, i, j])):
                    AI_slope_010[i, j] = np.nan
                    AI_r_010[i, j] = np.nan
                    AI_extreme_010[i, j] = np.nan
                    AI_slope_010_all[i, j] = np.nan
                    AI_r_010_all[i, j] = np.nan
                    AI_extreme_010_all[i, j] = np.nan

                else:  # Ensure center pixel sequence is not all NaN, at least one valid value
                    test1 = pr_anomaly[:, i - 1:i + 2, j - 1:j + 2].flatten()
                    test2 = NDVI_anomaly[:, i - 1:i + 2, j - 1:j + 2].flatten()  # Extract 8-neighborhood
                    pr_purified, NDVI_purified = jinhua(test1, test2)  # Purify two arrays
                    if np.all(np.isnan(test1)) or np.all(np.isnan(test2)):  # Must check after purification, as both might become NaN after purification
                        AI_slope_010[i, j] = np.nan
                        AI_r_010[i, j] = np.nan
                        AI_extreme_010[i, j] = np.nan
                        AI_slope_010_all[i, j] = np.nan
                        AI_r_010_all[i, j] = np.nan
                        AI_extreme_010_all[i, j] = np.nan
                    else:
                        pr_sorted, NDVI_sorted = sort_by_A(pr_purified, NDVI_purified)  # After purification, now sort by moisture from low to high
                        pr_sorted, NDVI_sorted = moving_average(pr_sorted, NDVI_sorted, len(pr_sorted) // 40)
                        mid = len(pr_sorted) // 2  # Divide by median, ensuring at least 20 samples on each side
                        x1 = pr_sorted[0:mid]
                        y1 = NDVI_sorted[0:mid]
                        slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
                        x2 = pr_sorted[mid:]
                        y2 = NDVI_sorted[mid:]
                        slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
                        max_NDVI = np.mean(NDVI_sorted[-5:])  # Extract NDVI values for the highest precipitation years, last 5
                        min_NDVI = np.mean(NDVI_sorted[:5])  # Extract NDVI values for the lowest precipitation years, first 5
                        mean_NDVI = np.mean(NDVI_sorted)  # Calculate the mean of the entire sequence
                        positive = (max_NDVI - mean_NDVI)
                        negative = (mean_NDVI - min_NDVI)

                        # Direct calculation without considering significance
                        AI_slope_010_all[i, j] = (slope2 - slope1) / (slope2 + slope1)
                        AI_r_010_all[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                        AI_extreme_010_all[i, j] = (positive - negative) / (positive + negative)

                        # Start conditional judgment
                        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # If both dry and wet years respond positively and linearly significantly
                            AI_slope_010[i, j] = (slope2 - slope1) / (slope2 + slope1)
                            AI_r_010[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                            AI_extreme_010[i, j] = (positive - negative) / (positive + negative)

                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:  # If dry year responds positively and linearly significantly, but wet year responds significantly negatively
                            AI_slope_010[i, j] = -1
                            AI_r_010[i, j] = -1
                            AI_extreme_010[i, j] = -1
                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 > 0.1:  # If dry year responds positively and linearly significantly, but wet year shows no significant linear response
                            AI_slope_010[i, j] = -1
                            AI_r_010[i, j] = -1
                            AI_extreme_010[i, j] = -1
                        elif slope1 < 0 or p_value1 > 0.1:  # If dry year does not respond, skip calculation directly
                            AI_slope_010[i, j] = np.nan
                            AI_r_010[i, j] = np.nan
                            AI_extreme_010[i, j] = np.nan
        return AI_slope_010, AI_r_010, AI_extreme_010, AI_slope_010_all, AI_r_010_all, AI_extreme_010_all

def calculate_AI_24(pr_anomaly, NDVI_anomaly, n):  # 24-neighborhood, input n, adaptive to three spatial resolutions
    # Define empty variables to store results
    AI_slope_050 = np.empty((360, 720), dtype=float)
    AI_r_050 = np.empty((360, 720), dtype=float)
    AI_extreme_050 = np.empty((360, 720), dtype=float)
    AI_slope_050_all = np.empty((360, 720), dtype=float)
    AI_r_050_all = np.empty((360, 720), dtype=float)
    AI_extreme_050_all = np.empty((360, 720), dtype=float)
    AI_slope_025 = np.empty((720, 1440), dtype=float)
    AI_r_025 = np.empty((720, 1440), dtype=float)
    AI_extreme_025 = np.empty((720, 1440), dtype=float)
    AI_slope_025_all = np.empty((720, 1440), dtype=float)
    AI_r_025_all = np.empty((720, 1440), dtype=float)
    AI_extreme_025_all = np.empty((720, 1440), dtype=float)
    AI_slope_010 = np.empty((1800, 3600), dtype=float)
    AI_r_010 = np.empty((1800, 3600), dtype=float)
    AI_extreme_010 = np.empty((1800, 3600), dtype=float)
    AI_slope_010_all = np.empty((1800, 3600), dtype=float)
    AI_r_010_all = np.empty((1800, 3600), dtype=float)
    AI_extreme_010_all = np.empty((1800, 3600), dtype=float)

    if n == 0.5:  # Size (360, 720)
        for i in range(360):
            for j in range(720):
                if np.all(np.isnan(pr_anomaly[:, i, j])) or np.all(np.isnan(NDVI_anomaly[:, i, j])):  # If both center pixel sequences are all NaN
                    AI_slope_050[i, j] = np.nan
                    AI_r_050[i, j] = np.nan
                    AI_extreme_050[i, j] = np.nan
                    AI_slope_050_all[i, j] = np.nan
                    AI_r_050_all[i, j] = np.nan
                    AI_extreme_050_all[i, j] = np.nan
                else:  # Ensure center pixel sequence is not all NaN, at least one valid value
                    test1 = pr_anomaly[:, i - 2:i + 3, j - 2:j + 3].flatten()
                    test2 = NDVI_anomaly[:, i - 2:i + 3, j - 2:j + 3].flatten()  # Extract 24-neighborhood
                    pr_purified, NDVI_purified = jinhua(test1, test2)  # Purify two arrays
                    if np.all(np.isnan(test1)) or np.all(np.isnan(test2)):  # Must check after purification, as both might become NaN after purification
                        AI_slope_050[i, j] = np.nan
                        AI_r_050[i, j] = np.nan
                        AI_extreme_050[i, j] = np.nan
                        AI_slope_050_all[i, j] = np.nan
                        AI_r_050_all[i, j] = np.nan
                        AI_extreme_050_all[i, j] = np.nan
                    else:
                        pr_sorted, NDVI_sorted = sort_by_A(pr_purified, NDVI_purified)  # After purification, now sort by moisture from low to high
                        pr_sorted, NDVI_sorted = moving_average(pr_sorted, NDVI_sorted, len(pr_sorted) // 40)
                        mid = len(pr_sorted) // 2  # Divide by median, ensuring at least 20 samples on each side
                        x1 = pr_sorted[0:mid]
                        y1 = NDVI_sorted[0:mid]
                        slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
                        x2 = pr_sorted[mid:]
                        y2 = NDVI_sorted[mid:]
                        slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
                        max_NDVI = np.mean(NDVI_sorted[-5:])  # Extract NDVI values for the highest precipitation years, last 5
                        min_NDVI = np.mean(NDVI_sorted[:5])  # Extract NDVI values for the lowest precipitation years, first 5
                        mean_NDVI = np.mean(NDVI_sorted)  # Calculate the mean of the entire sequence
                        positive = (max_NDVI - mean_NDVI)
                        negative = (mean_NDVI - min_NDVI)

                        # Direct calculation without considering significance
                        AI_slope_050_all[i, j] = (slope2 - slope1) / (slope2 + slope1)
                        AI_r_050_all[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                        AI_extreme_050_all[i, j] = (positive - negative) / (positive + negative)

                        # Start conditional judgment
                        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # If both dry and wet years respond positively and linearly significantly
                            AI_slope_050[i, j] = (slope2 - slope1) / (slope2 + slope1)
                            AI_r_050[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                            AI_extreme_050[i, j] = (positive - negative) / (positive + negative)

                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:  # If dry year responds positively and linearly significantly, but wet year responds significantly negatively
                            AI_slope_050[i, j] = -1
                            AI_r_050[i, j] = -1
                            AI_extreme_050[i, j] = -1
                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 > 0.1:  # If dry year responds positively and linearly significantly, but wet year shows no significant linear response
                            AI_slope_050[i, j] = -1
                            AI_r_050[i, j] = -1
                            AI_extreme_050[i, j] = -1
                        elif slope1 < 0 or p_value1 > 0.1:  # If dry year does not respond, skip calculation directly
                            AI_slope_050[i, j] = np.nan
                            AI_r_050[i, j] = np.nan
                            AI_extreme_050[i, j] = np.nan
        return AI_slope_050, AI_r_050, AI_extreme_050, AI_slope_050_all, AI_r_050_all, AI_extreme_050_all

    elif n == 0.25:  # Size (720, 1440)
        for i in range(720):
            for j in range(1440):
                if np.all(np.isnan(pr_anomaly[:, i, j])) or np.all(np.isnan(NDVI_anomaly[:, i, j])):
                    AI_slope_025[i, j] = np.nan
                    AI_r_025[i, j] = np.nan
                    AI_extreme_025[i, j] = np.nan
                    AI_slope_025_all[i, j] = np.nan
                    AI_r_025_all[i, j] = np.nan
                    AI_extreme_025_all[i, j] = np.nan

                else:  # Ensure center pixel sequence is not all NaN, at least one valid value
                    test1 = pr_anomaly[:, i - 2:i + 3, j - 2:j + 3].flatten()
                    test2 = NDVI_anomaly[:, i - 2:i + 3, j - 2:j + 3].flatten()  # Extract 24-neighborhood
                    pr_purified, NDVI_purified = jinhua(test1, test2)  # Purify two arrays
                    if np.all(np.isnan(test1)) or np.all(np.isnan(test2)):  # Must check after purification, as both might become NaN after purification
                        AI_slope_025[i, j] = np.nan
                        AI_r_025[i, j] = np.nan
                        AI_extreme_025[i, j] = np.nan
                        AI_slope_025_all[i, j] = np.nan
                        AI_r_025_all[i, j] = np.nan
                        AI_extreme_025_all[i, j] = np.nan
                    else:
                        pr_sorted, NDVI_sorted = sort_by_A(pr_purified, NDVI_purified)  # After purification, now sort by moisture from low to high
                        mid = len(pr_sorted) // 2  # Divide by median, ensuring at least 20 samples on each side
                        x1 = pr_sorted[0:mid]
                        y1 = NDVI_sorted[0:mid]
                        slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
                        x2 = pr_sorted[mid:]
                        y2 = NDVI_sorted[mid:]
                        slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
                        max_NDVI = np.mean(NDVI_sorted[-5:])  # Extract NDVI values for the highest precipitation years, last 5
                        min_NDVI = np.mean(NDVI_sorted[:5])  # Extract NDVI values for the lowest precipitation years, first 5
                        mean_NDVI = np.mean(NDVI_sorted)  # Calculate the mean of the entire sequence
                        positive = (max_NDVI - mean_NDVI)
                        negative = (mean_NDVI - min_NDVI)

                        # Direct calculation without considering significance
                        AI_slope_025_all[i, j] = (slope2 - slope1) / (slope2 + slope1)
                        AI_r_025_all[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                        AI_extreme_025_all[i, j] = (positive - negative) / (positive + negative)

                        # Start conditional judgment
                        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # If both dry and wet years respond positively and linearly significantly
                            AI_slope_025[i, j] = (slope2 - slope1) / (slope2 + slope1)
                            AI_r_025[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                            AI_extreme_025[i, j] = (positive - negative) / (positive + negative)

                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:  # If dry year responds positively and linearly significantly, but wet year responds significantly negatively
                            AI_slope_025[i, j] = -1
                            AI_r_025[i, j] = -1
                            AI_extreme_025[i, j] = -1
                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 > 0.1:  # If dry year responds positively and linearly significantly, but wet year shows no significant linear response
                            AI_slope_025[i, j] = -1
                            AI_r_025[i, j] = -1
                            AI_extreme_025[i, j] = -1
                        elif slope1 < 0 or p_value1 > 0.1:  # If dry year does not respond, skip calculation directly
                            AI_slope_025[i, j] = np.nan
                            AI_r_025[i, j] = np.nan
                            AI_extreme_025[i, j] = np.nan
        return AI_slope_025, AI_r_025, AI_extreme_025, AI_slope_025_all, AI_r_025_all, AI_extreme_025_all

    elif n == 0.1:  # Size (1800, 3600)
        for i in range(1800):
            for j in range(3600):
                if np.all(np.isnan(pr_anomaly[:, i, j])) or np.all(np.isnan(NDVI_anomaly[:, i, j])):
                    AI_slope_010[i, j] = np.nan
                    AI_r_010[i, j] = np.nan
                    AI_extreme_010[i, j] = np.nan
                    AI_slope_010_all[i, j] = np.nan
                    AI_r_010_all[i, j] = np.nan
                    AI_extreme_010_all[i, j] = np.nan

                else:  # Ensure center pixel sequence is not all NaN, at least one valid value
                    test1 = pr_anomaly[:, i - 2:i + 3, j - 2:j + 3].flatten()
                    test2 = NDVI_anomaly[:, i - 2:i + 3, j - 2:j + 3].flatten()  # Extract 24-neighborhood
                    pr_purified, NDVI_purified = jinhua(test1, test2)  # Purify two arrays
                    if np.all(np.isnan(test1)) or np.all(np.isnan(test2)):  # Must check after purification, as both might become NaN after purification
                        AI_slope_010[i, j] = np.nan
                        AI_r_010[i, j] = np.nan
                        AI_extreme_010[i, j] = np.nan
                        AI_slope_010_all[i, j] = np.nan
                        AI_r_010_all[i, j] = np.nan
                        AI_extreme_010_all[i, j] = np.nan
                    else:
                        pr_sorted, NDVI_sorted = sort_by_A(pr_purified, NDVI_purified)  # After purification, now sort by moisture from low to high
                        mid = len(pr_sorted) // 2  # Divide by median, ensuring at least 20 samples on each side
                        x1 = pr_sorted[0:mid]
                        y1 = NDVI_sorted[0:mid]
                        slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
                        x2 = pr_sorted[mid:]
                        y2 = NDVI_sorted[mid:]
                        slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
                        max_NDVI = np.mean(NDVI_sorted[-5:])  # Extract NDVI values for the highest precipitation years, last 5
                        min_NDVI = np.mean(NDVI_sorted[:5])  # Extract NDVI values for the lowest precipitation years, first 5
                        mean_NDVI = np.mean(NDVI_sorted)  # Calculate the mean of the entire sequence
                        positive = (max_NDVI - mean_NDVI)
                        negative = (mean_NDVI - min_NDVI)

                        # Direct calculation without considering significance
                        AI_slope_010_all[i, j] = (slope2 - slope1) / (slope2 + slope1)
                        AI_r_010_all[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                        AI_extreme_010_all[i, j] = (positive - negative) / (positive + negative)

                        # Start conditional judgment
                        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # If both dry and wet years respond positively and linearly significantly
                            AI_slope_010[i, j] = (slope2 - slope1) / (slope2 + slope1)
                            AI_r_010[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                            AI_extreme_010[i, j] = (positive - negative) / (positive + negative)

                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:  # If dry year responds positively and linearly significantly, but wet year responds significantly negatively
                            AI_slope_010[i, j] = -1
                            AI_r_010[i, j] = -1
                            AI_extreme_010[i, j] = -1
                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 > 0.1:  # If dry year responds positively and linearly significantly, but wet year shows no significant linear response
                            AI_slope_010[i, j] = -1
                            AI_r_010[i, j] = -1
                            AI_extreme_010[i, j] = -1
                        elif slope1 < 0 or p_value1 > 0.1:  # If dry year does not respond, skip calculation directly
                            AI_slope_010[i, j] = np.nan
                            AI_r_010[i, j] = np.nan
                            AI_extreme_010[i, j] = np.nan
        return AI_slope_010, AI_r_010, AI_extreme_010, AI_slope_010_all, AI_r_010_all, AI_extreme_010_all

函数1：3种AI指标，单邻域，分段平均40段，自适应空间分辨率
函数2：3种AI指标，8邻域，分段平均40段，自适应空间分辨率
函数3：3种AI指标，24邻域，分段平均40段，自适应空间分辨率


In [8]:
print('Step 1: Calculate moisture anomaly, NDVI anomaly, and remove interannual trends from each')
NDVI_anomaly_025 = np.empty((41, 720, 1440))
for i in range(720):
    for j in range(1440):
        if np.isnan(NDVI_year_025[:, i, j]).any():  # Ensure the NDVI sequence has no missing values
            NDVI_anomaly_025[:, i, j] = np.nan
        else:
            y = NDVI_year_025[:, i, j]
            x = np.arange(len(y))

            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            y_pred = slope * x + intercept
            y_detrended = y - y_pred  # Remove simple linear trend

            a = y_detrended
            b = np.mean(a)  # Start calculating anomaly values, convert NDVI to anomaly values, unified with other indicators
            c = np.std(a)
            d = (a - b) / c
            NDVI_anomaly_025[:, i, j] = d

LAI_anomaly_025 = np.empty((40, 720, 1440))
for i in range(720):
    for j in range(1440):
        if np.isnan(LAI_year_025[:, i, j]).any():  # Ensure the LAI sequence has no missing values
            LAI_anomaly_025[:, i, j] = np.nan
        else:
            y = LAI_year_025[:, i, j]
            x = np.arange(len(y))

            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            y_pred = slope * x + intercept
            y_detrended = y - y_pred  # Remove simple linear trend

            a = y_detrended
            b = np.mean(a)  # Start calculating anomaly values, convert LAI to anomaly values, unified with other indicators
            c = np.std(a)
            d = (a - b) / c
            LAI_anomaly_025[:, i, j] = d

NDVI_anomaly_050 = np.empty((41, 360, 720))
for i in range(360):
    for j in range(720):
        if np.isnan(NDVI_year_050[:, i, j]).any():  # Ensure the NDVI sequence has no missing values
            NDVI_anomaly_050[:, i, j] = np.nan
        else:
            y = NDVI_year_050[:, i, j]
            x = np.arange(len(y))

            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            y_pred = slope * x + intercept
            y_detrended = y - y_pred  # Remove simple linear trend

            a = y_detrended
            b = np.mean(a)  # Start calculating anomaly values, convert NDVI to anomaly values, unified with other indicators
            c = np.std(a)
            d = (a - b) / c
            NDVI_anomaly_050[:, i, j] = d
# ----------------------------------------------------------------------------------------------------------------------------------------------------------
Terra_anomaly_025 = np.empty((41, 720, 1440))
for i in range(720):
    for j in range(1440):
        if np.isnan(Terra_year_025[:, i, j]).any():
            Terra_anomaly_025[:, i, j] = np.nan
        else:
            y = Terra_year_025[:, i, j]
            x = np.arange(len(y))

            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            y_pred = slope * x + intercept
            y_detrended = y - y_pred

            a = y_detrended
            b = np.mean(a)
            c = np.std(a)
            d = (a - b) / c
            Terra_anomaly_025[:, i, j] = d
# ----------------------------------------------------------------------------------------------------------------------------------------------------------
CRU_anomaly_025 = np.empty((41, 720, 1440))
for i in range(720):
    for j in range(1440):
        if np.isnan(CRU_year_025[:, i, j]).any():
            CRU_anomaly_025[:, i, j] = np.nan
        else:
            y = CRU_year_025[:, i, j]
            x = np.arange(len(y))

            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            y_pred = slope * x + intercept
            y_detrended = y - y_pred

            a = y_detrended
            b = np.mean(a)
            c = np.std(a)
            d = (a - b) / c
            CRU_anomaly_025[:, i, j] = d

# Special handling for CRU data
for i in range(720):
    for j in range(1440):
        if np.all(CRU_anomaly_025[:, i, j] == CRU_anomaly_025[0, i, j]):  # If all X values are the same, linear regression cannot be performed
            CRU_anomaly_025[:, i, j] = np.nan

CRU_anomaly_050 = np.empty((41, 360, 720))
for i in range(360):
    for j in range(720):
        if np.isnan(CRU_year_050[:, i, j]).any():
            CRU_anomaly_050[:, i, j] = np.nan
        else:
            y = CRU_year_050[:, i, j]
            x = np.arange(len(y))

            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            y_pred = slope * x + intercept
            y_detrended = y - y_pred

            a = y_detrended
            b = np.mean(a)
            c = np.std(a)
            d = (a - b) / c
            CRU_anomaly_050[:, i, j] = d

# Special handling for CRU data
for i in range(360):
    for j in range(720):
        if np.all(CRU_anomaly_050[:, i, j] == CRU_anomaly_050[0, i, j]):  # If all X values are the same, linear regression cannot be performed
            CRU_anomaly_050[:, i, j] = np.nan

print('Shape of NDVI_anomaly_025:', NDVI_anomaly_025.shape)
print('Shape of NDVI_anomaly_050:', NDVI_anomaly_050.shape)
print('Shape of LAI_anomaly_025:', LAI_anomaly_025.shape)
print('Shape of CRU_anomaly_025:', CRU_anomaly_025.shape)
print('Shape of CRU_anomaly_050:', CRU_anomaly_050.shape)
print('Shape of Terra_anomaly_025:', Terra_anomaly_025.shape)

步骤1：计算水分异常，NDVI异常，并且去除各自趋势


C:\Users\JYQ\AppData\Local\Temp\ipykernel_212376\4108951704.py:94: RuntimeWarning: invalid value encountered in divide
  d=(a-b)/c
C:\anaconda\lib\site-packages\numpy\core\_methods.py:180: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


NDVI_yichang_025的规格： (41, 720, 1440)
NDVI_yichang_050的规格： (41, 360, 720)
LAI_yichang_025的规格： (40, 720, 1440)
CRU_yichang_025的规格： (41, 720, 1440)
CRU_yichang_050的规格： (41, 360, 720)
Terra_yichang_025的规格： (41, 720, 1440)


In [9]:
print("Step 2: Use calculation function to compute all indicators, order as follows:")
# 0.25 spatial resolution, single neighborhood, 8-neighborhood, 24-neighborhood
AI_slope_CRU_025_1, AI_r_CRU_025_1, AI_extreme_CRU_025_1, AI_slope_CRU_025_1_all, AI_r_CRU_025_1_all, AI_extreme_CRU_025_1_all = calculate_AI_1(CRU_anomaly_025, NDVI_anomaly_025, 0.25)
AI_slope_CRU_025_8, AI_r_CRU_025_8, AI_extreme_CRU_025_8, AI_slope_CRU_025_8_all, AI_r_CRU_025_8_all, AI_extreme_CRU_025_8_all = calculate_AI_8(CRU_anomaly_025, NDVI_anomaly_025, 0.25)
AI_slope_CRU_025_24, AI_r_CRU_025_24, AI_extreme_CRU_025_24, AI_slope_CRU_025_24_all, AI_r_CRU_025_24_all, AI_extreme_CRU_025_24_all = calculate_AI_24(CRU_anomaly_025, NDVI_anomaly_025, 0.25)

# 0.5 spatial resolution, single neighborhood, 8-neighborhood, 24-neighborhood
AI_slope_CRU_050_1, AI_r_CRU_050_1, AI_extreme_CRU_050_1, AI_slope_CRU_050_1_all, AI_r_CRU_050_1_all, AI_extreme_CRU_050_1_all = calculate_AI_1(CRU_anomaly_050, NDVI_anomaly_050, 0.5)
AI_slope_CRU_050_8, AI_r_CRU_050_8, AI_extreme_CRU_050_8, AI_slope_CRU_050_8_all, AI_r_CRU_050_8_all, AI_extreme_CRU_050_8_all = calculate_AI_8(CRU_anomaly_050, NDVI_anomaly_050, 0.5)
AI_slope_CRU_050_24, AI_r_CRU_050_24, AI_extreme_CRU_050_24, AI_slope_CRU_050_24_all, AI_r_CRU_050_24_all, AI_extreme_CRU_050_24_all = calculate_AI_24(CRU_anomaly_050, NDVI_anomaly_050, 0.5)

步骤2：使用计算函数，计算出所有指标，顺序如下：


In [10]:
print('Step 3: Before plotting, convert all indicators to TIFF format, naming convention: data+resolution+neighborhood_selection+"_tif"')
# The following code converts 0.25° spatial resolution numpy arrays to xarray raster data
left_180 = np.arange(-180, 0, 0.25)
right_180 = np.arange(0.25, 180.25, 0.25)
longitude = np.concatenate((left_180, right_180))
up_90 = np.arange(90, 0, -0.25)
down_90 = np.arange(-0.25, -90.25, -0.25)
latitude = np.concatenate((up_90, down_90))

# 0.25 spatial resolution, single neighborhood
AI_slope_CRU_025_1_tif = xr.DataArray(AI_slope_CRU_025_1, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_CRU_025_1_tif = xr.DataArray(AI_r_CRU_025_1, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_CRU_025_1_tif = xr.DataArray(AI_extreme_CRU_025_1, dims=['y', 'x'], coords=[latitude, longitude])
AI_slope_CRU_025_1_all_tif = xr.DataArray(AI_slope_CRU_025_1_all, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_CRU_025_1_all_tif = xr.DataArray(AI_r_CRU_025_1_all, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_CRU_025_1_all_tif = xr.DataArray(AI_extreme_CRU_025_1_all, dims=['y', 'x'], coords=[latitude, longitude])

# 0.25 spatial resolution, 8-neighborhood
AI_slope_CRU_025_8_tif = xr.DataArray(AI_slope_CRU_025_8, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_CRU_025_8_tif = xr.DataArray(AI_r_CRU_025_8, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_CRU_025_8_tif = xr.DataArray(AI_extreme_CRU_025_8, dims=['y', 'x'], coords=[latitude, longitude])
AI_slope_CRU_025_8_all_tif = xr.DataArray(AI_slope_CRU_025_8_all, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_CRU_025_8_all_tif = xr.DataArray(AI_r_CRU_025_8_all, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_CRU_025_8_all_tif = xr.DataArray(AI_extreme_CRU_025_8_all, dims=['y', 'x'], coords=[latitude, longitude])

# 0.25 spatial resolution, 24-neighborhood
AI_slope_CRU_025_24_tif = xr.DataArray(AI_slope_CRU_025_24, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_CRU_025_24_tif = xr.DataArray(AI_r_CRU_025_24, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_CRU_025_24_tif = xr.DataArray(AI_extreme_CRU_025_24, dims=['y', 'x'], coords=[latitude, longitude])
AI_slope_CRU_025_24_all_tif = xr.DataArray(AI_slope_CRU_025_24_all, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_CRU_025_24_all_tif = xr.DataArray(AI_r_CRU_025_24_all, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_CRU_025_24_all_tif = xr.DataArray(AI_extreme_CRU_025_24_all, dims=['y', 'x'], coords=[latitude, longitude])

# The following code converts 0.5° spatial resolution numpy arrays to xarray raster data
left_180 = np.arange(-179.8, 0, 0.5)
right_180 = np.arange(0.3, 180.3, 0.5)
longitude = np.concatenate((left_180, right_180))
up_90 = np.arange(89.75, 0, -0.5)
down_90 = np.arange(-0.25, -90.25, -0.5)
latitude = np.concatenate((up_90, down_90))

# 0.5 spatial resolution, single neighborhood
AI_slope_CRU_050_1_tif = xr.DataArray(AI_slope_CRU_050_1, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_CRU_050_1_tif = xr.DataArray(AI_r_CRU_050_1, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_CRU_050_1_tif = xr.DataArray(AI_extreme_CRU_050_1, dims=['y', 'x'], coords=[latitude, longitude])
AI_slope_CRU_050_1_all_tif = xr.DataArray(AI_slope_CRU_050_1_all, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_CRU_050_1_all_tif = xr.DataArray(AI_r_CRU_050_1_all, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_CRU_050_1_all_tif = xr.DataArray(AI_extreme_CRU_050_1_all, dims=['y', 'x'], coords=[latitude, longitude])

# 0.5 spatial resolution, 8-neighborhood
AI_slope_CRU_050_8_tif = xr.DataArray(AI_slope_CRU_050_8, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_CRU_050_8_tif = xr.DataArray(AI_r_CRU_050_8, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_CRU_050_8_tif = xr.DataArray(AI_extreme_CRU_050_8, dims=['y', 'x'], coords=[latitude, longitude])
AI_slope_CRU_050_8_all_tif = xr.DataArray(AI_slope_CRU_050_8_all, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_CRU_050_8_all_tif = xr.DataArray(AI_r_CRU_050_8_all, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_CRU_050_8_all_tif = xr.DataArray(AI_extreme_CRU_050_8_all, dims=['y', 'x'], coords=[latitude, longitude])

# 0.5 spatial resolution, 24-neighborhood
AI_slope_CRU_050_24_tif = xr.DataArray(AI_slope_CRU_050_24, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_CRU_050_24_tif = xr.DataArray(AI_r_CRU_050_24, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_CRU_050_24_tif = xr.DataArray(AI_extreme_CRU_050_24, dims=['y', 'x'], coords=[latitude, longitude])
AI_slope_CRU_050_24_all_tif = xr.DataArray(AI_slope_CRU_050_24_all, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_CRU_050_24_all_tif = xr.DataArray(AI_r_CRU_050_24_all, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_CRU_050_24_all_tif = xr.DataArray(AI_extreme_CRU_050_24_all, dims=['y', 'x'], coords=[latitude, longitude])

步骤3: 在绘图之前，要先把这些指标全转换成tiff格式，命名格式为 数据+分辨率+邻域选择+ " _tif " 


In [499]:
print('步骤4_1：出图AI_slope,AI_extreme,AI_r')
fig, ax = plt.subplots(2, 2, figsize=(12, 6))
world = gpd.read_file('D:\JYQ_data\JYQ\世界地图\世界各大洲际的边界/各洲的边界线.shp')#读入我的矢量边界数据，需要替换绝对路径
# world shp
world.plot(ax=ax[0, 0], color='gray', linewidth=0.1)  
world.plot(ax=ax[0, 1], color='gray', linewidth=0.1)  
world.plot(ax=ax[1, 0], color='gray', linewidth=0.1)  

sm1 = ScalarMappable(cmap='seismic_r', norm=plt.Normalize(vmin=-1, vmax=1))
sm1.set_array([])  

AI_slope_CRU_025_8_tif.plot(ax=ax[0,0], cmap='seismic_r', vmin=-1, vmax=1, label='CRU' ,add_colorbar=False)
cbar0 = fig.colorbar(sm1, ax=ax[0,0], orientation='vertical', shrink=0.75)
AI_r_CRU_025_8_tif.plot(ax=ax[0,1], cmap='seismic_r', vmin=-1, vmax=1, label='CRU' ,add_colorbar=False)
cbar1 = fig.colorbar(sm1, ax=ax[0,1], orientation='vertical', shrink=0.75)
AI_extreme_CRU_025_8_tif.plot(ax=ax[1,0], cmap='seismic_r', vmin=-1, vmax=1, label='CRU' ,add_colorbar=False)
cbar2 = fig.colorbar(sm1, ax=ax[1,0], orientation='vertical', shrink=0.75)
ax[1,1].axis('off')  


for a in [ax[0,0], ax[0,1], ax[1,0]]:
    a.set_xticks([-180, -120, -60, 0, 60, 120, 180])  
    a.set_xticklabels(['180°W', '120°W', '60°W', '0°', '60°E', '120°E', '180°E'])
    a.set_xlim([-180, 180])  


for a in [ax[0,0], ax[0,1], ax[1,0]]:
    a.set_yticks([-90, -60, -30, 0, 30, 60, 90])  
    a.set_yticklabels(['90°S', '60°S', '30°S', '0°', '30°N', '60°N', '90°N'])
    a.set_ylim([-90, 90])  

ax[0,0].set_xlabel('')  
ax[0,0].set_ylabel('')  
ax[0,1].set_xlabel('')  
ax[0,1].set_ylabel('')  
ax[1,0].set_xlabel('')  
ax[1,0].set_ylabel('')  

# plt.savefig(r'D:\BaiduSyncdisk\毕业论文\英文论文的图\三图\结果\图1_1', dpi=1500, bbox_inches='tight')
plt.tight_layout()
plt.show()

步骤4_1：出图AI_slope,AI_extreme,AI_r


In [167]:
print("Step 4_2: Area statistics for three indicators, and mean values")
unique_values0, counts0 = np.unique(ganshi_025, return_counts=True)  # counts0 represents the number of points for each climate zone
for value, count in zip(unique_values0, counts0):  # This is used to calculate the area of different climate zones
    print(f"Value {value} appears {count} times")

步骤4_2 三种指标的面积统计，以及均值


In [327]:
print("Step 4_3: The proportion of vegetation exhibiting asymmetric response characteristics in the global coverage area")
# Number of points with positive response
array1 = AI_slope_CRU_025_8.copy()
mask1 = np.where(
    array1 > 0,  # Condition 1: >0 → 1
    1,
    np.where(
        array1 == -1,  # Condition 2: == -1 → -2
        -2,
        np.where(
            (array1 < 0) & (array1 > -1),  # Condition 3: -1 < x < 0 → -1
            -1,
            np.nan  # Other cases (x = 0 or x < -1) → NaN
        )
    )
)
# Value 1 represents positive response, -1 represents negative response, -2 represents AI indicator equal to -1
total_count = np.count_nonzero((veg == 1) | (veg == 2) | (veg == 3) | (veg == 4) | (veg == 5) | (veg == 6) | (veg == 8) | (veg == 9) | (veg == 12))
unique_values0, counts0 = np.unique(mask1, return_counts=True)  # Count how many times 1 and -1 appear
for value, count in zip(unique_values0, counts0):
    print(f"Value {value} appears {count} times")
print("Total vegetation points:", total_count)

步骤4_3 具有非对称响应特征的植被占据全球的覆盖区域的多少
数值 -2.0 出现了 23935 次
数值 -1.0 出现了 25298 次
数值 1.0 出现了 15337 次
数值 nan 出现了 972230 次
植被总点: 155891


In [500]:
print("step4_4: Pie chart statistics for step 4_3")
# Data
outer_labels = ['Negative2', 'Negative1', 'Positive', 'NR']  # Outer pie chart labels
outer_sizes = [0.1535, 0.16, 0.1, 0.5858]  # Outer pie chart data

# Colors
outer_colors = ['Purple', 'Red', 'Blue', 'gray']  # Outer colors

fig, ax = plt.subplots()

# Outer pie chart
ax.pie(outer_sizes, labels=outer_labels, radius=1, colors=outer_colors,
       wedgeprops=dict(width=0.3, edgecolor='w', linewidth=1, alpha=0.7), autopct='%1.1f%%', pctdistance=0.85, labeldistance=1.05, textprops={'fontsize': 12, 'fontweight': 'bold'})
# Set as circle
ax.set_aspect('equal')
plt.show()

步骤4_4 对4_3步骤进行饼图统计


In [501]:
print('Step 5: Correlation coefficient decreases with increasing wetness')
#Filter out areas where both dry and wet years are significant
result_r=np.empty((720, 1440))
a=AI_slope_CRU_025_8.copy()
for i in range(720):
    for j in range(1440):
        if np.isnan(a[i,j]):
            result_r[i,j]=np.nan
        else:
            x=CRU_yichang_025[:,i,j]
            y=NDVI_yichang_025[:,i,j]
            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            result_r[i,j]=r_value

ganshi_025_temp=ganshi_raw_025
result_r_mask = ~np.isnan(result_r)
ganshi_025_temp_masked= np.where(result_r_mask, ganshi_025_temp, np.nan)


a,b=jinhua(ganshi_025_temp_masked.flatten(),result_r.flatten())
a_sorted,b_sorted=sort_by_A(a,b)
a_avg,b_avg=moving_average(a_sorted, b_sorted, len(a_sorted)//150) 


fig = plt.figure(figsize=(5.5, 5))
plt.plot(a_avg,b_avg,color='red',label='R',alpha=0.9)
plt.xlabel('humidity(Arid→ Humid)')
plt.ylabel('R_value')
plt.legend(loc='upper right')
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\相关系数.png', dpi=600, bbox_inches='tight')
plt.show()            

步骤5：相关系数随着湿润度的增加而降低


In [506]:
print("Step 6: Remove cases where AI_slope = -1, draw double-layer pie chart - vegetation type explains positive/negative response")
a = AI_slope_CRU_025_8.copy()  # Prevent modification of the original array
a[a == -1] = np.nan  # First, remove areas where the original array equals -1
array = a
array[array < 0] = -1  # For easy subsequent operations with veg
array[array > 0] = 1

result = array * veg
unique_values, counts = np.unique(result, return_counts=True)
value_counts = dict(zip(unique_values, counts))
print(value_counts)

# Data
outer_labels = ['Farm', 'Grass', 'Savanna', 'Shrub', 'other', 'other', 'Shrub', 'Savanna', 'Grass', 'Farm']  # Outer pie chart labels
outer_sizes = [0.0775, 0.2469, 0.1302, 0.1006, 0.0449, 0.0636, 0.0711, 0.0636, 0.1527, 0.0489]  # Outer pie chart data

inner_labels = ['Negative', 'Positive']  # Inner pie chart data
inner_sizes = [0.6, 0.4]  # Inner pie chart data
# Colors
outer_colors = ['Orange', 'Green', 'Brown', 'pink', 'gray', '#D3D3D3', 'pink', 'Brown', 'Green', 'Orange']  # Outer colors
inner_colors = ['red', 'blue']  # Inner colors

fig, ax = plt.subplots()

# Outer pie chart
ax.pie(outer_sizes, labels=outer_labels, radius=1, colors=outer_colors,
       wedgeprops=dict(width=0.3, edgecolor='w', linewidth=1, alpha=0.7), autopct='%1.1f%%', pctdistance=0.85, labeldistance=1.05, textprops={'fontsize': 12, 'fontweight': 'bold'})
# Inner pie chart
ax.pie(inner_sizes, labels=inner_labels, radius=0.7, colors=inner_colors,
       wedgeprops=dict(width=0.3, edgecolor='w', alpha=0.7), autopct='%1.1f%%', pctdistance=0.5, labeldistance=0.35,
       textprops={'fontsize': 12, 'fontweight': 'bold'})

theta1 = 0  # Start angle of the first segment
theta2 = 360 * inner_sizes[0] / sum(inner_sizes)  # End angle of the first segment (i.e., start angle of the second segment)
center = (0, 0)  # Center of the pie chart
inner_radius = 0.7  # Radius of inner ring
outer_radius = 1.0  # Radius of outer ring

# Define extension length
extension_length = 0.2  # Length to extend outward

# Draw first dashed line (start angle)
ax.plot([center[0] + inner_radius * np.cos(theta1 * np.pi / 180), center[0] + (outer_radius + extension_length) * np.cos(theta1 * np.pi / 180)],
        [center[1] + inner_radius * np.sin(theta1 * np.pi / 180), center[1] + (outer_radius + extension_length) * np.sin(theta1 * np.pi / 180)],
        linestyle='--', color='black', linewidth=1.5)

# Draw second dashed line (end angle)
ax.plot([center[0] + inner_radius * np.cos(theta2 * np.pi / 180), center[0] + (outer_radius + extension_length) * np.cos(theta2 * np.pi / 180)],
        [center[1] + inner_radius * np.sin(theta2 * np.pi / 180), center[1] + (outer_radius + extension_length) * np.sin(theta2 * np.pi / 180)],
        linestyle='--', color='black', linewidth=1.5)
# plt.savefig(r'D:\BaiduSyncdisk\毕业论文\AI_slope统计图', dpi=1000, bbox_inches='tight')
# Set as circle
ax.set_aspect('equal')
plt.show()

步骤6：去掉AI_slope=-1的情况，绘制双层饼图-植被种类解释正负响应
{-16.0: 67, -15.0: 98, -13.0: 260, -12.0: 3148, -11.0: 1524, -10.0: 10034, -8.0: 5290, -6.0: 4086, -5.0: 189, -4.0: 194, -3.0: 65, -2.0: 277, -1.0: 23, 0.0: 16086, 1.0: 30, 2.0: 298, 3.0: 47, 4.0: 97, 5.0: 168, 6.0: 2891, 8.0: 2584, 10.0: 6204, 11.0: 761, 12.0: 1989, 13.0: 137, 15.0: 64, 16.0: 41, nan: 980148}


In [507]:
print("Step 7: Single-layer pie chart, vegetation composition where AI_slope equals -1")
b = AI_slope_CRU_025_8.copy()  # Prevent modification of the original array
mask = b == -1  # Select only the areas that originally equal -1
b_masked = np.where(mask, b, np.nan)

result2 = b_masked * veg
unique_values, counts = np.unique(result2, return_counts=True)
value_counts = dict(zip(unique_values, counts))
print(value_counts)

# Data
outer_labels = ['Savanna', 'Grass', 'Shrub', 'Farm', 'EBF', 'MF', 'Other']  # Outer pie chart labels
outer_sizes = [0.3398, 0.1880, 0.1476, 0.1151, 0.0597, 0.042, 0.1115]  # Outer pie chart data

# Colors
outer_colors = ['Orange', 'Green', 'Brown', 'pink', '#006400', '#00FFFF', 'gray']  # Outer colors

fig, ax = plt.subplots()

# Outer pie chart
ax.pie(outer_sizes, labels=outer_labels, radius=1, colors=outer_colors,
       wedgeprops=dict(width=0.3, edgecolor='w', linewidth=1, alpha=0.7), autopct='%1.1f%%', pctdistance=0.85, labeldistance=1.05, textprops={'fontsize': 12, 'fontweight': 'bold'})
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\植被类型贡献2', dpi=1000, bbox_inches='tight')
# Set as circle
ax.set_aspect('equal')
plt.show()

步骤7：单层饼图，AI_slope等于-1的植被组成部分
{-16.0: 91, -15.0: 88, -13.0: 297, -12.0: 2668, -11.0: 1047, -10.0: 4500, -8.0: 8132, -6.0: 3532, -5.0: 1005, -4.0: 491, -3.0: 311, -2.0: 1430, -1.0: 234, -0.0: 109, nan: 1012865}


In [521]:
print("Step 8: Spatial scatter density plots for 4 vegetation types (Grassland, Shrub, Cropland, Savanna)")
result = veg * mask1  # mask1 was mentioned earlier
temp_ai = AI_slope_CRU_025_8.copy()
temp_ai = np.where(temp_ai == -1, np.nan, temp_ai)  # Remove areas where the value is -1

# Extract areas where result equals 6 and -6 (Grassland), and retrieve corresponding precipitation and AI indicators
mask_grass = (result == 6) | (result == -6)
CRU_year_mean_temp2 = CRU_year_mean.copy()
AI_slope_CRU_025_8_temp2 = temp_ai.copy()
mask_grass_precip = np.where(mask_grass, CRU_year_mean_temp2, np.nan)
mask_grass_AI_slope = np.where(mask_grass, AI_slope_CRU_025_8_temp2, np.nan)
a, b = jinhua(mask_grass_precip.flatten(), mask_grass_AI_slope.flatten())
a_sorted, b_sorted = sort_by_A(a, b)
a_avg1, b_avg1 = moving_average(a_sorted, b_sorted, len(a_sorted) // 150)  # Start piecewise averaging

# Note variable names: I copied and pasted without changing variable names, but variables are overwritten each time, which does not affect the result
mask_shrub = (result == 8) | (result == -8)
CRU_year_mean_temp2 = CRU_year_mean.copy()
AI_slope_CRU_025_8_temp2 = temp_ai.copy()
mask_shrub_precip = np.where(mask_shrub, CRU_year_mean_temp2, np.nan)
mask_shrub_AI_slope = np.where(mask_shrub, AI_slope_CRU_025_8_temp2, np.nan)
a, b = jinhua(mask_shrub_precip.flatten(), mask_shrub_AI_slope.flatten())
a_sorted, b_sorted = sort_by_A(a, b)
a_avg2, b_avg2 = moving_average(a_sorted, b_sorted, len(a_sorted) // 150)  # Start piecewise averaging

mask_savanna = (result == 10) | (result == -10)
CRU_year_mean_temp2 = CRU_year_mean.copy()
AI_slope_CRU_025_8_temp2 = temp_ai.copy()
mask_savanna_precip = np.where(mask_savanna, CRU_year_mean_temp2, np.nan)
mask_savanna_AI_slope = np.where(mask_savanna, AI_slope_CRU_025_8_temp2, np.nan)
a, b = jinhua(mask_savanna_precip.flatten(), mask_savanna_AI_slope.flatten())
a_sorted, b_sorted = sort_by_A(a, b)
a_avg3, b_avg3 = moving_average(a_sorted, b_sorted, len(a_sorted) // 150)  # Start piecewise averaging

mask_crop = (result == 12) | (result == -12)
CRU_year_mean_temp2 = CRU_year_mean.copy()
AI_slope_CRU_025_8_temp2 = temp_ai.copy()
mask_crop_precip = np.where(mask_crop, CRU_year_mean_temp2, np.nan)
mask_crop_AI_slope = np.where(mask_crop, AI_slope_CRU_025_8_temp2, np.nan)
a, b = jinhua(mask_crop_precip.flatten(), mask_crop_AI_slope.flatten())
a_sorted, b_sorted = sort_by_A(a, b)
a_avg4, b_avg4 = moving_average(a_sorted, b_sorted, len(a_sorted) // 150)  # Start piecewise averaging

fig, axs = plt.subplots(2, 2, figsize=(8, 8))  # 2 rows, 2 columns
sns.kdeplot(x=a_avg1, y=b_avg1, cmap="Blues", fill=True, ax=axs[0, 0])
sns.kdeplot(x=a_avg2, y=b_avg2, cmap="Blues", fill=True, ax=axs[0, 1])
sns.kdeplot(x=a_avg3, y=b_avg3, cmap="Blues", fill=True, ax=axs[1, 0])
sns.kdeplot(x=a_avg4, y=b_avg4, cmap="Blues", fill=True, ax=axs[1, 1])
labels = ['a', 'b', 'c', 'd']
for i, ax in enumerate(axs.flat):
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1)
    ax.set_xlim(0, 1000)
    ax.set_xlabel('Pr(mm/year)')
    ax.set_ylabel('AI_slope')
    ax.text(0.05, 0.95, labels[i], transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='top')
#     ax.set_title(titles[i])  # Dynamically set titles

axs[0, 0].set_title('Shrub')
axs[0, 1].set_title('Savanna')
axs[1, 0].set_title('Grass')
axs[1, 1].set_title('Farm')
plt.tight_layout()
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\植被统计直方图', dpi=1000, bbox_inches='tight')
plt.show()

步骤8：4种植被的空间散点密度图（草地，灌木，农田，稀树草原）


In [522]:
print('Step 9: Frequency density histograms (only retaining areas where all three have values) and box plots for three indicators')
AI_extreme_CRU_025_8[AI_extreme_CRU_025_8 < -1] = np.nan
AI_extreme_CRU_025_8[AI_extreme_CRU_025_8 > 1] = np.nan

# Create mask (retain intersection of three indicators)
mask = ~np.isnan(AI_slope_CRU_025_8) & ~np.isnan(AI_r_CRU_025_8) & ~np.isnan(AI_extreme_CRU_025_8)
AI_slope_masked = np.where(mask, AI_slope_CRU_025_8, np.nan)
AI_r_masked = np.where(mask, AI_r_CRU_025_8, np.nan)
AI_extreme_masked = np.where(mask, AI_extreme_CRU_025_8, np.nan)

# Prepare box plot data (original code part 2)
a = AI_slope_CRU_025_8.flatten()
b = AI_r_CRU_025_8.flatten()
c = AI_extreme_CRU_025_8.flatten()
a_no_nan = a[~np.isnan(a)]
b_no_nan = b[~np.isnan(b)]
c_no_nan = c[~np.isnan(c)]

# --- Plotting section ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))  # 1 row, 2 columns
plt.rcParams['axes.linewidth'] = 1  # Default is usually 1.0, set to 0.5 for thinner lines
# ---- Left subplot: KDE plot ----
sns.kdeplot(AI_slope_masked.flatten(), fill=True, color='red', alpha=0.1, label=r'$\mathrm{AI}_{\mathrm{S}}$', clip=(-1, 1), ax=ax1)
sns.kdeplot(AI_r_masked.flatten(), fill=True, color='blue', alpha=0.1, label=r'$\mathrm{AI}_{\mathrm{R}}$', clip=(-1, 1), ax=ax1)
sns.kdeplot(AI_extreme_masked.flatten(), fill=True, color='green', alpha=0.1, label=r'$\mathrm{AI}_{\mathrm{E}}$', clip=(-1, 1), ax=ax1)
ax1.set_xlabel('Asymmetric Index (AI)')
ax1.legend(loc='upper right')
# ax1.text(0.02, 0.98, '(a)', transform=ax1.transAxes, fontsize=14, fontweight='bold', va='top')

# ---- Right subplot: Box plot (add mean values) ----
datasets = [a_no_nan, b_no_nan, c_no_nan]
boxplot = ax2.boxplot(datasets, patch_artist=True)

# Set box colors
colors = ['lightpink', 'lightblue', 'lightgreen']
for patch, color in zip(boxplot['boxes'], colors):
    patch.set_facecolor(color)

# Add mean value markers
for i, data in enumerate(datasets):
    # Calculate mean
    mean_val = np.mean(data)
    # Draw mean line (red dashed line)
    ax2.hlines(mean_val, xmin=i+0.8, xmax=i+1.2, colors='red', linestyles='dashed', linewidth=2, label='Mean' if i==0 else "")
    # Add mean value label
    ax2.text(i+1, mean_val, f'{mean_val:.2f}', ha='center', va='bottom', color='red')

ax2.set_title('Boxplot of three indicators')
ax2.set_ylabel('AI')
ax2.set_xticks(range(1, len(datasets)+1), ['AI$S$', 'AI$R$', 'AI$E$'])
# ax2.text(0.02, 0.98, '(b)', transform=ax2.transAxes, fontsize=14, fontweight='bold', va='top')

# Add legend (only show "Mean" label once)
handles, labels = ax2.get_legend_handles_labels()
ax2.legend(handles, labels, loc='upper right')
# plt.savefig(r'D:\BaiduSyncdisk\毕业论文\周杰要的图\直方图3', dpi=1500, bbox_inches='tight')
plt.tight_layout()
plt.show()

步骤9：三种指标的频率密度直方图（只保留三者都有值的地方），还有箱线图


In [550]:
print('Step 10: Gradient plot of AI variation with humidity')
from scipy.interpolate import UnivariateSpline
from scipy.signal import savgol_filter
# Need to remove areas containing -1
AI_slope_CRU_025_8_temp = AI_slope_CRU_025_8.copy()
AI_r_CRU_025_8_temp = AI_r_CRU_025_8.copy()
AI_extreme_CRU_025_8_temp = AI_extreme_CRU_025_8.copy()  # Create temporary variables

AI_slope_CRU_025_8_temp[AI_slope_CRU_025_8_temp == -1] = np.nan
AI_r_CRU_025_8_temp[AI_r_CRU_025_8_temp == -1] = np.nan
AI_extreme_CRU_025_8_temp[AI_extreme_CRU_025_8_temp == -1] = np.nan  # Remove areas with value -1

AI_slope_purified, humidity_purified1 = jinhua(AI_slope_CRU_025_8_temp.flatten(), ganshi_raw_025.flatten())
humidity_sorted1, AI_slope_sorted = sort_by_A(humidity_purified1, AI_slope_purified)
humidity_avg1, AI_slope_avg = moving_average(humidity_sorted1, AI_slope_sorted, len(AI_slope_sorted) // 50)
AI_slope_avg_smooth = savgol_filter(AI_slope_avg, 8, 2)

AI_r_purified, humidity_purified1 = jinhua(AI_r_CRU_025_8_temp.flatten(), ganshi_raw_025.flatten())
humidity_sorted1, AI_r_sorted = sort_by_A(humidity_purified1, AI_r_purified)
humidity_avg2, AI_r_avg = moving_average(humidity_sorted1, AI_r_sorted, len(AI_r_sorted) // 50)
AI_r_avg_smooth = savgol_filter(AI_r_avg, 8, 2)

AI_extreme_purified, humidity_purified1 = jinhua(AI_extreme_CRU_025_8_temp.flatten(), ganshi_raw_025.flatten())
humidity_sorted1, AI_extreme_sorted = sort_by_A(humidity_purified1, AI_extreme_purified)
humidity_avg3, AI_extreme_avg = moving_average(humidity_sorted1, AI_extreme_sorted, len(AI_extreme_sorted) // 50)
AI_extreme_avg_smooth = savgol_filter(AI_extreme_avg, 8, 2)

# Calculate humidity_sorted when AI = -1
AI_slope_CRU_025_8_temp = AI_slope_CRU_025_8.copy()  # Create temporary variable
AI_slope_CRU_025_8_temp = np.where(
    AI_slope_CRU_025_8_temp == -1,  # Condition: equals -1
    AI_slope_CRU_025_8_temp,        # If True, keep original value
    np.nan                          # If False, set to nan
)
AI_slope_purified4, humidity_purified4 = jinhua(AI_slope_CRU_025_8_temp.flatten(), ganshi_raw_025.flatten())
humidity_sorted4, AI_slope_sorted4 = sort_by_A(humidity_purified4, AI_slope_purified4)

# Create figure
fig = plt.figure(figsize=(12, 6.2))
ax1 = plt.gca()  # Get current axis object
ax2 = ax1.twinx()  # Create a second axis object sharing X-axis but with independent Y-axis
plt.xlim(0, 1.2)
ax1.tick_params(axis='both', labelsize=30, length=15, width=3)
ax2.tick_params(axis='both', labelsize=30, length=15, width=3)

# Plot histogram (using left Y-axis)
counts, bins, patches = ax1.hist(humidity_sorted1, bins='auto', density=True, alpha=0.75, color='skyblue', rwidth=0.9, label='AI≠-1')  # Input required
counts, bins, patches = ax1.hist(humidity_sorted4, bins='auto', density=True, alpha=0.75, color='coral', rwidth=0.9, label='AI=-1')  # Input required
ax1.set_ylabel('Frequency Density', color='black', fontsize=28)
ax1.tick_params(axis='y', labelcolor='black')
# ax1.set_xlabel('Aridity index', fontsize=28)
# ax1.legend(loc='upper left', fontsize=28)  # Show histogram legend
# Plot line graph (using right Y-axis)
ax2.plot(humidity_avg1, AI_slope_avg_smooth, label="AI$_S$", color='red', linewidth=3)
ax2.plot(humidity_avg1, AI_r_avg_smooth, label="AI$_R$", color='blue', linewidth=3)
ax2.plot(humidity_avg1, AI_extreme_avg_smooth, label="AI$_E$", color='green', linewidth=3)
ax2.set_ylabel('AI', fontsize=30)
# ax2.legend(loc='upper right', fontsize=28)
ax2.axhline(y=0, color='black', linestyle='--', linewidth=1.5)
# Thicken the four borders
# Merge legends
handles1, labels1 = ax1.get_legend_handles_labels()  # Get histogram legend handles and labels
handles2, labels2 = ax2.get_legend_handles_labels()  # Get line graph legend handles and labels

# Display uniformly on ax1 or ax2 (recommend ax1)
ax1.legend(handles1 + handles2, labels1 + labels2,
           loc='upper right', fontsize=24, frameon=True)

# Hide separate legend for ax2 (avoid duplication)
ax2.legend().set_visible(False)
for spine in ax1.spines.values():
    spine.set_linewidth(3)  # Here 2.5 is line width, you can adjust it, e.g., to 3, 4, etc.

for spine in ax2.spines.values():
    spine.set_linewidth(3)  # Thicken borders for dual axes too!

# Add vertical dashed lines
for point in mark_points:
    ax2.axvline(x=point, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
plt.savefig(r'D:\BaiduSyncdisk\蒋奕秋2022112587\三图\梯度图2', dpi=1500, bbox_inches='tight')
plt.tight_layout()
plt.show()

步骤10：AI随着湿度变化的的梯度图


In [549]:
print('Follow-up statistical step for step 10: Calculate percentage for each climate zone')
temp = ganshi_025.copy()
arr = AI_slope_CRU_025_8.copy()
result = np.where((arr == -1) | np.isnan(arr), arr, 1)
result2 = result * temp
unique_values0, counts0 = np.unique(result2, return_counts=True)
for value, count in zip(unique_values0, counts0):
    print(f"Value {value} appears {count} times")

步骤10的后续统计步骤：统计每个气候区的百分比
数值 -5.0 出现了 11972 次
数值 -4.0 出现了 4928 次
数值 -3.0 出现了 6103 次
数值 -2.0 出现了 907 次
数值 -1.0 出现了 17 次
数值 0.0 出现了 14 次
数值 1.0 出现了 15 次
数值 2.0 出现了 14489 次
数值 3.0 出现了 20638 次
数值 4.0 出现了 5914 次
数值 5.0 出现了 15583 次
数值 nan 出现了 956220 次


In [329]:
print("Step 11: Fitting plot for areas with aridity between 0.3-0.7")
mask_03_07 = (ganshi_raw_025 >= 0.3) & (ganshi_raw_025 <= 0.7)  # Set mask for aridity between 0.3-0.7
mask_03_07_expanded = np.broadcast_to(mask_03_07, CRU_anomaly_025.shape)  # Expand mask to same shape for application
CRU_anomaly_025_mask_03_07 = np.where(mask_03_07_expanded, CRU_anomaly_025, np.nan)  # Select only points with aridity between 0.3-0.7
NDVI_anomaly_025_mask_03_07 = np.where(mask_03_07_expanded, NDVI_anomaly_025, np.nan)  # Selection completed

# Spatial averaging
pr_mask_03_07 = np.nanmean(CRU_anomaly_025_mask_03_07, axis=(1, 2))
NDVI_mask_03_07 = np.nanmean(NDVI_anomaly_025_mask_03_07, axis=(1, 2))
# Apply piecewise averaging, as there are too many points
pr_purified, NDVI_purified = jinhua(CRU_anomaly_025_mask_03_07.flatten(), NDVI_anomaly_025_mask_03_07.flatten())
pr_sorted, NDVI_sorted = sort_by_A(pr_purified, NDVI_purified)
pr_avg, NDVI_avg = moving_average(pr_sorted, NDVI_sorted, len(pr_sorted) // 100)
# Draw spatial scatter plot
split_arrays_pro(pr_avg, NDVI_avg, 'Aridity ranging from 0.3 to 0.7')

步骤11：干燥度在0.3-0.7的地方的拟合图


(0.20017088218104334, 0.06354539788087976, 0.14443121819654212)

In [330]:
print('Step 12: US-Ne2 site data')  # Already extracted in Excel beforehand
x = [1261.325, 743.114, 891.798, 791.464, 863.854, 834.842, 1120.14, 1074.547, 671.322, 915.416, 805.18, 756.666, 652.637]  # Precipitation
y = [3761.87, 2154.586, 3640.71, 1887.206, 3254.02, 2199.407, 3673.28, 1576.131, 3912.82, 2916.22, 3370.4, 3298.45, 571.586]  # Vegetation GPP
a, b = sort_by_A(x, y)
# Calculate normalized precipitation and vegetation, then plot according to time series
normalized_x = (x - np.min(x)) / (np.max(x) - np.min(x))
normalized_y = (y - np.min(y)) / (np.max(y) - np.min(y))
time = generate_year_sequence(2001, 2013)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))  # 1 row, 2 columns
ax1.plot(a, b, 'o--', color='black', label='Data Points and Trend Line')
ax1.set_title('US-Ne2')
ax1.set_xlabel('Pr(mm)')
ax1.set_ylabel('GPP')

ax2.plot(time, normalized_x, 'o--', color='blue', label='Normalized Precipitation')
ax2.plot(time, normalized_y, 'o--', color='red', label='Normalized GPP')
ax2.set_title('US-Ne2')  # Set title for second subplot
ax2.set_xlabel('time')  # Set x-axis label
ax2.legend()  # Add legend
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\US-NE2站点图', dpi=1000, bbox_inches='tight')
plt.show()

步骤12：US-Ne2站点数据


In [537]:
print('Step 13: Pairwise comparison plots for AI_slope, AI_r, AI_extreme, and correlation coefficient matrix')
import statsmodels.api as sm  # Add at the beginning of the file
# Retain common parts of the three arrays
A = AI_slope_CRU_025_8.copy()
B = AI_r_CRU_025_8.copy()
C = AI_extreme_CRU_025_8.copy()
# Remove all values equal to -1
A[A == -1] = np.nan
A[A == 0] = np.nan
B[B == -1] = np.nan
B[B == 0] = np.nan
C[C == -1] = np.nan
C[C == 0] = np.nan
valid_mask = ~np.isnan(A) & ~np.isnan(C) & ~np.isnan(B)
A_valid = A[valid_mask]  # Note: A_valid now becomes a 1D array using mask functionality
B_valid = B[valid_mask]
C_valid = C[valid_mask]

# Create DataFrame
data = pd.DataFrame({
    'AI_slope': A_valid,
    'AI_r': B_valid,
    'AI_extreme': C_valid
})
# Generate correlation coefficient matrix results
corr_matrix = data.corr()
sns.set_theme(style="ticks", font_scale=0.9)

# Create PairGrid object, customize diagonal and off-diagonal plots
g = sns.PairGrid(data, diag_sharey=False, height=2.5)
# Diagonal: Kernel density plot
g.map_diag(sns.kdeplot, fill=True, alpha=0.8, color='#1f77b4',
           edgecolor='black', linewidth=0.5, bw_adjust=0.8)
# Off-diagonal: 2D kernel density contours or Hexbin heatmap
g.map_upper(sns.kdeplot, cmap='Blues', fill=True, levels=10, alpha=0.7)
g.map_lower(sns.histplot, cmap='Reds', cbar=True,
            bins=50, alpha=0.9, edgecolor='gray', linewidth=0.1)

# Add fitted lines and equations (only in lower-left triangle region)
for i in range(len(data.columns)):
    for j in range(len(data.columns)):
        if i > j:  # Add regression lines only in lower triangle region
            ax = g.axes[i, j]
            x = data[data.columns[j]]
            y = data[data.columns[i]]
            sns.regplot(x=x, y=y, scatter=False,
                        line_kws={'color': 'darkred', 'lw': 1.5}, ax=ax)
            model = sm.OLS(y, sm.add_constant(x)).fit()
            equation = f"ρ={corr_matrix.iloc[i, j]:.2f}\ny={model.params[1]:.2f}x+{model.params[0]:.2f}"
            # Modify text parameters: white background box + black text + increased font size
            ax.text(0.05, 0.85, equation, transform=ax.transAxes,
                    fontsize=10, color='black', ha='left',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.8))

# Optimize layout and annotations
plt.suptitle("Cross-Data Comparison (Density & Hexbin)", y=1.02, fontsize=11)
g.fig.subplots_adjust(wspace=0.1, hspace=0.1)
[ax.set_xticks([]) if i % 3 != 0 else None for i, ax in enumerate(g.axes.flat)]  # Reduce tick density

# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\不同数据源的时间趋势的比较', dpi=1000, bbox_inches='tight')
plt.show()

步骤13：AI_slope,AI_r,AI_extreme两两比对图，以及相关系数矩阵


In [540]:
print('Step 14: Overlay display of 2D distribution plots')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.patches import Patch

# Create DataFrame, data from Step 13 above
data = pd.DataFrame({
    'AI_slope': A_valid,
    'AI_r': B_valid,
    'AI_extreme': C_valid
})

# Create figure and axes
fig, ax = plt.subplots(figsize=(8, 8))
# First group: AI_slope vs AI_r (blue)
sns.kdeplot(data=data, x='AI_slope', y='AI_r', cmap='Blues', fill=False, ax=ax, linewidths=1.5)
x1 = data['AI_slope'].values
y1 = data['AI_r'].values
slope1, intercept1, r1, p1, _ = stats.linregress(x1, y1)
r_squared1 = r1 ** 2
x_fit1 = np.linspace(min(x1), max(x1), 100)
y_fit1 = slope1 * x_fit1 + intercept1
ax.plot(x_fit1, y_fit1, color='blue', linestyle='--', linewidth=2.5)
ax.text(0.05, 0.95,
        rf'$\mathrm{{AI}}_{{\mathrm{{S}}}}-\mathrm{{AI}}_{{\mathrm{{R}}}}$: '
        rf'y = {slope1:.2f}x + {intercept1:.2f},  $R^2$ = {r_squared1:.2f}',
        transform=ax.transAxes, fontsize=14,
        bbox=dict(facecolor='white', alpha=1), color='blue')

# Second group: AI_slope vs AI_extreme (red)
sns.kdeplot(data=data, x='AI_slope', y='AI_extreme', cmap='Reds', fill=False, ax=ax, linewidths=1.5)
x2 = data['AI_slope'].values
y2 = data['AI_extreme'].values
slope2, intercept2, r2, p2, _ = stats.linregress(x2, y2)
r_squared2 = r2 ** 2
x_fit2 = np.linspace(min(x2), max(x2), 100)
y_fit2 = slope2 * x_fit2 + intercept2
ax.plot(x_fit2, y_fit2, color='red', linestyle='--', linewidth=2.5)
ax.text(0.05, 0.9,
        rf'$\mathrm{{AI}}_{{\mathrm{{S}}}}-\mathrm{{AI}}_{{\mathrm{{E}}}}$: '
        rf'y = {slope2:.2f}x + {intercept2:.2f},  $R^2$ = {r_squared2:.2f}',
        transform=ax.transAxes, fontsize=15,
        bbox=dict(facecolor='white', alpha=1), color='red')

# Third group: AI_r vs AI_extreme (green)
sns.kdeplot(data=data, x='AI_r', y='AI_extreme', cmap='Greens', fill=False, ax=ax, linewidths=1.5)
x3 = data['AI_r'].values
y3 = data['AI_extreme'].values
slope3, intercept3, r3, p3, _ = stats.linregress(x3, y3)
r_squared3 = r3 ** 2
x_fit3 = np.linspace(min(x3), max(x3), 100)
y_fit3 = slope3 * x_fit3 + intercept3
ax.plot(x_fit3, y_fit3, color='green', linestyle='--', linewidth=2.5)
ax.text(0.05, 0.85,
        rf'$\mathrm{{AI}}_{{\mathrm{{R}}}}-\mathrm{{AI}}_{{\mathrm{{E}}}}$: '
        rf'y = {slope3:.2f}x , $R^2$ = {r_squared3:.2f}',
        transform=ax.transAxes, fontsize=16,
        bbox=dict(facecolor='white', alpha=1), color='green')

# Set axis labels
ax.set_xlabel("AI", fontsize=20)
ax.set_ylabel("AI", fontsize=20)

# Set axis tick font size
ax.tick_params(axis='both', labelsize=19, length=10)

# Thicken borders
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.tight_layout()
# Save image (optional)
# plt.savefig(r'D:\BaiduSyncdisk\蒋奕秋2022112587\三图\图2', dpi=1500, bbox_inches='tight')
plt.show()


步骤14:二维分布图的重叠显示


In [334]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
print("Step 15: Spatial fitting curves for 14 sites")


# Define function to calculate R-squared
def calculate_r_squared(y_actual, y_predicted):
    ss_res = np.sum((y_actual - y_predicted) ** 2)  # Residual sum of squares
    ss_tot = np.sum((y_actual - np.mean(y_actual)) ** 2)  # Total sum of squares
    return 1 - (ss_res / ss_tot)


# Data
a1 = [1714.532, 1477.597, 1865, 1587.653, 1112.003, 1736.474, 2054.633, 1934.86, 1494.392, 1622.39, 2287.404, 1588.314, 1731.419, 1808.966]
b1 = [1507.668, 3503.49, 3751.91, 3342, 3524.99, 3236.57, 3553.75, 3625.65, 4312.21, 4861.66, 4181.17, 4250.87, 4389.51, 4406.04]
sai_a1 = (np.array(a1) - np.mean(a1)) / np.std(a1)
sai_b1 = (np.array(b1) - np.mean(b1)) / np.std(b1)

a3 = [569.284, 182.427, 411.478, 385.11, 660.086, 1143.884, 1008.333, 583.866, 512.759, 492.825]
b3 = [926.276, 109.5531, 361.538, 373.538, 352.976, 325.17, 392.986, 495.628, 246.8373, 414.747]
sai_a3 = (np.array(a3) - np.mean(a3)) / np.std(a3)
sai_b3 = (np.array(b3) - np.mean(b3)) / np.std(b3)

a5 = [557.697, 901.3, 592.715, 486, 1018.224, 743.033, 658.9, 582.388, 499.865, 419.016]
b5 = [1081.44, 1590.56, 939.76, 1161.01, 2026.26, 1811.64, 1440.11, 1568.67, 980.21, 1553.20]
sai_a5 = (np.array(a5) - np.mean(a5)) / np.std(a5)
sai_b5 = (np.array(b5) - np.mean(b5)) / np.std(b5)

a7 = [248.489, 308.726, 283.452, 557.022, 728.058, 508.927, 384.618, 539.254, 680.649, 408.055, 703.971, 516.896, 450.434]
b7 = [2826.8, 2872.77, 3247.35, 3156.42, 3280.12, 3138.29, 3273.77, 3383.21, 3432.64, 3519.51, 3706.1, 3687.72, 3870.61]
sai_a7 = (np.array(a7) - np.mean(a7)) / np.std(a7)
sai_b7 = (np.array(b7) - np.mean(b7)) / np.std(b7)

a8 = [1261.325, 743.114, 891.798, 791.464, 863.854, 834.842, 1120.14, 1074.547, 671.322, 915.416, 805.18, 756.666, 652.637]
b8 = [3761.81, 2133.355, 3602.25, 1825.374, 3251.82, 2116.884, 3656.58, 1589.483, 3940.1, 2919.73, 3375.47, 3275.79, 598.804]
sai_a8 = (np.array(a8) - np.mean(a8)) / np.std(a8)
sai_b8 = (np.array(b8) - np.mean(b8)) / np.std(b8)

a9 = [285.496, 334.522, 288.544, 329.94, 401.571, 229.612, 404.622, 404.364, 307.078, 319.899, 359.416]
b9 = [533.543, 717.658, 538.089, 631.995, 855.295, 325.71, 760.394, 609.816, 721.783, 530.947, 662.313]
sai_a9 = (np.array(a9) - np.mean(a9)) / np.std(a9)
sai_b9 = (np.array(b9) - np.mean(b9)) / np.std(b9)

a10 = [697.401, 491.644, 516.493, 502.466, 487.601, 794.668, 737.108, 390.398, 376.932, 547.116, 706.028, 684.02, 804.672, 202.172, 645.926]
b10 = [2128.42, 1572.56, 1250.67, 1763.48, 1240.26, 2063.80, 1236.42, 1302.37, 687.79, 1351.46, 1453.79, 1323.61, 1234.17, 908.79, 1190.59]
sai_a10 = (np.array(a10) - np.mean(a10)) / np.std(a10)
sai_b10 = (np.array(b10) - np.mean(b10)) / np.std(b10)

a11 = [735.173, 661.83, 640.974, 484.864, 612.936, 820.215, 673.46, 641.139, 685.132, 646.507, 584.801, 652.165, 962.19, 739.915, 1054.115, 1035.155]
b11 = [1714.468, 1754.28, 1925.221, 1618.136, 1688.974, 1831.93, 1654.379, 1777.91, 1840.414, 1708.065, 1726.39, 1651.422, 1779.106, 1881.724, 1735.334, 1805.364]
sai_a11 = (np.array(a11) - np.mean(a11)) / np.std(a11)
sai_b11 = (np.array(b11) - np.mean(b11)) / np.std(b11)

a12 = [289.642, 162.26, 274.193, 312.928, 311.912, 246.253, 335.534, 290.322, 314.96, 263.652, 414.528]
b12 = [523.902, 236.4777, 389.46, 574.954, 588.334, 508.035, 761.582, 393.002, 695.93, 454.054, 732.669]
sai_a12 = (np.array(a12) - np.mean(a12)) / np.std(a12)
sai_b12 = (np.array(b12) - np.mean(b12)) / np.std(b12)

a13 = [1159.206, 596.169, 274.062, 256.238, 695.295, 453.461, 566.702, 297.275, 327.731, 501.871, 365.368, 383.311, 536.275, 321.31]
b13 = [2807.64, 2656.61, 919.569, 1249.575, 2893.62, 1588.334, -162.9782, 694.089, 603.629, 3521.44, 2889.54, 2408.94, 3478.68, 4082.6]
sai_a13 = (np.array(a13) - np.mean(a13)) / np.std(a13)
sai_b13 = (np.array(b13) - np.mean(b13)) / np.std(b13)

a14 = [425.564, 516.493, 524.5, 487.596, 782.288, 701.292, 402.848, 370.582, 572.774, 713.02, 645.92, 738.13, 150.464, 608.328]
b14 = [1262.92, 1731.867, 1953.29, 1749.643, 2375.39, 1731.583, 1712.178, 1425.592, 1836.801, 2054.699, 2219.17, 1632.549, 1343.329, 1491.369]
sai_a14 = (np.array(a14) - np.mean(a14)) / np.std(a14)
sai_b14 = (np.array(b14) - np.mean(b14)) / np.std(b14)

# Combine all data
combined_a = np.concatenate([sai_a1, sai_a3, sai_a5, sai_a7, sai_a8, sai_a9, sai_a10, sai_a11, sai_a12, sai_a13, sai_a14])
combined_b = np.concatenate([sai_b1, sai_b3, sai_b5, sai_b7, sai_b8, sai_b9, sai_b10, sai_b11, sai_b12, sai_b13, sai_b14])

# Polynomial fitting (e.g., 5th degree polynomial)
degree = 5  # Can adjust the degree
coefficients = np.polyfit(combined_a, combined_b, degree)
poly_func = np.poly1d(coefficients)  # Create polynomial function

# Calculate fitted values
combined_a_fit = np.linspace(min(combined_a), max(combined_a), 500)
combined_b_fit = poly_func(combined_a_fit)

# Calculate R-squared
b_pred = poly_func(combined_a)
r_squared = calculate_r_squared(combined_b, b_pred)

# Plot graph
plt.figure(figsize=(10, 6))
plt.scatter(combined_a, combined_b, color='blue', label='Sites', alpha=0.6)
plt.plot(combined_a_fit, combined_b_fit, color='red', label=f'Polynomial Fit (Degree={degree}, R²={r_squared:.2f})')
plt.xlabel("SAI_PR", fontsize=12)
plt.ylabel("SAI_GPP", fontsize=12)
plt.legend()
plt.title("Polynomial Fit of All Sites", fontsize=14)
plt.grid(True)
plt.show()

步骤15：14个站点的空间拟合曲线


In [335]:
print('Step 16: Distribution plots for three sites with large precipitation differences')
# Define first dataset
shi_a1 = np.array([1101.7, 1226.5, 1326.1, 1077.4, 1469.6, 932.7, 954.9, 1027.2, 1082.8, 803.444, 1006.682, 1312.8, 1175.4, 1484.2, 1144.3, 1190, 1626.1, 1329.1, 1161.2, 1634.7])
shi_b1 = np.array([2310.16, 2764.21, 2679.06, 2608.09, 2614.77, 2828.69, 2459.91, 2687.64, 2959.3, 3256.45, 3062.69, 3473.11, 3487.06, 2646.1, 3328.56, 3378.25, 3273.18, 3189.93, 4053.78, 3503.99])

# Define second dataset
shi_a2 = np.array([289.642, 162.26, 274.193, 312.928, 311.912, 246.253, 335.534, 290.322, 314.96, 263.652, 414.528])
shi_b2 = np.array([523.902, 236.4777, 389.46, 574.954, 588.334, 508.035, 761.582, 393.002, 695.93, 454.054, 732.669])

shi_a3 = np.array([3072.2, 3509.6, 3550.48, 2838, 3118.4, 3221.2, 3147.8, 3300.8, 2966.538, 2734.4])
shi_b3 = np.array([7318.76, 6990.64, 7206.23, 7090.36, 7351.53, 7842.14, 8026.51, 7755.6, 7187.98, 6408.64])

# Define linear fitting function
def linear(x, m, c):
    return m * x + c

# Define data processing and plotting function
def process_and_plot(shi_a, shi_b, color_data, label_data):
    # Split data: divide based on median of shi_a
    median_value = np.median(shi_a)
    shi_a_small = shi_a[shi_a <= median_value]
    shi_b_small = shi_b[shi_a <= median_value]
    shi_a_large = shi_a[shi_a > median_value]
    shi_b_large = shi_b[shi_a > median_value]

    # Perform linear fitting on both parts
    params_small, _ = curve_fit(linear, shi_a_small, shi_b_small)
    params_large, _ = curve_fit(linear, shi_a_large, shi_b_large)

    # Get fitting parameters
    m_small, c_small = params_small
    m_large, c_large = params_large

    # Generate fitted curves
    a_small_fit = np.linspace(min(shi_a_small), max(shi_a_small), 100)
    b_small_fit = linear(a_small_fit, m_small, c_small)
    a_large_fit = np.linspace(min(shi_a_large), max(shi_a_large), 100)
    b_large_fit = linear(a_large_fit, m_large, c_large)

    # Plot data points and fitted curves
    plt.scatter(shi_a, shi_b, color=color_data, alpha=0.7, label=f'{label_data} ')
    plt.plot(a_small_fit, b_small_fit, color='blue', linestyle='--')
    plt.plot(a_large_fit, b_large_fit, color='red', linestyle='--')
    plt.axvline(x=median_value, color='black', linestyle=':', label=f'{label_data} Median_Pr={median_value:.2f}mm')

# Plot
plt.figure(figsize=(10, 6))

# Process first dataset
process_and_plot(shi_a1, shi_b1, color_data='green', label_data='US_Ha1')

# Process second dataset
process_and_plot(shi_a2, shi_b2, color_data='orange', label_data='US_Wkg')

# Process third dataset
process_and_plot(shi_a3, shi_b3, color_data='skyblue', label_data='GF_Guy')

# Set legend and labels
plt.xlabel('PR(mm/yr)')
plt.ylabel('GPP(gC/m²/yr)')
plt.title('Sites')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


步骤16：三个降水相差很大的站点的分布图


In [64]:
print('Step 17: Distribution plots showing neighborhood effects')
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable

fig, ax = plt.subplots(3, 1, figsize=(6, 8))
world = gpd.read_file(r'D:/JYQ_data/JYQ/世界地图/world_line/new_world_line.shp')  # Remember to replace with continental map
# Use loop to plot the same world map boundaries on each subplot
for i in range(3):
    world.plot(ax=ax[i], color='gray', linewidth=0.1)

sm1 = ScalarMappable(cmap='seismic_r', norm=plt.Normalize(vmin=-1, vmax=1))
sm1.set_array([])  # Need an empty array to create colorbar

AI_slope_CRU_025_1_tif.plot(ax=ax[0], cmap='seismic_r', vmin=-1, vmax=1, label='CRU', add_colorbar=False)
ax[0].set_title('AI_slope(single pixel)')
cbar0 = fig.colorbar(sm1, ax=ax[0], orientation='vertical')

AI_slope_CRU_025_8_tif.plot(ax=ax[1], cmap='seismic_r', vmin=-1, vmax=1, label='CRU', add_colorbar=False)
ax[1].set_title('AI_slope(8-neighborhood)')
cbar1 = fig.colorbar(sm1, ax=ax[1], orientation='vertical')

AI_slope_CRU_025_24_tif.plot(ax=ax[2], cmap='seismic_r', vmin=-1, vmax=1, label='CRU', add_colorbar=False)
ax[2].set_title('AI_slope(24-neighborhood)')
cbar2 = fig.colorbar(sm1, ax=ax[2], orientation='vertical')

for i in range(3):
    world.plot(ax=ax[i], color='gray', linewidth=0.1)
    ax[i].set_xlabel('')  # Remove X-axis tick labels
    ax[i].set_ylabel('')  # Remove Y-axis tick labels

plt.tight_layout()
plt.show()
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\邻域不同造成的影响', dpi=1500, bbox_inches='tight')

步骤17：邻域造成的影响分布图


In [104]:
print("Step 18: Consistency comparison between 8-neighborhood and 24-neighborhood")
# Copy to prevent modification of original data
A = AI_slope_CRU_025_8.copy()
temp = AI_slope_CRU_025_24.copy()

# Remove all values equal to -1 and 0
temp[temp == -1] = np.nan
temp[temp == 0] = np.nan
A[A == -1] = np.nan
A[A == 0] = np.nan

# Retain consistent areas
valid_mask2 = ~np.isnan(A) & ~np.isnan(temp)
A_valid2 = A[valid_mask2]
temp2 = temp[valid_mask2]

# Create DataFrame
data = pd.DataFrame({
    '8-neighborhood': A_valid2,
    '24-neighborhood': temp2,
})

# Calculate correlation matrix
corr_matrix = data.corr()
print(corr_matrix)

# Set theme style
sns.set_theme(style="ticks")

# Draw pairplot
g = sns.pairplot(data)

# Add linear fit lines to each scatter plot and display fitting equation
for i in range(len(g.axes)):
    for j in range(len(g.axes)):
        if i != j:  # Add fit line only to scatter plots, skip diagonal histograms
            # Get current subplot Axes object
            ax = g.axes[i, j]

            # Calculate linear fit using statsmodels
            x = data[data.columns[j]]
            y = data[data.columns[i]]
            x = sm.add_constant(x)  # Add constant term
            model = sm.OLS(y, x).fit()
            slope, intercept = model.params[1], model.params[0]

            # Draw fit line
            sns.regplot(x=data.columns[j], y=data.columns[i], data=data,
                        scatter_kws={'s': 20, 'alpha': 0.5},
                        line_kws={'color': 'red', 'lw': 1},
                        ax=ax)

            # Add fitting equation text
            equation = f"y = {slope:.2f}x + {intercept:.2f}"
            ax.text(0.05, 0.9, equation, transform=ax.transAxes, fontsize=8, color='red')
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\不同AI指标的比较图', dpi=1000, bbox_inches='tight')
# Display image
plt.show()

步骤18：邻域不同，8邻域和24邻域的一致性比对
                 8-neighborhood  24-neighborhood
8-neighborhood         1.000000         0.943518
24-neighborhood        0.943518         1.000000


In [71]:
print('Step 19: Distribution plots showing the effects of different precipitation data sources')
AI_slope_Terra_025_8, AI_r_Terra_025_8, AI_extreme_Terra_025_8, AI_slope_Terra_025_8_all, AI_r_Terra_025_8_all, AI_extreme_Terra_025_8_all = calculate_AI_8(Terra_anomaly_025, NDVI_anomaly_025, 0.25)
#### Convert array format to TIFF format. Since Terra is selected, recalculate and convert to TIFF format
longitude_left = np.arange(-180, 0, 0.25)
longitude_right = np.arange(0.25, 180.25, 0.25)
longitude = np.concatenate((longitude_left, longitude_right))
latitude_up = np.arange(90, 0, -0.25)
latitude_down = np.arange(-0.25, -90.25, -0.25)
latitude = np.concatenate((latitude_up, latitude_down))
AI_slope_Terra_025_8_tif = xr.DataArray(AI_slope_Terra_025_8, dims=['y', 'x'], coords=[latitude, longitude])
#### Conversion completed

fig, ax = plt.subplots(2, 1, figsize=(15, 8))
world = gpd.read_file(r'D:/JYQ_data/JYQ/世界地图/world_line/new_world_line.shp')
# Use loop to plot the same world map boundaries on each subplot
world.plot(ax=ax[0], color='gray', linewidth=0.1)
world.plot(ax=ax[1], color='gray', linewidth=0.1)
sm1 = ScalarMappable(cmap='seismic_r', norm=plt.Normalize(vmin=-1, vmax=1))
sm1.set_array([])  # Need an empty array to create colorbar

# CRU
AI_slope_CRU_025_8_tif.plot(ax=ax[0], cmap='seismic_r', vmin=-1, vmax=1, label='CRU', add_colorbar=False)
ax[0].set_title('AI_slope(CRU_NDVI)')
cbar0 = fig.colorbar(sm1, ax=ax[0], orientation='vertical', shrink=1)
# Terra
AI_slope_Terra_025_8_tif.plot(ax=ax[1], cmap='seismic_r', vmin=-1, vmax=1, label='Terra', add_colorbar=False)
ax[1].set_title('AI_slope(TerraClimate_NDVI)')
cbar1 = fig.colorbar(sm1, ax=ax[1], orientation='vertical', shrink=1)
plt.tight_layout()
plt.savefig(r'D:\BaiduSyncdisk\结果\照片\降水数据来源不同造成的影响', dpi=1000, bbox_inches='tight')
for i in range(2):
    world.plot(ax=ax[i], color='gray', linewidth=0.1)
    ax[i].set_xlabel('')  # Remove X-axis tick labels
    ax[i].set_ylabel('')  # Remove Y-axis tick labels
    ax[i].text(0.02, 0.98, f'({chr(97+i)})', transform=ax[i].transAxes,
               fontsize=14, fontweight='bold', va='top')
plt.show()

步骤19：不同降水数据造成的影响的分布图


In [69]:
print("Step 20: Function for effects of different vegetation data (because LAI data has different specifications)")
def calculate_AI_8_new(pr_anomaly, NDVI_anomaly, n):  # 8-neighborhood, input n, adaptive to three spatial resolutions
    # Define empty variables to store results
    AI_slope_050 = np.empty((360, 720), dtype=float)
    AI_r_050 = np.empty((360, 720), dtype=float)
    AI_extreme_050 = np.empty((360, 720), dtype=float)
    AI_slope_050_all = np.empty((360, 720), dtype=float)
    AI_r_050_all = np.empty((360, 720), dtype=float)
    AI_extreme_050_all = np.empty((360, 720), dtype=float)
    AI_slope_025 = np.empty((720, 1440), dtype=float)
    AI_r_025 = np.empty((720, 1440), dtype=float)
    AI_extreme_025 = np.empty((720, 1440), dtype=float)
    AI_slope_025_all = np.empty((720, 1440), dtype=float)
    AI_r_025_all = np.empty((720, 1440), dtype=float)
    AI_extreme_025_all = np.empty((720, 1440), dtype=float)
    AI_slope_010 = np.empty((1800, 3600), dtype=float)
    AI_r_010 = np.empty((1800, 3600), dtype=float)
    AI_extreme_010 = np.empty((1800, 3600), dtype=float)
    AI_slope_010_all = np.empty((1800, 3600), dtype=float)
    AI_r_010_all = np.empty((1800, 3600), dtype=float)
    AI_extreme_010_all = np.empty((1800, 3600), dtype=float)
    if n == 0.25:  # Size (720, 1440)
        for i in range(720):
            for j in range(1440):
                if np.all(np.isnan(pr_anomaly[:, i, j])) or np.all(np.isnan(NDVI_anomaly[:, i, j])):
                    AI_slope_025[i, j] = np.nan
                    AI_r_025[i, j] = np.nan
                    AI_extreme_025[i, j] = np.nan
                    AI_slope_025_all[i, j] = np.nan
                    AI_r_025_all[i, j] = np.nan
                    AI_extreme_025_all[i, j] = np.nan

                else:  # Ensure center pixel sequence is not all NaN, at least one valid value
                    test1 = pr_anomaly[:, i - 1:i + 2, j - 1:j + 2].flatten()
                    test2 = NDVI_anomaly[:, i - 1:i + 2, j - 1:j + 2].flatten()  # Extract 8-neighborhood
                    pr_purified, NDVI_purified = jinhua(test1, test2)  # Purify two arrays
                    if np.all(np.isnan(test1)) or np.all(np.isnan(test2)):  # Must check after purification, as both might become NaN after purification
                        AI_slope_025[i, j] = np.nan
                        AI_r_025[i, j] = np.nan
                        AI_extreme_025[i, j] = np.nan
                        AI_slope_025_all[i, j] = np.nan
                        AI_r_025_all[i, j] = np.nan
                        AI_extreme_025_all[i, j] = np.nan
                    else:
                        pr_sorted, NDVI_sorted = sort_by_A(pr_purified, NDVI_purified)  # After purification, now sort by moisture from low to high
                        mid = len(pr_sorted) // 2  # Divide by median, ensuring at least 20 samples on each side
                        x1 = pr_sorted[0:mid]
                        y1 = NDVI_sorted[0:mid]
                        slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
                        x2 = pr_sorted[mid:]
                        y2 = NDVI_sorted[mid:]
                        slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
                        max_NDVI = np.mean(NDVI_sorted[-5:])  # Extract NDVI values for the highest precipitation years, last 5
                        min_NDVI = np.mean(NDVI_sorted[:5])  # Extract NDVI values for the lowest precipitation years, first 5
                        mean_NDVI = np.mean(NDVI_sorted)  # Calculate the mean of the entire sequence
                        positive = (max_NDVI - mean_NDVI)
                        negative = (mean_NDVI - min_NDVI)

                        # Direct calculation without considering significance
                        AI_slope_025_all[i, j] = (slope2 - slope1) / (slope2 + slope1)
                        AI_r_025_all[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                        AI_extreme_025_all[i, j] = (positive - negative) / (positive + negative)

                        # Start conditional judgment
                        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # If both dry and wet years respond positively and linearly significantly
                            AI_slope_025[i, j] = (slope2 - slope1) / (slope2 + slope1)
                            AI_r_025[i, j] = (r_value2 - r_value1) / (r_value2 + r_value1)
                            AI_extreme_025[i, j] = (positive - negative) / (positive + negative)

                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:  # If dry year responds positively and linearly significantly, but wet year responds significantly negatively
                            AI_slope_025[i, j] = -1
                            AI_r_025[i, j] = -1
                            AI_extreme_025[i, j] = -1
                        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 > 0.1:  # If dry year responds positively and linearly significantly, but wet year shows no significant linear response
                            AI_slope_025[i, j] = -1
                            AI_r_025[i, j] = -1
                            AI_extreme_025[i, j] = -1
                        elif slope1 < 0 or p_value1 > 0.1:  # If dry year does not respond, skip calculation directly
                            AI_slope_025[i, j] = np.nan
                            AI_r_025[i, j] = np.nan
                            AI_extreme_025[i, j] = np.nan
        return AI_slope_025, AI_r_025, AI_extreme_025

步骤20：不同植被数据造成的影响的函数（因为LAI的数据规格不一样）


In [72]:
print('Step 21: Distribution plots showing the effects of different vegetation data sources')
# AI_slope_CRU_025_8_LAI, AI_r_CRU_025_8_LAI, AI_extreme_CRU_025_8_LAI = calculate_AI_8_new(CRU_anomaly_025[0:40, :, :], LAI_anomaly_025, 0.25)
# #### Convert array format to TIFF format. Since Terra is selected, recalculate and convert to TIFF format
# longitude_left = np.arange(-180, 0, 0.25)
# longitude_right = np.arange(0.25, 180.25, 0.25)
# longitude = np.concatenate((longitude_left, longitude_right))
# latitude_up = np.arange(90, 0, -0.25)
# latitude_down = np.arange(-0.25, -90.25, -0.25)
# latitude = np.concatenate((latitude_up, latitude_down))
# AI_slope_CRU_025_8_LAI_tif = xr.DataArray(AI_slope_CRU_025_8_LAI, dims=['y', 'x'], coords=[latitude, longitude])
# #### Conversion completed

fig, ax = plt.subplots(2, 1, figsize=(15, 8))
world = gpd.read_file(r'D:/JYQ_data/JYQ/世界地图/world_line/new_world_line.shp')
# Use loop to plot the same world map boundaries on each subplot
world.plot(ax=ax[0], color='gray', linewidth=0.1)
world.plot(ax=ax[1], color='gray', linewidth=0.1)
sm1 = ScalarMappable(cmap='seismic_r', norm=plt.Normalize(vmin=-1, vmax=1))
sm1.set_array([])  # Need an empty array to create colorbar

# CRU
AI_slope_CRU_025_8_tif.plot(ax=ax[0], cmap='seismic_r', vmin=-1, vmax=1, label='CRU', add_colorbar=False)
ax[0].set_title('AI_slope(CRU_NDVI)')
cbar0 = fig.colorbar(sm1, ax=ax[0], orientation='vertical', shrink=1)
# LAI
AI_slope_CRU_025_8_LAI_tif.plot(ax=ax[1], cmap='seismic_r', vmin=-1, vmax=1, label='Terra', add_colorbar=False)
ax[1].set_title('AI_slope(CRU_LAI)')
cbar1 = fig.colorbar(sm1, ax=ax[1], orientation='vertical', shrink=1)
plt.tight_layout()

for i in range(2):
    world.plot(ax=ax[i], color='gray', linewidth=0.1)
    ax[i].set_xlabel('')  # Remove X-axis tick labels
    ax[i].set_ylabel('')  # Remove Y-axis tick labels
    ax[i].text(0.02, 0.98, f'({chr(97+i)})', transform=ax[i].transAxes,
               fontsize=14, fontweight='bold', va='top')
plt.savefig(r'D:\BaiduSyncdisk\结果\照片\植被数据来源不同造成的影响（LAI）', dpi=1000, bbox_inches='tight')
plt.show()

步骤21：不同植被数据造成的影响分布图


In [80]:
print('Step 22: Pairwise comparison plots for different data sources')
# Retain common parts of the three arrays
A = AI_slope_CRU_025_8.copy()
B = AI_slope_Terra_025_8.copy()
C = AI_slope_CRU_025_8_LAI.copy()
# Remove all values equal to -1
A[A == -1] = np.nan
A[A == 0] = np.nan
B[B == -1] = np.nan
B[B == 0] = np.nan
C[C == -1] = np.nan
C[C == 0] = np.nan
valid_mask = ~np.isnan(A) & ~np.isnan(C) & ~np.isnan(B)
A_valid = A[valid_mask]
B_valid = B[valid_mask]
C_valid = C[valid_mask]

# Create DataFrame
data = pd.DataFrame({
    'CRU_NDVI': A_valid,
    'TerraClimate_NDVI': B_valid,
    'CRU_LAI': C_valid
})
corr_matrix = data.corr()
print(corr_matrix)
sns.set_theme(style="ticks", font_scale=0.9)

# Create PairGrid object, customize diagonal and off-diagonal plots
g = sns.PairGrid(data, diag_sharey=False, height=2.5)
# Diagonal: Kernel density plot
g.map_diag(sns.kdeplot, fill=True, alpha=0.8, color='#1f77b4',
           edgecolor='black', linewidth=0.5, bw_adjust=0.8)
# Off-diagonal: 2D kernel density contours or Hexbin heatmap
g.map_upper(sns.kdeplot, cmap='Blues', fill=True, levels=10, alpha=0.7)
g.map_lower(sns.histplot, cmap='Reds', cbar=True,
            bins=50, alpha=0.9, edgecolor='gray', linewidth=0.1)

# Add fitted lines and equations (only in lower-left triangle region)
for i in range(len(data.columns)):
    for j in range(len(data.columns)):
        if i > j:  # Add regression lines only in lower triangle region
            ax = g.axes[i, j]
            x = data[data.columns[j]]
            y = data[data.columns[i]]
            sns.regplot(x=x, y=y, scatter=False,
                        line_kws={'color': 'darkred', 'lw': 1.5}, ax=ax)
            model = sm.OLS(y, sm.add_constant(x)).fit()
            equation = f"ρ={corr_matrix.iloc[i, j]:.2f}\ny={model.params[1]:.2f}x+{model.params[0]:.2f}"
            # Modify text parameters: white background box + black text + increased font size
            ax.text(0.05, 0.85, equation, transform=ax.transAxes,
                    fontsize=10, color='black', ha='left',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.8))

# Optimize layout and annotations
plt.suptitle("Cross-Data Comparison (Density & Hexbin)", y=1.02, fontsize=11)
g.fig.subplots_adjust(wspace=0.1, hspace=0.1)
[ax.set_xticks([]) if i % 3 != 0 else None for i, ax in enumerate(g.axes.flat)]  # Reduce tick density

# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\不同数据源的时间趋势的比较', dpi=1000, bbox_inches='tight')
plt.show()

步骤22:不同数据来源的两两比对图
                   CRU_NDVI  TerraClimate_NDVI   CRU_LAI
CRU_NDVI           1.000000           0.813519  0.465668
TerraClimate_NDVI  0.813519           1.000000  0.424551
CRU_LAI            0.465668           0.424551  1.000000


In [74]:
print("Step 23: Frequency histograms showing the effects of different resolutions (cannot draw pairwise comparison plots)")
fig = plt.figure(figsize=(5.5, 5))
sns.kdeplot(AI_slope_CRU_025_8.flatten(), fill=True, color='red', alpha=0.1, label='0.25°', clip=(-1, 1))
sns.kdeplot(AI_slope_CRU_050_8.flatten(), fill=True, color='blue', alpha=0.1, label='0.5°', clip=(-1, 1))
plt.xlabel('Asymmetric Index (AI)')
plt.legend(loc='upper right')
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\不同分辨率的直方图.png', dpi=1500, bbox_inches='tight')
plt.show()

步骤24：不同分辨率产生的影响的频率直方图（无法绘制两两比对图）


In [73]:
print("Step 24: Frequency histograms showing the effects of different precipitation and vegetation data")
fig = plt.figure(figsize=(5.5, 5))
sns.kdeplot(AI_slope_CRU_025_8.flatten(), fill=True, color='red', alpha=0.1, label='CRU', clip=(-1, 1))
sns.kdeplot(AI_slope_Terra_025_8.flatten(), fill=True, color='blue', alpha=0.1, label='TerraClimate', clip=(-1, 1))
plt.xlabel('Asymmetric Index (AI)')
plt.legend(loc='upper right')
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\不同降水的直方图.png', dpi=1500, bbox_inches='tight')
plt.show()

步骤23：不同降水和植被数据产生的影响的频率直方图


In [78]:
print("Step 25-1: Example of a single point in northern Russia explaining why NDVI and LAI differ")
# Open shrubland in northern Russia: (70.25°N, 147.5°E), (79.0, 1310.0)
x2 = CRU_anomaly_025[:, 77:80, 1309:1311].flatten()
y2 = NDVI_anomaly_025[:, 77:80, 1309:1311].flatten()

x3 = CRU_anomaly_025[0:40, 77:80, 1309:1311].flatten()
y3 = LAI_anomaly_025[:, 77:80, 1309:1311].flatten()
split_arrays_ultra(x2, y2, 'Russia North(NDVI)')
split_arrays_ultra(x3, y3, 'Russia North(LAI)')
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\俄罗斯北部的开阔灌木.png', dpi=1500, bbox_inches='tight')

步骤25-1：俄罗斯北部单点举例，为什么NDVI和LAI会有差异


(0.39015966722141754, -0.33937939331581063, -0.00898011945742142)

In [76]:
print('Step 25-2: Time series for northern Russia')
NDVI_anomaly_025[:, 78, 1310]
LAI_anomaly_025[:, 78, 1310]
fig, ax = plt.subplots()
plt.plot(CRU_anomaly_025[:, 78, 1310], color='blue')
plt.plot(NDVI_anomaly_025[:, 78, 1310], color='red')
plt.plot(LAI_anomaly_025[:, 78, 1310], color='green')
plt.show()

步骤25-2：俄罗斯北部的时间序列


In [127]:
print('Step 25-3: Time series for eastern Australia')
NDVI_anomaly_025[:, 460, 1328]
LAI_anomaly_025[:, 460, 1328]

x2 = CRU_anomaly_025[:, 459:461, 1327:1329].flatten()
y2 = NDVI_anomaly_025[:, 459:461, 1327:1329].flatten()

x3 = CRU_anomaly_025[:40, 461:463, 1320:1322].flatten()
y3 = LAI_anomaly_025[:, 461:463, 1320:1322].flatten()
x3.shape, y3.shape
split_arrays_ultra(x2, y2, 'Eastern Australia(NDVI)')
split_arrays_ultra(x3, y3, 'Eastern Australia(LAI)')
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\俄罗斯北部的开阔灌木.png', dpi=1500, bbox_inches='tight')

步骤25-3：澳大利亚东部的时间序列


(0.09973610562467891, 0.3797633143865819, 0.23994323956107544)

In [67]:
print('Step 26: Spatial distribution plots showing the effects of different resolutions')
fig, ax = plt.subplots(2, 1, figsize=(15, 8))
world = gpd.read_file(r'D:/JYQ_data/JYQ/世界地图/world_line/new_world_line.shp')
# Use loop to plot the same world map boundaries on each subplot
world.plot(ax=ax[0], color='gray', linewidth=0.1)
world.plot(ax=ax[1], color='gray', linewidth=0.1)
sm1 = ScalarMappable(cmap='seismic_r', norm=plt.Normalize(vmin=-1, vmax=1))
sm1.set_array([])  # Need an empty array to create colorbar

# CRU
AI_slope_CRU_025_8_tif.plot(ax=ax[0], cmap='seismic_r', vmin=-1, vmax=1, label='CRU', add_colorbar=False)
ax[0].set_title('AI_slope(0.25° resolution)')
cbar0 = fig.colorbar(sm1, ax=ax[0], orientation='vertical', shrink=1)
# Terra
AI_slope_CRU_050_8_tif.plot(ax=ax[1], cmap='seismic_r', vmin=-1, vmax=1, label='Terra', add_colorbar=False)
ax[1].set_title('AI_slope(0.5° resolution)')
cbar1 = fig.colorbar(sm1, ax=ax[1], orientation='vertical', shrink=1)
plt.tight_layout()
for i in range(2):
    world.plot(ax=ax[i], color='gray', linewidth=0.1)
    ax[i].set_xlabel('')  # Remove X-axis tick labels
    ax[i].set_ylabel('')  # Remove Y-axis tick labels
    ax[i].text(0.02, 0.98, f'({chr(97+i)})', transform=ax[i].transAxes,
               fontsize=14, fontweight='bold', va='top')
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\分辨率不同造成的影响', dpi=1000, bbox_inches='tight')
plt.show()

步骤26：不同分辨率对结果产生的影响的空间分布图


#The following section is for time trend plots.

In [409]:
print('Step 27-1: Functions for calculating AI index sequences and AI trends for 30, 25, and 20-year time windows')


def caluate_AI_sequence_and_slope_30(array_A, array_B):  # 30-year time window, input is a 41 * 3 * 3 window
    AI_slope_list = []  # Used to store AI_extreme values after each iteration
    AI_r_list = []
    AI_extreme_list = []
    for i in range(12):
        pr_window = array_A[i:i + 30, :, :]
        NDVI_window = array_B[i:i + 30, :, :]  # Length 30, step size 1, total 11 sliding steps, each sliding extracts a 30 * 3 * 3 window

        # Flatten arrays and calculate three indicators as usual
        pr_window_flatten = pr_window.flatten()
        NDVI_window_flatten = NDVI_window.flatten()  # Flatten
        pr_window_flatten_purified, NDVI_window_flatten_purified = jinhua(pr_window_flatten, NDVI_window_flatten)
        pr_window_sorted, NDVI_window_sorted = sort_by_A(pr_window_flatten_purified,
                                                         NDVI_window_flatten_purified)  # Sort
        # First half is dry period, extract and start fitting
        mid = len(pr_window_sorted) // 2
        x1 = pr_window_sorted[0:mid]
        y1 = NDVI_window_sorted[0:mid]
        x1 = np.array(x1)
        y1 = np.array(y1)
        slope1, intercept1, r1, p_value1, std_err1 = linregress(x1, y1)
        # Second half is wet period, extract and start fitting
        x2 = pr_window_sorted[mid:]
        y2 = NDVI_window_sorted[mid:]
        x2 = np.array(x2)
        y2 = np.array(y2)
        slope2, intercept2, r2, p_value2, std_err2 = linregress(x2, y2)
        max_NDVI = np.mean(NDVI_window_sorted[-5:])  # Extract NDVI values for the highest precipitation years, last 5
        min_NDVI = np.mean(NDVI_window_sorted[:5])  # Extract NDVI values for the lowest precipitation years, first 5
        mean_NDVI = np.mean(NDVI_window_sorted)  # Calculate the mean of the entire sequence
        positive = (max_NDVI - mean_NDVI)
        negative = (mean_NDVI - min_NDVI)

        a = np.nan
        c = np.nan
        # If both dry and wet years respond
        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # Relax conditions appropriately here
            a = (slope2 - slope1) / (slope2 + slope1)
            b = (r2 - r1) / (r2 + r1)
            c = (positive - negative) / (positive + negative)
        # If dry year responds, wet year negatively responds
        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:
            a = -1
            b = -1
            c = -1
        # If dry year responds, wet year does not respond
        elif slope1 > 0 and p_value1 < 0.1 and p_value2 > 0.1:
            a = -1
            b = -1
            c = -1
        # If dry year does not respond, and AI_extreme exceeds the range (-1,1)
        elif slope1 < 0 or p_value1 > 0.1 or c > 1 or c < -1:
            a = np.nan
            b = np.nan
            c = np.nan
        AI_slope_list.append(a)
        AI_r_list.append(b)
        AI_extreme_list.append(c)

    AI_slope_arr = np.array(AI_slope_list)  # Convert list to array for further operations
    AI_r_arr = np.array(AI_r_list)
    AI_extreme_arr = np.array(AI_extreme_list)
    non_nan_values = AI_slope_arr[~np.isnan(AI_slope_arr)]  # Extract non-NaN values
    non_nan_values2 = AI_r_arr[~np.isnan(AI_r_arr)]
    non_nan_values3 = AI_extreme_arr[~np.isnan(AI_extreme_arr)]
    if len(non_nan_values) == 0:
        slope4 = np.nan
        slope5 = np.nan
        slope6 = np.nan
    else:
        x = np.arange(1, len(non_nan_values) + 1)  # Calculate AI index trend based on obtained AI index time series
        x2 = np.arange(1, len(non_nan_values2) + 1)
        x3 = np.arange(1, len(non_nan_values3) + 1)
        slope4, intercept4, r_value4, p_value4, std_err4 = linregress(x, non_nan_values)
        slope5, intercept5, r_value5, p_value5, std_err5 = linregress(x, non_nan_values2)
        slope6, intercept6, r_value6, p_value6, std_err6 = linregress(x, non_nan_values3)
        if p_value4 > 0.1:  # If AI index trend does not show significant upward trend, discard it
            slope4 = np.nan
        if p_value5 > 0.1:  # If AI index trend does not show significant upward trend, discard it
            slope5 = np.nan
        if p_value6 > 0.1:  # If AI index trend does not show significant upward trend, discard it
            slope6 = np.nan
    return AI_slope_arr, slope4, AI_r_arr, slope5, AI_extreme_arr, slope6


def caluate_AI_sequence_and_slope_20(array_A, array_B):  # 20-year time window, input is a 41 * 3 * 3 window
    AI_slope_list = []  # Used to store AI_extreme values after each iteration
    AI_r_list = []
    AI_extreme_list = []
    for i in range(22):
        pr_window = array_A[i:i + 20, :, :]
        NDVI_window = array_B[i:i + 20, :, :]  # Length 20, step size 1, total 22 sliding steps, each sliding extracts a 20 * 3 * 3 window

        # Flatten arrays and calculate three indicators as usual
        pr_window_flatten = pr_window.flatten()
        NDVI_window_flatten = NDVI_window.flatten()  # Flatten
        pr_window_flatten_purified, NDVI_window_flatten_purified = jinhua(pr_window_flatten, NDVI_window_flatten)
        pr_window_sorted, NDVI_window_sorted = sort_by_A(pr_window_flatten_purified,
                                                         NDVI_window_flatten_purified)  # Sort
        # First half is dry period, extract and start fitting
        mid = len(pr_window_sorted) // 2
        x1 = pr_window_sorted[0:mid]
        y1 = NDVI_window_sorted[0:mid]
        x1 = np.array(x1)
        y1 = np.array(y1)
        slope1, intercept1, r1, p_value1, std_err1 = linregress(x1, y1)
        # Second half is wet period, extract and start fitting
        x2 = pr_window_sorted[mid:]
        y2 = NDVI_window_sorted[mid:]
        x2 = np.array(x2)
        y2 = np.array(y2)
        slope2, intercept2, r2, p_value2, std_err2 = linregress(x2, y2)
        max_NDVI = np.mean(NDVI_window_sorted[-5:])  # Extract NDVI values for the highest precipitation years, last 5
        min_NDVI = np.mean(NDVI_window_sorted[:5])  # Extract NDVI values for the lowest precipitation years, first 5
        mean_NDVI = np.mean(NDVI_window_sorted)  # Calculate the mean of the entire sequence
        positive = (max_NDVI - mean_NDVI)
        negative = (mean_NDVI - min_NDVI)

        a = np.nan
        c = np.nan
        # If both dry and wet years respond
        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # Relax conditions appropriately here
            a = (slope2 - slope1) / (slope2 + slope1)
            b = (r2 - r1) / (r2 + r1)
            c = (positive - negative) / (positive + negative)
        # If dry year responds, wet year negatively responds
        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:
            a = -1
            b = -1
            c = -1
        # If dry year responds, wet year does not respond
        elif slope1 > 0 and p_value1 < 0.1 and p_value2 > 0.1:
            a = -1
            b = -1
            c = -1
        # If dry year does not respond
        elif slope1 < 0 or p_value1 > 0.1 or c > 1 or c < -1:
            a = np.nan
            b = np.nan
            c = np.nan
        # Additional logical check, add another condition
        if c > 1 or c < -1:
            a = np.nan
            b = np.nan
            c = np.nan

        AI_slope_list.append(a)
        AI_r_list.append(b)
        AI_extreme_list.append(c)

    AI_slope_arr = np.array(AI_slope_list)  # Convert list to array for further operations
    AI_r_arr = np.array(AI_r_list)
    AI_extreme_arr = np.array(AI_extreme_list)
    non_nan_values = AI_slope_arr[~np.isnan(AI_slope_arr)]  # Extract non-NaN values
    non_nan_values2 = AI_r_arr[~np.isnan(AI_r_arr)]
    non_nan_values3 = AI_extreme_arr[~np.isnan(AI_extreme_arr)]
    if len(non_nan_values) == 0:
        slope4 = np.nan
        slope5 = np.nan
        slope6 = np.nan
    else:
        x = np.arange(1, len(non_nan_values) + 1)  # Calculate AI index trend based on obtained AI index time series
        x2 = np.arange(1, len(non_nan_values2) + 1)
        x3 = np.arange(1, len(non_nan_values3) + 1)
        slope4, intercept4, r_value4, p_value4, std_err4 = linregress(x, non_nan_values)
        slope5, intercept5, r_value5, p_value5, std_err5 = linregress(x, non_nan_values2)
        slope6, intercept6, r_value6, p_value6, std_err6 = linregress(x, non_nan_values3)
        if p_value4 > 0.1:  # If AI index trend does not show significant upward trend, discard it
            slope4 = np.nan
        if p_value5 > 0.1:  # If AI index trend does not show significant upward trend, discard it
            slope5 = np.nan
        if p_value6 > 0.1:  # If AI index trend does not show significant upward trend, discard it
            slope6 = np.nan
    return AI_slope_arr, slope4, AI_r_arr, slope5, AI_extreme_arr, slope6


def caluate_AI_sequence_and_slope_25(array_A, array_B):  # 25-year time window, input is a 41 * 3 * 3 window
    AI_slope_list = []  # Used to store AI_extreme values after each iteration
    AI_r_list = []
    AI_extreme_list = []
    for i in range(17):
        pr_window = array_A[i:i + 25, :, :]
        NDVI_window = array_B[i:i + 25, :, :]  # Length 25, step size 1, total 17 sliding steps, each sliding extracts a 25 * 3 * 3 window

        # Flatten arrays and calculate three indicators as usual
        pr_window_flatten = pr_window.flatten()
        NDVI_window_flatten = NDVI_window.flatten()  # Flatten
        pr_window_flatten_purified, NDVI_window_flatten_purified = jinhua(pr_window_flatten, NDVI_window_flatten)
        pr_window_sorted, NDVI_window_sorted = sort_by_A(pr_window_flatten_purified,
                                                         NDVI_window_flatten_purified)  # Sort
        # First half is dry period, extract and start fitting
        mid = len(pr_window_sorted) // 2
        x1 = pr_window_sorted[0:mid]
        y1 = NDVI_window_sorted[0:mid]
        x1 = np.array(x1)
        y1 = np.array(y1)
        slope1, intercept1, r1, p_value1, std_err1 = linregress(x1, y1)
        # Second half is wet period, extract and start fitting
        x2 = pr_window_sorted[mid:]
        y2 = NDVI_window_sorted[mid:]
        x2 = np.array(x2)
        y2 = np.array(y2)
        slope2, intercept2, r2, p_value2, std_err2 = linregress(x2, y2)
        max_NDVI = np.mean(NDVI_window_sorted[-5:])  # Extract NDVI values for the highest precipitation years, last 5
        min_NDVI = np.mean(NDVI_window_sorted[:5])  # Extract NDVI values for the lowest precipitation years, first 5
        mean_NDVI = np.mean(NDVI_window_sorted)  # Calculate the mean of the entire sequence
        positive = (max_NDVI - mean_NDVI)
        negative = (mean_NDVI - min_NDVI)

        a = np.nan
        c = np.nan
        # If both dry and wet years respond
        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # Relax conditions appropriately here
            a = (slope2 - slope1) / (slope2 + slope1)
            b = (r2 - r1) / (r2 + r1)
            c = (positive - negative) / (positive + negative)
        # If dry year responds, wet year negatively responds
        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:
            a = -1
            b = -1
            c = -1
        # If dry year responds, wet year does not respond
        elif slope1 > 0 and p_value1 < 0.1 and p_value2 > 0.1:
            a = -1
            b = -1
            c = -1
        # If dry year does not respond, or AI_extreme exceeds the range (-1,1)
        elif slope1 < 0 or p_value1 > 0.1 or c > 1 or c < -1:
            a = np.nan
            b = np.nan
            c = np.nan
        AI_slope_list.append(a)
        AI_r_list.append(b)
        AI_extreme_list.append(c)

    AI_slope_arr = np.array(AI_slope_list)  # Convert list to array for further operations
    AI_r_arr = np.array(AI_r_list)
    AI_extreme_arr = np.array(AI_extreme_list)
    non_nan_values = AI_slope_arr[~np.isnan(AI_slope_arr)]  # Extract non-NaN values
    non_nan_values2 = AI_r_arr[~np.isnan(AI_r_arr)]
    non_nan_values3 = AI_extreme_arr[~np.isnan(AI_extreme_arr)]
    if len(non_nan_values) == 0:
        slope4 = np.nan
        slope5 = np.nan
        slope6 = np.nan
    else:
        x = np.arange(1, len(non_nan_values) + 1)  # Calculate AI index trend based on obtained AI index time series
        x2 = np.arange(1, len(non_nan_values2) + 1)
        x3 = np.arange(1, len(non_nan_values3) + 1)
        slope4, intercept4, r_value4, p_value4, std_err4 = linregress(x, non_nan_values)
        slope5, intercept5, r_value5, p_value5, std_err5 = linregress(x, non_nan_values2)
        slope6, intercept6, r_value6, p_value6, std_err6 = linregress(x, non_nan_values3)
        if p_value4 > 0.1:  # If AI index trend does not show significant upward trend, discard it
            slope4 = np.nan
        if p_value5 > 0.1:  # If AI index trend does not show significant upward trend, discard it
            slope5 = np.nan
        if p_value6 > 0.1:  # If AI index trend does not show significant upward trend, discard it
            slope6 = np.nan
    return AI_slope_arr, slope4, AI_r_arr, slope5, AI_extreme_arr, slope6

步骤27-1：30，25，20年时间窗口的AI指标序列和AI趋势的函数


In [410]:
print("Step 27-2: Functions for calculating indicators for 30, 25, 20-year time windows, regardless of resolution, referencing the previous three functions")


def AI_sequence_and_slope_30(pr_anomaly, NDVI_anomaly, n):
    if n == 0.25:
        AI_slope_30_025 = np.empty((12, 720, 1440), dtype=float)
        AI_slope_30_025_trend = np.empty((720, 1440), dtype=float)
        AI_r_30_025 = np.empty((12, 720, 1440), dtype=float)
        AI_r_30_025_trend = np.empty((720, 1440), dtype=float)
        AI_extreme_30_025 = np.empty((12, 720, 1440), dtype=float)
        AI_extreme_30_025_trend = np.empty((720, 1440), dtype=float)

        for i in range(720):
            for j in range(1440):
                # If extracted data is all empty
                array_A = pr_anomaly[:, i - 2:i + 3, j - 2:j + 3]
                array_B = NDVI_anomaly[:, i - 2:i + 3, j - 2:j + 3]
                if len(array_A.flatten()) == 0 or len(array_B.flatten()) == 0 or np.isnan(array_A).any() or np.isnan(array_B).any():
                    AI_slope_30_025[:, i, j] = np.nan
                    AI_slope_30_025_trend[i, j] = np.nan
                    AI_r_30_025[:, i, j] = np.nan
                    AI_r_30_025_trend[i, j] = np.nan
                    AI_extreme_30_025[:, i, j] = np.nan
                    AI_extreme_30_025_trend[i, j] = np.nan
                else:
                    AI_slope_arr, slope1, AI_r_arr, slope2, AI_extreme_arr, slope3 = caluate_AI_sequence_and_slope_30(array_A, array_B)
                    AI_slope_30_025[:, i, j] = AI_slope_arr
                    AI_slope_30_025_trend[i, j] = slope1
                    AI_r_30_025[:, i, j] = AI_r_arr
                    AI_r_30_025_trend[i, j] = slope2
                    AI_extreme_30_025[:, i, j] = AI_extreme_arr
                    AI_extreme_30_025_trend[i, j] = slope3
        return AI_slope_30_025, AI_slope_30_025_trend, AI_r_30_025, AI_r_30_025_trend, AI_extreme_30_025, AI_extreme_30_025_trend


def AI_sequence_and_slope_25(pr_anomaly, NDVI_anomaly, n):
    if n == 0.25:
        AI_slope_25_025 = np.empty((17, 720, 1440), dtype=float)
        AI_slope_25_025_trend = np.empty((720, 1440), dtype=float)
        AI_r_25_025 = np.empty((17, 720, 1440), dtype=float)
        AI_r_25_025_trend = np.empty((720, 1440), dtype=float)
        AI_extreme_25_025 = np.empty((17, 720, 1440), dtype=float)
        AI_extreme_25_025_trend = np.empty((720, 1440), dtype=float)

        for i in range(720):
            for j in range(1440):
                # If extracted data is all empty
                array_A = pr_anomaly[:, i - 2:i + 3, j - 2:j + 3]
                array_B = NDVI_anomaly[:, i - 2:i + 3, j - 2:j + 3]
                if len(array_A.flatten()) == 0 or len(array_B.flatten()) == 0 or np.isnan(array_A).any() or np.isnan(array_B).any():
                    AI_slope_25_025[:, i, j] = np.nan
                    AI_slope_25_025_trend[i, j] = np.nan
                    AI_r_25_025[:, i, j] = np.nan
                    AI_r_25_025_trend[i, j] = np.nan
                    AI_extreme_25_025[:, i, j] = np.nan
                    AI_extreme_25_025_trend[i, j] = np.nan
                else:
                    AI_slope_arr, slope1, AI_r_arr, slope2, AI_extreme_arr, slope3 = caluate_AI_sequence_and_slope_25(array_A, array_B)
                    AI_slope_25_025[:, i, j] = AI_slope_arr
                    AI_slope_25_025_trend[i, j] = slope1
                    AI_r_25_025[:, i, j] = AI_r_arr
                    AI_r_25_025_trend[i, j] = slope2
                    AI_extreme_25_025[:, i, j] = AI_extreme_arr
                    AI_extreme_25_025_trend[i, j] = slope3
        return AI_slope_25_025, AI_slope_25_025_trend, AI_r_25_025, AI_r_25_025_trend, AI_extreme_25_025, AI_extreme_25_025_trend


def AI_sequence_and_slope_20(pr_anomaly, NDVI_anomaly, n):
    if n == 0.25:
        AI_slope_20_025 = np.empty((22, 720, 1440), dtype=float)
        AI_slope_20_025_trend = np.empty((720, 1440), dtype=float)
        AI_r_20_025 = np.empty((22, 720, 1440), dtype=float)
        AI_r_20_025_trend = np.empty((720, 1440), dtype=float)
        AI_extreme_20_025 = np.empty((22, 720, 1440), dtype=float)
        AI_extreme_20_025_trend = np.empty((720, 1440), dtype=float)

        for i in range(720):
            for j in range(1440):
                # If extracted data is all empty
                array_A = pr_anomaly[:, i - 2:i + 3, j - 2:j + 3]
                array_B = NDVI_anomaly[:, i - 2:i + 3, j - 2:j + 3]
                if len(array_A.flatten()) == 0 or len(array_B.flatten()) == 0 or np.isnan(array_A).any() or np.isnan(array_B).any():
                    AI_slope_20_025[:, i, j] = np.nan
                    AI_slope_20_025_trend[i, j] = np.nan
                    AI_r_20_025[:, i, j] = np.nan
                    AI_r_20_025_trend[i, j] = np.nan
                    AI_extreme_20_025[:, i, j] = np.nan
                    AI_extreme_20_025_trend[i, j] = np.nan
                else:
                    AI_slope_arr, slope1, AI_r_arr, slope2, AI_extreme_arr, slope3 = caluate_AI_sequence_and_slope_20(array_A, array_B)
                    AI_slope_20_025[:, i, j] = AI_slope_arr
                    AI_slope_20_025_trend[i, j] = slope1
                    AI_r_20_025[:, i, j] = AI_r_arr
                    AI_r_20_025_trend[i, j] = slope2
                    AI_extreme_20_025[:, i, j] = AI_extreme_arr
                    AI_extreme_20_025_trend[i, j] = slope3
        return AI_slope_20_025, AI_slope_20_025_trend, AI_r_20_025, AI_r_20_025_trend, AI_extreme_20_025, AI_extreme_20_025_trend

步骤27—2： 30，25，20年时间窗口，无论多少分辨率,开始计算指标的函数，要引用到前面三个函数


In [411]:
print('Step 28-1: Call functions to calculate multiple time windows and trend distributions')
# AI_slope_30_025, AI_slope_30_025_trend, AI_r_30_025, AI_r_30_025_trend, AI_extreme_30_025, AI_extreme_30_025_trend = AI_sequence_and_slope_30(CRU_anomaly_025, NDVI_year_025, 0.25)
# AI_slope_25_025, AI_slope_25_025_trend, AI_r_25_025, AI_r_25_025_trend, AI_extreme_25_025, AI_extreme_25_025_trend = AI_sequence_and_slope_25(CRU_anomaly_025, NDVI_year_025, 0.25)
AI_slope_20_025, AI_slope_20_025_trend, AI_r_20_025, AI_r_20_025_trend, AI_extreme_20_025, AI_extreme_20_025_trend = AI_sequence_and_slope_20(CRU_anomaly_025, NDVI_year_025, 0.25)

# The following code converts 0.25° spatial resolution numpy arrays to xarray raster data
longitude_left = np.arange(-180, 0, 0.25)
longitude_right = np.arange(0.25, 180.25, 0.25)
longitude = np.concatenate((longitude_left, longitude_right))
latitude_up = np.arange(90, 0, -0.25)
latitude_down = np.arange(-0.25, -90.25, -0.25)
latitude = np.concatenate((latitude_up, latitude_down))

# 0.25 spatial resolution, 8-neighborhood, 3 time windows
AI_slope_30_025_trend_tif = xr.DataArray(AI_slope_30_025_trend, dims=['y', 'x'], coords=[latitude, longitude])
AI_slope_25_025_trend_tif = xr.DataArray(AI_slope_25_025_trend, dims=['y', 'x'], coords=[latitude, longitude])
AI_slope_20_025_trend_tif = xr.DataArray(AI_slope_20_025_trend, dims=['y', 'x'], coords=[latitude, longitude])

AI_r_30_025_trend_tif = xr.DataArray(AI_r_30_025_trend, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_25_025_trend_tif = xr.DataArray(AI_r_25_025_trend, dims=['y', 'x'], coords=[latitude, longitude])
AI_r_20_025_trend_tif = xr.DataArray(AI_r_20_025_trend, dims=['y', 'x'], coords=[latitude, longitude])

AI_extreme_30_025_trend_tif = xr.DataArray(AI_extreme_30_025_trend, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_25_025_trend_tif = xr.DataArray(AI_extreme_25_025_trend, dims=['y', 'x'], coords=[latitude, longitude])
AI_extreme_20_025_trend_tif = xr.DataArray(AI_extreme_20_025_trend, dims=['y', 'x'], coords=[latitude, longitude])

步骤28： 调用函数，计算多时间窗口，以及趋势分布


C:\anaconda\lib\site-packages\scipy\stats\_stats_mstats_common.py:182: RuntimeWarning: invalid value encountered in double_scalars
  slope = ssxym / ssxm
C:\anaconda\lib\site-packages\scipy\stats\_stats_mstats_common.py:196: RuntimeWarning: invalid value encountered in sqrt
  t = r * np.sqrt(df / ((1.0 - r + TINY)*(1.0 + r + TINY)))
C:\anaconda\lib\site-packages\scipy\stats\_stats_mstats_common.py:199: RuntimeWarning: invalid value encountered in double_scalars
  slope_stderr = np.sqrt((1 - r**2) * ssym / ssxm / df)


In [166]:
print('Step 28_2: Global overall trend for 30-year time window (annual growth rate, and proportion of vegetation cover types?)')
# Annual growth rate is 1.5%
global_mean = np.nanmean(AI_slope_30_025, axis=(1, 2))
# Counting
non_nan_count1 = np.count_nonzero(~np.isnan(AI_slope_30_025_trend))
target_values = [1, 2, 3, 4, 5, 6, 8, 9, 12]

# Count total numbers equal to 1, 2, 3, 4, 5, 6, 8, 9, 12
total_count = 0
for val in target_values:
    total_count += np.count_nonzero(veg == val)

print(f"Total numbers equal to 1, 2, 3, 4, 5, 6, 8, 9, 12: {total_count:,}")
non_nan_count1, total_count

步骤28_2：30年时间窗口的全球整体趋势（年均增长率，以及占植被覆盖类型的多少？）
等于1、2、3、4、5的数字总数: 155,891


(18154, 155891)

In [1]:
print('Step 29_1: Distribution plots for 20-year time window')
fig, ax = plt.subplots(3, 2, figsize=(12, 8))
world = gpd.read_file(r'D:\JYQ_data\JYQ\世界地图\世界各大洲际的边界/各洲的边界线.shp')
# Use loop to plot the same world map boundaries on each subplot
world.plot(ax=ax[0, 0], color='gray', linewidth=0.1)
world.plot(ax=ax[1, 0], color='gray', linewidth=0.1)
world.plot(ax=ax[2, 0], color='gray', linewidth=0.1)
sm1 = ScalarMappable(cmap='seismic_r', norm=plt.Normalize(vmin=-0.3, vmax=0.3))
sm1.set_array([])  # Need an empty array to create colorbar

AI_slope_20_025_trend_tif.plot(ax=ax[0, 0], cmap='seismic_r', vmin=-0.3, vmax=0.3, label='CRU', add_colorbar=False)
AI_r_20_025_trend_tif.plot(ax=ax[1, 0], cmap='seismic_r', vmin=-0.3, vmax=0.3, label='CRU', add_colorbar=False)
AI_extreme_20_025_trend_tif.plot(ax=ax[2, 0], cmap='seismic_r', vmin=-0.3, vmax=0.3, label='CRU', add_colorbar=False)
ax[2, 1].axis('off')  # Hide axis
ax[0, 0].set_title('Trends of AI$_{\mathrm{S}}$(20-year time window)')
ax[1, 0].set_title(r'Trends of AI$_{\mathrm{R}}$ (20-year time window)')
ax[2, 0].set_title('Trends of AI$_{\mathrm{E}}$(20-year time window)')
cbar0 = fig.colorbar(sm1, ax=ax[0, 0], orientation='vertical', shrink=1)
cbar1 = fig.colorbar(sm1, ax=ax[1, 0], orientation='vertical', shrink=1)
cbar2 = fig.colorbar(sm1, ax=ax[2, 0], orientation='vertical', shrink=1)
ax[0, 0].set_xlabel('')
ax[0, 0].set_ylabel('')
ax[1, 0].set_xlabel('')
ax[1, 0].set_ylabel('')
ax[2, 0].set_xlabel('')
ax[2, 0].set_ylabel('')
ax[0, 1].set_xlabel('')
ax[0, 1].set_ylabel('')
ax[1, 1].set_xlabel('')
ax[1, 1].set_ylabel('')

# Manually add longitude information
for a in [ax[0, 0], ax[1, 0], ax[2, 0]]:
    a.set_xticks([-180, -120, -60, 0, 60, 120, 180])  # Longitude
    a.set_xticklabels(['180°W', '120°W', '60°W', '0°', '60°E', '120°E', '180°E'])
    a.set_xlim([-180, 180])  # Longitude range

# Manually add latitude information
for a in [ax[0, 0], ax[1, 0], ax[2, 0]]:
    a.set_yticks([-90, -60, -30, 0, 30, 60, 90])  # Latitude
    a.set_yticklabels(['90°S', '60°S', '30°S', '0°', '30°N', '60°N', '90°N'])
    a.set_ylim([-90, 90])  # Latitude range
ax[0, 1].axis('off')  # Turn off axis display
ax[1, 1].axis('off')  # Turn off axis display
ax[2, 1].axis('off')  # Turn off axis display
plt.tight_layout()
# plt.savefig(r'D:\BaiduSyncdisk\毕业论文\英文论文的图\三图\图3_1', dpi=1500, bbox_inches='tight')
plt.show()


Step 29_1: Distribution plots for 20-year time window


In [374]:
# import scipy.stats as stats
print('Step 29_2: Pairwise comparison plots for different time windows')
# Retain common parts of the three arrays
valid_mask = ~np.isnan(AI_slope_30_025_trend) & ~np.isnan(AI_slope_25_025_trend) & ~np.isnan(AI_slope_20_025_trend)
A = AI_slope_30_025_trend.copy()
B = AI_slope_25_025_trend.copy()
C = AI_slope_20_025_trend.copy()
A_valid = A[valid_mask]
B_valid = B[valid_mask]
C_valid = C[valid_mask]

# Create DataFrame
data = pd.DataFrame({
    '30_years': A_valid,
    '25_years': B_valid,
    '20_years': C_valid
})
corr_matrix = data.corr()

print(corr_matrix)
# Set theme style
sns.set_theme(style="ticks")

# Draw pairplot
g = sns.pairplot(data)

# Add linear fit line, equation, R and P values to each scatter plot
for i in range(len(g.axes)):
    for j in range(len(g.axes)):
        if i != j:  # Skip diagonal histograms
            ax = g.axes[i, j]
            x = data[data.columns[j]]
            y = data[data.columns[i]]

            # Calculate linear fit (statsmodels)
            x_const = sm.add_constant(x)
            model = sm.OLS(y, x_const).fit()
            slope, intercept = model.params[1], model.params[0]

            # Calculate Pearson correlation coefficient and P-value (SciPy)
            r, p_value = stats.pearsonr(x, y)

            # Draw fit line
            sns.regplot(x=data.columns[j], y=data.columns[i], data=data,
                        scatter_kws={'s': 20, 'alpha': 0.5},
                        line_kws={'color': 'red', 'lw': 1},
                        ax=ax)

            # Add text: fitting equation + R + P-value
            text = (
                f"y = {slope:.2f}x + {intercept:.2f}\n"  # Equation
                f"R = {r:.2f}\n"  # Correlation coefficient
                f"P = {p_value:.2f}"  # P-value
            )
            ax.text(0.05, 0.8, text, transform=ax.transAxes,
                    fontsize=8, color='red', va='top')

plt.show()


步骤29_2：不同时间窗口的两两比对图
          30_years  25_years  20_years
30_years  1.000000  0.601285  0.510176
25_years  0.601285  1.000000  0.614334
20_years  0.510176  0.614334  1.000000


In [87]:
print('Step 30: Proof graph showing insufficient effectiveness of trends in 20-year time window')
fig, ax = plt.subplots(figsize=(8, 8))
world = gpd.read_file(r'D:/JYQ_data/JYQ/世界地图/world_line/new_world_line.shp')
# Use loop to plot the same world map boundaries on each subplot
world.plot(ax=ax, color='gray', linewidth=0.1)
sm1 = plt.cm.ScalarMappable(cmap='seismic_r', norm=plt.Normalize(vmin=-0.3, vmax=0.3))
sm1.set_array([])  # Need an empty array to create colorbar

AI_slope_20_025_trend_tif.plot(ax=ax, cmap='seismic_r', vmin=-0.5, vmax=0.5, label='CRU', add_colorbar=False)
ax.set_title('Trends of AI_slope (20-year time window)')
cbar0 = fig.colorbar(sm1, orientation='vertical', shrink=0.6)  # Modify shrink parameter
ax.set_xlabel('Latitude')  # Modify label
ax.set_ylabel('Longitude')  # Modify label
plt.tight_layout()
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\不升邻域的结果', dpi=1000, bbox_inches='tight')
plt.show()

步骤30：20年时间窗口的趋势有效性太少的证明图


C:\Users\JYQ\AppData\Local\Temp\ipykernel_6984\2855960348.py:11: MatplotlibDeprecationWarning: Unable to determine Axes to steal space for Colorbar. Using gca(), but will raise in the future. Either provide the *cax* argument to use as the Axes for the Colorbar, provide the *ax* argument to steal space from it, or add *mappable* to an Axes.
  cbar0 = fig.colorbar(sm1, orientation='vertical', shrink=0.6)  # 修改shrink参数


In [137]:
print('Step 31-1: Frequency histograms for three time scales')
fig = plt.figure(figsize=(5.5, 5))
sns.kdeplot(AI_slope_30_025_trend.flatten(), fill=True, color='red', alpha=0.1, label='30-year', clip=(-0.3, 0.3))
sns.kdeplot(AI_slope_25_025_trend.flatten(), fill=True, color='blue', alpha=0.1, label='25-year', clip=(-0.3, 0.3))
sns.kdeplot(AI_slope_20_025_trend.flatten(), fill=True, color='green', alpha=0.1, label='20-year', clip=(-0.3, 0.3))
plt.xlabel('AI_slope trends')
plt.legend(loc='upper right')
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\不同时间窗口的分布图', dpi=1000, bbox_inches='tight')
plt.show()

步骤31：三种时间尺度的频率直方图


In [116]:
print('Step 31-2: Distribution plots for four cases of trends and positive/negative responses')
# Positive response set to 1, negative response set to -1. Trend increase set to 1, trend decrease set to 2
array1 = AI_slope_CRU_025_8.copy()
array2 = AI_slope_30_025_trend.copy()
array1.shape, array2.shape
mask1 = np.where(array1 > 0, 1, np.where(array1 < 0, -1, np.nan))
mask2 = np.where(array2 > 0, 1, np.where(array2 < 0, 2, np.nan))
mask3 = mask1 * mask2  # Now mask3 contains only 4 indicators: 1, -1, 2, -2

# save_ok(mask3, '四种分类3')
unique_values0, counts0 = np.unique(mask3, return_counts=True)  # counts0 represents the number of points for each climate zone
for value, count in zip(unique_values0, counts0):
    print(f"Value {value} appears {count} times")
# Data preparation
values = [3656, 5771, 2585, 1789]
labels = ['N_down', 'N_up', 'P_up', 'P_down']
colors = ['red', 'orange', 'blue', 'green']  # Custom colors
explode = (0.08, 0, 0.05, 0)  # Highlight the first slice

# Create pie chart
fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    values,
    labels=labels,
    autopct='%1.1f%%',  # Display percentages
    startangle=90,  # Start angle
    colors=colors,
    explode=explode,
    shadow=True,  # Add shadow
    textprops={'fontsize': 12}
)

# Beautify labels
plt.setp(autotexts, size=12, weight="bold")  # Bold percentage text
# Display chart
plt.tight_layout()
plt.show()

步骤31：趋势和正负响应的四种情况分布图
数值 -2.0 出现了 3656 次
数值 -1.0 出现了 5771 次
数值 1.0 出现了 2585 次
数值 2.0 出现了 1789 次
数值 nan 出现了 1022999 次


In [90]:
print('Step 32: Using Australia to demonstrate differences between three time windows')
print("Australia region verification")  # (486, 1220)
# AI_slope_30_025_trend[486, 1220], AI_slope_25_025_trend[486, 1220], AI_slope_20_025_trend[486, 1220]

fig, ax = plt.subplots()
data1 = AI_slope_30_025[:, 486, 1220]
x1 = np.arange(len(data1))
data2 = AI_slope_25_025[:, 486, 1220]
x2 = np.arange(len(data2))
data3 = AI_slope_20_025[:, 486, 1220]
x3 = np.arange(len(data3))

# Plot data points and lines
ax.plot(x1, data1, color='blue', marker='o', linestyle='-')
ax.plot(x2, data2, color='red', marker='o', linestyle='-')
ax.plot(x3, data3, color='green', marker='o', linestyle='-')

# Linear fitting


def plot_linear_fit(x, y, color, label):
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    y_fit = slope * x + intercept
    ax.plot(x, y_fit, color=color, linestyle='--', label=f'{label} fit (R²={r_value**2:.2f})')
    ax.text(x[-1], y_fit[-1], f'y = {slope:.2f}x + {intercept:.2f}\nR² = {r_value**2:.2f}', fontsize=10, ha='right', color=color)


# Plot fit lines and equations
plot_linear_fit(x1, data1, 'blue', '30-year window')
plot_linear_fit(x2, data2, 'red', '25-year window')
plot_linear_fit(x3, data3, 'green', '20-year window')

# Set chart title and axis labels
ax.set_title('Trends of Different Windows')
ax.set_xlabel('Time')
ax.set_ylabel('AI_slope')
# Add legend
ax.legend()
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\澳大利亚验证时间趋势', dpi=1000, bbox_inches='tight')
# Display chart
plt.show()

步骤32：用澳大利亚证明三种时间窗口的差异性
澳大利亚地区验证


In [97]:
print('Step 33: Frequency histograms for three time scales')
fig = plt.figure(figsize=(5.5, 5))
sns.kdeplot(AI_slope_30_025_trend.flatten(), fill=True, color='red', alpha=0.1, label='30-year', clip=(-0.3, 0.3))
sns.kdeplot(AI_slope_25_025_trend.flatten(), fill=True, color='blue', alpha=0.1, label='25-year', clip=(-0.3, 0.3))
sns.kdeplot(AI_slope_20_025_trend.flatten(), fill=True, color='green', alpha=0.1, label='20-year', clip=(-0.3, 0.3))
plt.xlabel('AI_slope trends')
plt.legend(loc='upper right')
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\不同时间窗口的分布图', dpi=1000, bbox_inches='tight')
plt.show()

步骤33：三种时间尺度的频率直方图


In [155]:
print("Step 34: Function definition for quantifying uncertainty effects of different precipitation and vegetation data on results")
def caluate_AI_sequence_and_slope_30_LAI(array_A, array_B):  # 30-year time window, input is a 41 * 3 * 3 window
    AI_slope_list = []  # Used to store AI_extreme values after each iteration
    for i in range(11):
        pr_window = array_A[i:i + 30, :, :]
        NDVI_window = array_B[i:i + 30, :, :]  # Length 30, step size 1, total 11 sliding steps, each sliding extracts a 30 * 3 * 3 window

        # Flatten arrays and calculate three indicators as usual
        pr_window_flatten = pr_window.flatten()
        NDVI_window_flatten = NDVI_window.flatten()  # Flatten
        pr_window_flatten_purified, NDVI_window_flatten_purified = jinhua(pr_window_flatten, NDVI_window_flatten)
        pr_window_sorted, NDVI_window_sorted = sort_by_A(pr_window_flatten_purified,
                                                         NDVI_window_flatten_purified)  # Sort
        # First half is dry period, extract and start fitting
        mid = len(pr_window_sorted) // 2
        x1 = pr_window_sorted[0:mid]
        y1 = NDVI_window_sorted[0:mid]
        x1 = np.array(x1)
        y1 = np.array(y1)
        slope1, intercept1, r1, p_value1, std_err1 = linregress(x1, y1)
        # Second half is wet period, extract and start fitting
        x2 = pr_window_sorted[mid:]
        y2 = NDVI_window_sorted[mid:]
        x2 = np.array(x2)
        y2 = np.array(y2)
        slope2, intercept2, r2, p_value2, std_err2 = linregress(x2, y2)

        a = np.nan
        # If both dry and wet years respond
        if slope1 > 0 and p_value1 < 0.1 and slope2 > 0 and p_value2 < 0.1:  # Relax conditions appropriately here
            a = (slope2 - slope1) / (slope2 + slope1)
        # If dry year responds, wet year negatively responds
        elif slope1 > 0 and p_value1 < 0.1 and slope2 < 0 and p_value2 < 0.1:
            a = -1
        # If dry year responds, wet year does not respond
        elif slope1 > 0 and p_value1 < 0.1 and p_value2 > 0.1:
            a = -1
        # If dry year does not respond
        elif slope1 < 0 or p_value1 > 0.1:
            a = np.nan
        AI_slope_list.append(a)

    AI_slope_arr = np.array(AI_slope_list)  # Convert list to array for further operations
    non_nan_values = AI_slope_arr[~np.isnan(AI_slope_arr)]  # Extract non-NaN values
    if len(non_nan_values) == 0:
        slope4 = np.nan
    else:
        x = np.arange(1, len(non_nan_values) + 1)  # Calculate AI index trend based on obtained AI index time series
        slope4, intercept4, r_value4, p_value4, std_err4 = linregress(x, non_nan_values)
        if p_value4 > 0.1:  # If AI index trend does not show significant upward trend, discard it
            slope4 = np.nan
    return AI_slope_arr, slope4


def AI_sequence_and_slope_30_LAI(pr_anomaly, NDVI_anomaly, n):
    if n == 0.25:
        AI_slope_30_025 = np.empty((11, 720, 1440), dtype=float)
        AI_slope_30_025_trend = np.empty((720, 1440), dtype=float)

        for i in range(720):
            for j in range(1440):
                # If extracted data is all empty
                array_A = pr_anomaly[:, i - 2:i + 3, j - 2:j + 3]
                array_B = NDVI_anomaly[:, i - 2:i + 3, j - 2:j + 3]
                if len(array_A.flatten()) == 0 or len(array_B.flatten()) == 0 or np.isnan(array_A).any() or np.isnan(array_B).any():
                    AI_slope_30_025[:, i, j] = np.nan
                    AI_slope_30_025_trend[i, j] = np.nan
                else:
                    AI_slope_arr, slope = caluate_AI_sequence_and_slope_30_LAI(array_A, array_B)
                    AI_slope_30_025[:, i, j] = AI_slope_arr
                    AI_slope_30_025_trend[i, j] = slope
        return AI_slope_30_025, AI_slope_30_025_trend


AI_slope_LAI_30_025, AI_slope_LAI_30_025_trend = AI_sequence_and_slope_30_LAI(CRU_anomaly_025[0:40, :, :], LAI_anomaly_025, 0.25)
AI_slope_Terra_30_025, AI_slope_Terra_30_025_trend = AI_sequence_and_slope_30(Terra_anomaly_025, NDVI_year_025, 0.25)

步骤35：不同降水数据，不同植被数据，对结果产生的不确定影响的函数定义


C:\anaconda\lib\site-packages\scipy\stats\_stats_mstats_common.py:182: RuntimeWarning: invalid value encountered in double_scalars
  slope = ssxym / ssxm
C:\anaconda\lib\site-packages\scipy\stats\_stats_mstats_common.py:196: RuntimeWarning: invalid value encountered in sqrt
  t = r * np.sqrt(df / ((1.0 - r + TINY)*(1.0 + r + TINY)))
C:\anaconda\lib\site-packages\scipy\stats\_stats_mstats_common.py:199: RuntimeWarning: invalid value encountered in double_scalars
  slope_stderr = np.sqrt((1 - r**2) * ssym / ssxm / df)


In [157]:
print('Step 35: Pairwise comparison plots for different data sources')
# Data preprocessing part remains unchanged (omitted)
valid_mask = ~np.isnan(AI_slope_30_025_trend) & ~np.isnan(AI_slope_Terra_30_025_trend) & ~np.isnan(AI_slope_LAI_30_025_trend)
A = AI_slope_30_025_trend.copy()
B = AI_slope_Terra_30_025_trend.copy()
C = AI_slope_LAI_30_025_trend.copy()
A_valid = A[valid_mask]
B_valid = B[valid_mask]
C_valid = C[valid_mask]

# Create DataFrame
data = pd.DataFrame({
    'CRU_NDVI': A_valid,
    'Terra_NDVI': B_valid,
    'CRU_LAI': C_valid
})
# Set theme style
sns.set_theme(style="ticks", font_scale=0.9)

# Create PairGrid object, customize diagonal and off-diagonal plots
g = sns.PairGrid(data, diag_sharey=False, height=2.5)
# Diagonal: Kernel density plot
g.map_diag(sns.kdeplot, fill=True, alpha=0.8, color='#1f77b4',
           edgecolor='black', linewidth=0.5, bw_adjust=0.8)
# Off-diagonal: 2D kernel density contours or Hexbin heatmap
g.map_upper(sns.kdeplot, cmap='Blues', fill=True, levels=10, alpha=0.7)
g.map_lower(sns.histplot, cmap='Reds', cbar=True,
            bins=50, alpha=0.9, edgecolor='gray', linewidth=0.1)

# Add fitted lines and equations (only in lower-left triangle region)
for i in range(len(data.columns)):
    for j in range(len(data.columns)):
        if i > j:  # Add regression lines only in lower triangle region
            ax = g.axes[i, j]
            x = data[data.columns[j]]
            y = data[data.columns[i]]
            sns.regplot(x=x, y=y, scatter=False,
                        line_kws={'color': 'darkred', 'lw': 1.5}, ax=ax)
            model = sm.OLS(y, sm.add_constant(x)).fit()
            equation = f"ρ={corr_matrix.iloc[i, j]:.2f}\ny={model.params[1]:.2f}x+{model.params[0]:.2f}"
            ax.text(0.05, 0.85, equation, transform=ax.transAxes,
                    fontsize=7, color='darkred', ha='left')

# Optimize layout and annotations
plt.suptitle("Cross-Data Comparison (Density & Hexbin)", y=1.02, fontsize=11)
g.fig.subplots_adjust(wspace=0.1, hspace=0.1)
[ax.set_xticks([]) if i % 3 != 0 else None for i, ax in enumerate(g.axes.flat)]  # Reduce tick density

# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\不同数据源的时间趋势的比较', dpi=1000, bbox_inches='tight')
plt.show()

步骤36：不同数据源的两两比对图


In [159]:
print('Step 36: Pairwise comparison plots for different data sources')
# Retain common parts of the three arrays
valid_mask = ~np.isnan(AI_slope_30_025_trend) & ~np.isnan(AI_slope_Terra_30_025_trend) & ~np.isnan(AI_slope_LAI_30_025_trend)
A = AI_slope_30_025_trend.copy()
B = AI_slope_Terra_30_025_trend.copy()
C = AI_slope_LAI_30_025_trend.copy()
A_valid = A[valid_mask]
B_valid = B[valid_mask]
C_valid = C[valid_mask]

# Create DataFrame
data = pd.DataFrame({
    'CRU_NDVI': A_valid,
    'Terra_NDVI': B_valid,
    'CRU_LAI': C_valid
})
corr_matrix = data.corr()
print(corr_matrix)
# Set theme style
sns.set_theme(style="ticks")

# Draw pairplot
g = sns.pairplot(data)

for i in range(len(g.axes)):
    for j in range(len(g.axes)):
        if i != j:  # Skip diagonal histograms
            ax = g.axes[i, j]
            x = data[data.columns[j]]
            y = data[data.columns[i]]

            # Calculate linear fit (statsmodels)
            x_const = sm.add_constant(x)
            model = sm.OLS(y, x_const).fit()
            slope, intercept = model.params[1], model.params[0]

            # Calculate Pearson correlation coefficient and P-value (SciPy)
            r, p_value = stats.pearsonr(x, y)

            # Draw fit line
            sns.regplot(x=data.columns[j], y=data.columns[i], data=data,
                        scatter_kws={'s': 20, 'alpha': 0.5},
                        line_kws={'color': 'red', 'lw': 1},
                        ax=ax)

            # Format P-value display (keep two decimal places, show "P<0.01" when less than 0.01)
            p_text = f"P = {p_value:.2f}" if p_value >= 0.01 else "P < 0.01"

            # Add text: fitting equation + R + P-value
            text = (
                f"y = {slope:.2f}x + {intercept:.2f}\n"
                f"R = {r:.2f}\n"
                f"{p_text}"
            )
            ax.text(0.05, 0.8, text, transform=ax.transAxes,
                    fontsize=8, color='red', va='top')
plt.show()


步骤36：不同数据源的两两比对图
            CRU_NDVI  Terra_NDVI   CRU_LAI
CRU_NDVI    1.000000    0.551925  0.235216
Terra_NDVI  0.551925    1.000000  0.178569
CRU_LAI     0.235216    0.178569  1.000000


# 不同时间尺度的趋势统计

In [553]:
print("Trend Statistics Step 1_1: Numerical calculation of trend changes for different vegetation types, serving for step 2")
# First convert the trend to a binary map
array = AI_r_20_025_trend
array[array < 0] = -1
array[array > 0] = 1

result = array * veg
unique_values, counts = np.unique(result, return_counts=True)
value_counts = dict(zip(unique_values, counts))
print(value_counts)

趋势统计步骤1_1：不同的植被类型的趋势变化的数值计算，为了步骤2服务的
{-13.0: 7, -12.0: 499, -11.0: 126, -10.0: 1613, -8.0: 1162, -6.0: 2059, -5.0: 59, -4.0: 40, -3.0: 21, -2.0: 309, -1.0: 11, 1.0: 15, 2.0: 175, 3.0: 31, 4.0: 129, 5.0: 177, 6.0: 2303, 8.0: 2618, 10.0: 6009, 11.0: 630, 12.0: 1758, 13.0: 37, nan: 1017012}


In [552]:
print("Trend Statistics Step 1_2: Numerical calculation of trend changes for different vegetation types, serving for step 2")
array = AI_slope_20_025_trend
array[array < 0] = -1
array[array > 0] = 1

result = array * veg
unique_values, counts = np.unique(result, return_counts=True)
value_counts = dict(zip(unique_values, counts))
print(value_counts)

趋势统计步骤1_2：不同的植被类型的趋势变化的数值计算，为了步骤2服务的
{-13.0: 7, -12.0: 573, -11.0: 131, -10.0: 1706, -8.0: 1230, -6.0: 2046, -5.0: 61, -4.0: 36, -3.0: 25, -2.0: 279, -1.0: 12, 1.0: 14, 2.0: 193, 3.0: 29, 4.0: 134, 5.0: 182, 6.0: 1938, 8.0: 2578, 10.0: 5758, 11.0: 608, 12.0: 1676, 13.0: 35, nan: 1017549}


In [551]:
print("Trend Statistics Step 1_3: Numerical calculation of trend changes for different vegetation types, serving for step 2")
array = AI_extreme_20_025_trend
array[array < 0] = -1
array[array > 0] = 1

result = array * veg
unique_values, counts = np.unique(result, return_counts=True)
value_counts = dict(zip(unique_values, counts))
print(value_counts)

趋势统计步骤1_3：不同的植被类型的趋势变化的数值计算，为了步骤2服务的
{-13.0: 10, -12.0: 510, -11.0: 125, -10.0: 1879, -8.0: 1187, -6.0: 1535, -5.0: 68, -4.0: 32, -3.0: 23, -2.0: 325, -1.0: 10, 1.0: 16, 2.0: 160, 3.0: 29, 4.0: 140, 5.0: 176, 6.0: 2588, 8.0: 2646, 10.0: 5825, 11.0: 598, 12.0: 1701, 13.0: 28, nan: 1017189}


In [3]:
print("Trend Statistics Step 2: Double-layer pie chart of 20-year time window trends for different vegetation types (AI_slope)")
# Data
outer_labels = ['Shrub', 'Grass', 'Savanna', 'Farm', 'Others', 'Others', 'Farm', 'Shrub', 'Savanna', 'Grass']  # Outer pie chart labels
outer_sizes = [0.1063, 0.0886, 0.0639, 0.0298, 0.0286, 0.062, 0.0871, 0.1007, 0.1339, 0.2991]  # Outer pie chart data

inner_labels = ['Downward', 'Upward']  # Inner pie chart data
inner_sizes = [0.3172, 0.6828]  # Inner pie chart data

# Colors
outer_colors = ['Pink', 'Green', 'Brown', 'Orange', 'gray', '#D3D3D3', 'Orange', 'Pink', 'Brown', 'Green']  # Outer colors
inner_colors = ['red', 'blue']  # Inner colors

# Create canvas
fig, ax = plt.subplots()

# Custom number label format (smaller font size)
def autopct_format(pct):
    return f'{pct:.1f}%' if pct > 0 else ''


# Outer pie chart
wedges_outer, texts_outer, autotexts_outer = ax.pie(
    outer_sizes,
    labels=outer_labels,
    radius=1,
    colors=outer_colors,
    wedgeprops=dict(width=0.3, edgecolor='w', linewidth=1, alpha=0.7),
    autopct=autopct_format,  # Use custom format function
    pctdistance=0.85,
    labeldistance=1.05,
    textprops={'fontsize': 16, 'fontweight': 'bold'}
)

# Set outer number label font size (smaller)
for autotext in autotexts_outer:
    autotext.set_fontsize(10)  # Reduce number label size
    autotext.set_fontweight('bold')  # Bold

# Inner pie chart
wedges_inner, texts_inner, autotexts_inner = ax.pie(
    inner_sizes,
    labels=inner_labels,
    radius=0.7,
    colors=inner_colors,
    wedgeprops=dict(width=0.3, edgecolor='w', alpha=0.7),
    autopct=autopct_format,  # Use custom format function
    pctdistance=0.5,
    labeldistance=0.35,
    textprops={'fontsize': 12, 'fontweight': 'bold'}
)

# Set inner number label font size (smaller)
for autotext in autotexts_inner:
    autotext.set_fontsize(10)  # Reduce number label size
    autotext.set_fontweight('bold')  # Bold

# Draw two dashed lines between the two parts of the inner ring, extending outside the ring
theta1 = 0  # Start angle of first segment
theta2 = 360 * inner_sizes[0] / sum(inner_sizes)  # End angle of first segment
center = (0, 0)  # Center of pie chart
inner_radius = 0.7  # Radius of inner ring
outer_radius = 1.0  # Radius of outer ring
extension_length = 0.2  # Length to extend outward

# Draw first dashed line (start angle)
ax.plot([center[0] + inner_radius * np.cos(theta1 * np.pi / 180), center[0] + (outer_radius + extension_length) * np.cos(theta1 * np.pi / 180)],
        [center[1] + inner_radius * np.sin(theta1 * np.pi / 180), center[1] + (outer_radius + extension_length) * np.sin(theta1 * np.pi / 180)],
        linestyle='--', color='black', linewidth=1.5)

# Draw second dashed line (end angle)
ax.plot([center[0] + inner_radius * np.cos(theta2 * np.pi / 180), center[0] + (outer_radius + extension_length) * np.cos(theta2 * np.pi / 180)],
        [center[1] + inner_radius * np.sin(theta2 * np.pi / 180), center[1] + (outer_radius + extension_length) * np.sin(theta2 * np.pi / 180)],
        linestyle='--', color='black', linewidth=1.5)

# Set as circle
ax.set_aspect('equal')

# Save image (optional)
plt.savefig(r'D:\BaiduSyncdisk\毕业论文\英文论文的图\三图\图3_3的饼图AI_slope2', dpi=1500, bbox_inches='tight', transparent=True)

plt.show()

趋势统计步骤2：不同植被类型的20年时间窗口的时间趋势双层饼图（AI_slope）


In [2]:
print("Trend Statistics Step 2: Double-layer pie chart of 20-year time window trends for different vegetation types (AI_r)")
# Data
outer_labels = ['Shrub', 'Grass', 'Savanna', 'Farm', 'Others', 'Others', 'Farm', 'Shrub', 'Savanna', 'Grass']  # Outer pie chart labels
outer_sizes = [0.1041, 0.0815, 0.0587, 0.0252, 0.029, 0.0603, 0.0888, 0.1164, 0.1323, 0.3037]  # Outer pie chart data

inner_labels = ['Downward', 'Upward']  # Inner pie chart data
inner_sizes = [0.2985, 0.7015]  # Inner pie chart data

# Colors
outer_colors = ['Pink', 'Green', 'Brown', 'Orange', 'gray', '#D3D3D3', 'Orange', 'Pink', 'Brown', 'Green']  # Outer colors
inner_colors = ['red', 'blue']  # Inner colors

# Create canvas
fig, ax = plt.subplots()

# Custom number label format (smaller font size)
def autopct_format(pct):
    return f'{pct:.1f}%' if pct > 0 else ''


# Outer pie chart
wedges_outer, texts_outer, autotexts_outer = ax.pie(
    outer_sizes,
    labels=outer_labels,
    radius=1,
    colors=outer_colors,
    wedgeprops=dict(width=0.3, edgecolor='w', linewidth=1, alpha=0.7),
    autopct=autopct_format,  # Use custom format function
    pctdistance=0.85,
    labeldistance=1.05,
    textprops={'fontsize': 16, 'fontweight': 'bold'}
)

# Set outer number label font size (smaller)
for autotext in autotexts_outer:
    autotext.set_fontsize(10)  # Reduce number label size
    autotext.set_fontweight('bold')  # Bold

# Inner pie chart
wedges_inner, texts_inner, autotexts_inner = ax.pie(
    inner_sizes,
    labels=inner_labels,
    radius=0.7,
    colors=inner_colors,
    wedgeprops=dict(width=0.3, edgecolor='w', alpha=0.7),
    autopct=autopct_format,  # Use custom format function
    pctdistance=0.5,
    labeldistance=0.35,
    textprops={'fontsize': 12, 'fontweight': 'bold'}
)

# Set inner number label font size (smaller)
for autotext in autotexts_inner:
    autotext.set_fontsize(10)  # Reduce number label size
    autotext.set_fontweight('bold')  # Bold

# Draw two dashed lines between the two parts of the inner ring, extending outside the ring
theta1 = 0  # Start angle of first segment
theta2 = 360 * inner_sizes[0] / sum(inner_sizes)  # End angle of first segment
center = (0, 0)  # Center of pie chart
inner_radius = 0.7  # Radius of inner ring
outer_radius = 1.0  # Radius of outer ring
extension_length = 0.2  # Length to extend outward

# Draw first dashed line (start angle)
ax.plot([center[0] + inner_radius * np.cos(theta1 * np.pi / 180), center[0] + (outer_radius + extension_length) * np.cos(theta1 * np.pi / 180)],
        [center[1] + inner_radius * np.sin(theta1 * np.pi / 180), center[1] + (outer_radius + extension_length) * np.sin(theta1 * np.pi / 180)],
        linestyle='--', color='black', linewidth=1.5)

# Draw second dashed line (end angle)
ax.plot([center[0] + inner_radius * np.cos(theta2 * np.pi / 180), center[0] + (outer_radius + extension_length) * np.cos(theta2 * np.pi / 180)],
        [center[1] + inner_radius * np.sin(theta2 * np.pi / 180), center[1] + (outer_radius + extension_length) * np.sin(theta2 * np.pi / 180)],
        linestyle='--', color='black', linewidth=1.5)

# Set as circle
ax.set_aspect('equal')

# Save image (optional)
plt.savefig(r'D:\BaiduSyncdisk\毕业论文\英文论文的图\三图\图3_3的饼图AI_r2', dpi=1500, bbox_inches='tight', transparent=True)

plt.show()

趋势统计步骤2：不同植被类型的20年时间窗口的时间趋势双层饼图（AI_r）


In [1]:
print("Trend Statistics Step 2: Double-layer pie chart of 20-year time window trends for different vegetation types (AI_extreme)")
# Data
outer_labels = ['Shrub', 'Grass', 'Savanna', 'Farm', 'Others', 'Others', 'Farm', 'Shrub', 'Savanna', 'Grass']  # Outer pie chart labels
outer_sizes = [0.0783, 0.0958, 0.0605, 0.026, 0.0303, 0.0585, 0.0867, 0.132, 0.1349, 0.297]  # Outer pie chart data

inner_labels = ['Downward', 'Upward']  # Inner pie chart data
inner_sizes = [0.2909, 0.7091]  # Inner pie chart data

# Colors
outer_colors = ['Pink', 'Green', 'Brown', 'Orange', 'gray', '#D3D3D3', 'Orange', 'Pink', 'Brown', 'Green']  # Outer colors
inner_colors = ['red', 'blue']  # Inner colors

# Create canvas
fig, ax = plt.subplots()

# Custom number label format (smaller font size)
def autopct_format(pct):
    return f'{pct:.1f}%' if pct > 0 else ''


# Outer pie chart
wedges_outer, texts_outer, autotexts_outer = ax.pie(
    outer_sizes,
    labels=outer_labels,
    radius=1,
    colors=outer_colors,
    wedgeprops=dict(width=0.3, edgecolor='w', linewidth=1, alpha=0.7),
    autopct=autopct_format,  # Use custom format function
    pctdistance=0.85,
    labeldistance=1.05,
    textprops={'fontsize': 16, 'fontweight': 'bold'}
)

# Set outer number label font size (smaller)
for autotext in autotexts_outer:
    autotext.set_fontsize(10)  # Reduce number label size
    autotext.set_fontweight('bold')  # Bold

# Inner pie chart
wedges_inner, texts_inner, autotexts_inner = ax.pie(
    inner_sizes,
    labels=inner_labels,
    radius=0.7,
    colors=inner_colors,
    wedgeprops=dict(width=0.3, edgecolor='w', alpha=0.7),
    autopct=autopct_format,  # Use custom format function
    pctdistance=0.5,
    labeldistance=0.35,
    textprops={'fontsize': 12, 'fontweight': 'bold'}
)

# Set inner number label font size (smaller)
for autotext in autotexts_inner:
    autotext.set_fontsize(10)  # Reduce number label size
    autotext.set_fontweight('bold')  # Bold

# Draw two dashed lines between the two parts of the inner ring, extending outside the ring
theta1 = 0  # Start angle of first segment
theta2 = 360 * inner_sizes[0] / sum(inner_sizes)  # End angle of first segment
center = (0, 0)  # Center of pie chart
inner_radius = 0.7  # Radius of inner ring
outer_radius = 1.0  # Radius of outer ring
extension_length = 0.2  # Length to extend outward

# Draw first dashed line (start angle)
ax.plot([center[0] + inner_radius * np.cos(theta1 * np.pi / 180), center[0] + (outer_radius + extension_length) * np.cos(theta1 * np.pi / 180)],
        [center[1] + inner_radius * np.sin(theta1 * np.pi / 180), center[1] + (outer_radius + extension_length) * np.sin(theta1 * np.pi / 180)],
        linestyle='--', color='black', linewidth=1.5)

# Draw second dashed line (end angle)
ax.plot([center[0] + inner_radius * np.cos(theta2 * np.pi / 180), center[0] + (outer_radius + extension_length) * np.cos(theta2 * np.pi / 180)],
        [center[1] + inner_radius * np.sin(theta2 * np.pi / 180), center[1] + (outer_radius + extension_length) * np.sin(theta2 * np.pi / 180)],
        linestyle='--', color='black', linewidth=1.5)

# Set as circle
ax.set_aspect('equal')

# Save image (optional)
plt.savefig(r'D:\BaiduSyncdisk\毕业论文\英文论文的图\三图\图3_3的饼图AI_extreme2', dpi=1500, bbox_inches='tight', transparent=True)

plt.show()

趋势统计步骤2：不同植被类型的20年时间窗口的时间趋势双层饼图（AI_extreme）


In [153]:
print("Trend Statistics Step 3_1: Calculation of time variation trends for 30-year time window in different aridity zones")
# Create masks
mask_dry = (ganshi_025 == 2)
mask_semi_dry = (ganshi_025 == 3)
mask_semi_wet = (ganshi_025 == 4)
mask_wet = (ganshi_025 == 5)

# Extract based on masks
AI_slope_30_025_dry = np.where(mask_dry, AI_slope_30_025, np.nan)  # Apply mask on corresponding two dimensions
AI_slope_30_025_semi_dry = np.where(mask_semi_dry, AI_slope_30_025, np.nan)
AI_slope_30_025_semi_wet = np.where(mask_semi_wet, AI_slope_30_025, np.nan)
AI_slope_30_025_wet = np.where(mask_wet, AI_slope_30_025, np.nan)

# Calculate regional averages
AI_slope_30_025_dry_point = np.nanmean(AI_slope_30_025_dry, axis=(1, 2))
AI_slope_30_025_semi_dry_point = np.nanmean(AI_slope_30_025_semi_dry, axis=(1, 2))
AI_slope_30_025_semi_wet_point = np.nanmean(AI_slope_30_025_semi_wet, axis=(1, 2))
AI_slope_30_025_wet_point = np.nanmean(AI_slope_30_025_wet, axis=(1, 2))

趋势统计步骤3_1：30年不同干燥区的时间变化趋势计算


In [125]:
# AI_slope, 30-year time window
print("Trend Statistics Step 3_2: 30-year time window, trends in different climate zones")
x = generate_year_sequence(1996, 2007)
y_dry = AI_slope_30_025_dry_point
slope_dry, intercept_dry, r_value_dry, p_value_dry, std_err_dry = linregress(x, y_dry)
y2_dry = slope_dry * x + intercept_dry

y_semi_dry = AI_slope_30_025_semi_dry_point
slope_semi_dry, intercept_semi_dry, r_value_semi_dry, p_value_semi_dry, std_err_semi_dry = linregress(x, y_semi_dry)
y2_semi_dry = slope_semi_dry * x + intercept_semi_dry

y_semi_wet = AI_slope_30_025_semi_wet_point
slope_semi_wet, intercept_semi_wet, r_value_semi_wet, p_value_semi_wet, std_err_semi_wet = linregress(x, y_semi_wet)
y2_semi_wet = slope_semi_wet * x + intercept_semi_wet

y_wet = AI_slope_30_025_wet_point
slope_wet, intercept_wet, r_value_wet, p_value_wet, std_err_wet = linregress(x, y_wet)
y2_wet = slope_wet * x + intercept_wet

fig, axs = plt.subplots(2, 2, figsize=(10, 8))


def format_equation(slope, intercept, r_value):
    if intercept >= 0:
        return f'y = {slope:.4f}x + {intercept:.2f}\n$R^2$ = {r_value**2:.2f}'
    else:
        return f'y = {slope:.4f}x - {-intercept:.2f}\n$R^2$ = {r_value**2:.2f}'


# Arid
axs[0, 0].plot(x, y_dry, color='black', linestyle='-', marker='o', markersize=10)
axs[0, 0].plot(x, y2_dry, color='black', linestyle='--', markersize=10)
axs[0, 0].text(0.5, 0.95, format_equation(slope_dry, intercept_dry, r_value_dry),
               transform=axs[0, 0].transAxes, fontsize=12, ha='center', va='top')

# Semi arid
axs[0, 1].plot(x, y_semi_dry, color='black', linestyle='-', marker='o', markersize=10)
axs[0, 1].plot(x, y2_semi_dry, color='black', linestyle='--', markersize=10)
axs[0, 1].text(0.5, 0.95, format_equation(slope_semi_dry, intercept_semi_dry, r_value_semi_dry),
               transform=axs[0, 1].transAxes, fontsize=12, ha='center', va='top')

# Semi humid
axs[1, 0].plot(x, y_semi_wet, color='black', linestyle='-', marker='o', markersize=10)
axs[1, 0].plot(x, y2_semi_wet, color='black', linestyle='--', markersize=10)
axs[1, 0].text(0.5, 0.95, format_equation(slope_semi_wet, intercept_semi_wet, r_value_semi_wet),
               transform=axs[1, 0].transAxes, fontsize=12, ha='center', va='top')

# Humid
axs[1, 1].plot(x, y_wet, color='black', linestyle='-', marker='o', markersize=10)
axs[1, 1].plot(x, y2_wet, color='black', linestyle='--', markersize=10)
axs[1, 1].text(0.5, 0.95, format_equation(slope_wet, intercept_wet, r_value_wet),
               transform=axs[1, 1].transAxes, fontsize=12, ha='center', va='top')

labels = ['(a)', '(b)', '(c)', '(d)']
for i, ax in enumerate(axs.flat):
    ax.text(0.05, 0.95, labels[i], transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top')
plt.tight_layout()
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\时间趋势.png', dpi=600, bbox_inches='tight')
plt.show()


趋势统计步骤3_2：30年时间窗口，不同气候区的趋势


In [95]:
print("Trend Statistics Step 4_1: Calculation of time variation trends for 25-year time window in different aridity zones")
# Extract based on masks
AI_slope_25_025_dry = np.where(mask_dry, AI_slope_25_025, np.nan)  # Apply mask on corresponding two dimensions
AI_slope_25_025_semi_dry = np.where(mask_semi_dry, AI_slope_25_025, np.nan)
AI_slope_25_025_semi_wet = np.where(mask_semi_wet, AI_slope_25_025, np.nan)
AI_slope_25_025_wet = np.where(mask_wet, AI_slope_25_025, np.nan)

# Calculate regional averages
AI_slope_25_025_dry_point = np.nanmean(AI_slope_25_025_dry, axis=(1, 2))
AI_slope_25_025_semi_dry_point = np.nanmean(AI_slope_25_025_semi_dry, axis=(1, 2))
AI_slope_25_025_semi_wet_point = np.nanmean(AI_slope_25_025_semi_wet, axis=(1, 2))
AI_slope_25_025_wet_point = np.nanmean(AI_slope_25_025_wet, axis=(1, 2))

趋势统计步骤4_1：25年不同干燥区的时间变化趋势计算


In [126]:
# AI_slope, 25-year time window
print("Trend Statistics Step 4_2: AI_slope, 25-year time window, trends in different climate zones")
x = np.arange(1994, 2011)  # 1994-2010
y_dry = AI_slope_25_025_dry_point
slope_dry, intercept_dry, r_value_dry, p_value_dry, std_err_dry = linregress(x, y_dry)
y2_dry = slope_dry * x + intercept_dry

y_semi_dry = AI_slope_25_025_semi_dry_point
slope_semi_dry, intercept_semi_dry, r_value_semi_dry, p_value_semi_dry, std_err_semi_dry = linregress(x, y_semi_dry)
y2_semi_dry = slope_semi_dry * x + intercept_semi_dry

y_semi_wet = AI_slope_25_025_semi_wet_point
slope_semi_wet, intercept_semi_wet, r_value_semi_wet, p_value_semi_wet, std_err_semi_wet = linregress(x, y_semi_wet)
y2_semi_wet = slope_semi_wet * x + intercept_semi_wet

y_wet = AI_slope_25_025_wet_point
slope_wet, intercept_wet, r_value_wet, p_value_wet, std_err_wet = linregress(x, y_wet)
y2_wet = slope_wet * x + intercept_wet

fig, axs = plt.subplots(2, 2, figsize=(10, 8))


def format_equation(slope, intercept, r_value):
    if intercept >= 0:
        return f'y = {slope:.4f}x + {intercept:.2f}\n$R^2$ = {r_value**2:.2f}'
    else:
        return f'y = {slope:.4f}x - {-intercept:.2f}\n$R^2$ = {r_value**2:.2f}'


# Arid
axs[0, 0].plot(x, y_dry, color='black', linestyle='-', marker='o', markersize=10)
axs[0, 0].plot(x, y2_dry, color='black', linestyle='--', markersize=10)
axs[0, 0].text(0.5, 0.95, format_equation(slope_dry, intercept_dry, r_value_dry),
               transform=axs[0, 0].transAxes, fontsize=12, ha='center', va='top')

# Semi arid
axs[0, 1].plot(x, y_semi_dry, color='black', linestyle='-', marker='o', markersize=10)
axs[0, 1].plot(x, y2_semi_dry, color='black', linestyle='--', markersize=10)
axs[0, 1].text(0.5, 0.95, format_equation(slope_semi_dry, intercept_semi_dry, r_value_semi_dry),
               transform=axs[0, 1].transAxes, fontsize=12, ha='center', va='top')

# Semi humid
axs[1, 0].plot(x, y_semi_wet, color='black', linestyle='-', marker='o', markersize=10)
axs[1, 0].plot(x, y2_semi_wet, color='black', linestyle='--', markersize=10)
axs[1, 0].text(0.5, 0.95, format_equation(slope_semi_wet, intercept_semi_wet, r_value_semi_wet),
               transform=axs[1, 0].transAxes, fontsize=12, ha='center', va='top')

# Humid
axs[1, 1].plot(x, y_wet, color='black', linestyle='-', marker='o', markersize=10)
axs[1, 1].plot(x, y2_wet, color='black', linestyle='--', markersize=10)
axs[1, 1].text(0.5, 0.95, format_equation(slope_wet, intercept_wet, r_value_wet),
               transform=axs[1, 1].transAxes, fontsize=12, ha='center', va='top')

labels = ['(a)', '(b)', '(c)', '(d)']
for i, ax in enumerate(axs.flat):
    ax.text(0.05, 0.95, labels[i], transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top')
plt.tight_layout()
# plt.savefig(r'D:\BaiduSyncdisk\结果\照片\时间趋势.png', dpi=600, bbox_inches='tight')
plt.show()

趋势统计步骤4_2：AI_slope,30年时间窗口，不同气候区的趋势


In [412]:
print("Trend Statistics Step 5_1: Calculation of time variation trends for 20-year time window in different aridity zones")
# Extract based on masks, 3 indicators, 4 climate zones, 3 time windows, total 24 lines of code
AI_slope_20_025_dry = np.where(mask_dry, AI_slope_20_025, np.nan)  # Apply mask on corresponding two dimensions
AI_slope_20_025_semi_dry = np.where(mask_semi_dry, AI_slope_20_025, np.nan)
AI_slope_20_025_semi_wet = np.where(mask_semi_wet, AI_slope_20_025, np.nan)
AI_slope_20_025_wet = np.where(mask_wet, AI_slope_20_025, np.nan)

AI_slope_20_025_dry_point = np.nanmean(AI_slope_20_025_dry, axis=(1, 2))
AI_slope_20_025_semi_dry_point = np.nanmean(AI_slope_20_025_semi_dry, axis=(1, 2))
AI_slope_20_025_semi_wet_point = np.nanmean(AI_slope_20_025_semi_wet, axis=(1, 2))
AI_slope_20_025_wet_point = np.nanmean(AI_slope_20_025_wet, axis=(1, 2))

# AI_r
AI_r_20_025_dry = np.where(mask_dry, AI_r_20_025, np.nan)  # Apply mask on corresponding two dimensions
AI_r_20_025_semi_dry = np.where(mask_semi_dry, AI_r_20_025, np.nan)
AI_r_20_025_semi_wet = np.where(mask_semi_wet, AI_r_20_025, np.nan)
AI_r_20_025_wet = np.where(mask_wet, AI_r_20_025, np.nan)

AI_r_20_025_dry_point = np.nanmean(AI_r_20_025_dry, axis=(1, 2))
AI_r_20_025_semi_dry_point = np.nanmean(AI_r_20_025_semi_dry, axis=(1, 2))
AI_r_20_025_semi_wet_point = np.nanmean(AI_r_20_025_semi_wet, axis=(1, 2))
AI_r_20_025_wet_point = np.nanmean(AI_r_20_025_wet, axis=(1, 2))

# AI_extreme
AI_extreme_20_025_dry = np.where(mask_dry, AI_extreme_20_025, np.nan)  # Apply mask on corresponding two dimensions
AI_extreme_20_025_semi_dry = np.where(mask_semi_dry, AI_extreme_20_025, np.nan)
AI_extreme_20_025_semi_wet = np.where(mask_semi_wet, AI_extreme_20_025, np.nan)
AI_extreme_20_025_wet = np.where(mask_wet, AI_extreme_20_025, np.nan)

AI_extreme_20_025_dry_point = np.nanmean(AI_extreme_20_025_dry, axis=(1, 2))
AI_extreme_20_025_semi_dry_point = np.nanmean(AI_extreme_20_025_semi_dry, axis=(1, 2))
AI_extreme_20_025_semi_wet_point = np.nanmean(AI_extreme_20_025_semi_wet, axis=(1, 2))
AI_extreme_20_025_wet_point = np.nanmean(AI_extreme_20_025_wet, axis=(1, 2))

趋势统计步骤5_1：20年不同干燥区的时间变化趋势计算


In [6]:
print("Trend Statistics Step 5_2: 20-year time window, AI_slope trends in different climate zones")
x = generate_year_sequence(1992, 2013)
y_dry = AI_slope_20_025_dry_point
y_semi_dry = AI_slope_20_025_semi_dry_point
y_semi_wet = AI_slope_20_025_semi_wet_point
y_wet = AI_slope_20_025_wet_point


# Calculate linear regression
def fit_line(x, y):
    slope, intercept, r_value, p_value, _ = linregress(x, y)
    y_fit = slope * x + intercept
    return slope, intercept, r_value, p_value, y_fit


slope_dry, intercept_dry, r_dry, p_dry, y2_dry = fit_line(x, y_dry)
slope_semi_dry, intercept_semi_dry, r_semi_dry, p_semi_dry, y2_semi_dry = fit_line(x, y_semi_dry)
slope_semi_wet, intercept_semi_wet, r_semi_wet, p_semi_wet, y2_semi_wet = fit_line(x, y_semi_wet)
slope_wet, intercept_wet, r_wet, p_wet, y2_wet = fit_line(x, y_wet)

# Create single plot
plt.figure(figsize=(11, 6))

# Define colors and labels
colors = ['red', 'orange', 'green', 'blue']
labels = ['Arid', 'Semi-arid', 'Semi-humid', 'Humid']

# Plot original data points + trend lines
plt.plot(x, y_dry, 'o', color=colors[0], markersize=8, label=labels[0])
plt.plot(x, y2_dry, '--', color=colors[0], linewidth=2)

plt.plot(x, y_semi_dry, 's', color=colors[1], markersize=8, label=labels[1])
plt.plot(x, y2_semi_dry, '--', color=colors[1], linewidth=2)

plt.plot(x, y_semi_wet, '^', color=colors[2], markersize=8, label=labels[2])
plt.plot(x, y2_semi_wet, '--', color=colors[2], linewidth=2)

plt.plot(x, y_wet, 'D', color=colors[3], markersize=8, label=labels[3])
plt.plot(x, y2_wet, '--', color=colors[3], linewidth=2)

# Add equation and R²


def format_equation(slope, intercept, p_value):
    sign = '+' if intercept >= 0 else '-'
    return f'y = {slope:.3f}x {sign} {abs(intercept):.2f}, $p$={p_value:.2f}'


plt.text(0.02, 0.95, format_equation(slope_dry, intercept_dry, p_dry),
         transform=plt.gca().transAxes, color=colors[0], fontsize=14)
plt.text(0.02, 0.88, format_equation(slope_semi_dry, intercept_semi_dry, p_semi_dry),
         transform=plt.gca().transAxes, color=colors[1], fontsize=14)
plt.text(0.3, 0.95, format_equation(slope_semi_wet, intercept_semi_wet, p_semi_wet),
         transform=plt.gca().transAxes, color=colors[2], fontsize=14)
plt.text(0.3, 0.88, format_equation(slope_wet, intercept_wet, p_wet),
         transform=plt.gca().transAxes, color=colors[3], fontsize=14)
for spine in plt.gca().spines.values():
    spine.set_linewidth(2.8)  # Set border line width, unit is points, can be adjusted to your desired value

# Beautify the plot
plt.xlabel('Year', fontsize=27)
plt.ylabel('AI$_{\mathrm{S}}$ ', fontsize=27)
# plt.title('Trends of AI$_{\mathrm{S}}$ in Different Climate Zones', fontsize=14)
plt.tick_params(axis='both', labelsize=27)  # Set tick label font size
plt.legend(loc='lower right', fontsize=16)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig(r'D:\BaiduSyncdisk\毕业论文\英文论文的图\三图\图3_4', dpi=1500, bbox_inches='tight')
plt.show()

趋势统计步骤5_2：20年时间窗口，AI_slope不同气候区的趋势


In [5]:
print("Trend Statistics Step 5_2: 20-year time window, AI_r trends in different climate zones")
x = generate_year_sequence(1992, 2013)
y_dry = AI_r_20_025_dry_point
y_semi_dry = AI_r_20_025_semi_dry_point
y_semi_wet = AI_r_20_025_semi_wet_point
y_wet = AI_r_20_025_wet_point


# Calculate linear regression
def fit_line(x, y):
    slope, intercept, r_value, p_value, _ = linregress(x, y)
    y_fit = slope * x + intercept
    return slope, intercept, r_value, p_value, y_fit


slope_dry, intercept_dry, r_dry, p_dry, y2_dry = fit_line(x, y_dry)
slope_semi_dry, intercept_semi_dry, r_semi_dry, p_semi_dry, y2_semi_dry = fit_line(x, y_semi_dry)
slope_semi_wet, intercept_semi_wet, r_semi_wet, p_semi_wet, y2_semi_wet = fit_line(x, y_semi_wet)
slope_wet, intercept_wet, r_wet, p_wet, y2_wet = fit_line(x, y_wet)

# Create single plot
plt.figure(figsize=(11, 6))

# Define colors and labels
colors = ['red', 'orange', 'green', 'blue']
labels = ['Arid', 'Semi-arid', 'Semi-humid', 'Humid']

# Plot original data points + trend lines
plt.plot(x, y_dry, 'o', color=colors[0], markersize=8, label=labels[0])
plt.plot(x, y2_dry, '--', color=colors[0], linewidth=2)

plt.plot(x, y_semi_dry, 's', color=colors[1], markersize=8, label=labels[1])
plt.plot(x, y2_semi_dry, '--', color=colors[1], linewidth=2)

plt.plot(x, y_semi_wet, '^', color=colors[2], markersize=8, label=labels[2])
plt.plot(x, y2_semi_wet, '--', color=colors[2], linewidth=2)

plt.plot(x, y_wet, 'D', color=colors[3], markersize=8, label=labels[3])
plt.plot(x, y2_wet, '--', color=colors[3], linewidth=2)

# Add equation and p-value


def format_equation(slope, intercept, p_value):
    sign = '+' if intercept >= 0 else '-'
    return f'y = {slope:.3f}x {sign} {abs(intercept):.2f}, $p$={p_value:.2f}'


plt.text(0.02, 0.95, format_equation(slope_dry, intercept_dry, p_dry),
         transform=plt.gca().transAxes, color=colors[0], fontsize=14)
plt.text(0.02, 0.88, format_equation(slope_semi_dry, intercept_semi_dry, p_semi_dry),
         transform=plt.gca().transAxes, color=colors[1], fontsize=14)
plt.text(0.3, 0.95, format_equation(slope_semi_wet, intercept_semi_wet, p_semi_wet),
         transform=plt.gca().transAxes, color=colors[2], fontsize=14)
plt.text(0.3, 0.88, format_equation(slope_wet, intercept_wet, p_wet),
         transform=plt.gca().transAxes, color=colors[3], fontsize=14)
for spine in plt.gca().spines.values():
    spine.set_linewidth(2.8)  # Set border line width, unit is points, can be adjusted to your desired value

# Beautify the plot
plt.xlabel('Year', fontsize=27)
plt.ylabel('AI$_{\mathrm{R}}$ ', fontsize=27)
# plt.title('Trends of AI$_{\mathrm{S}}$ in Different Climate Zones', fontsize=14)
plt.tick_params(axis='both', labelsize=27)  # Set tick label font size
plt.legend(loc='lower right', fontsize=18)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig(r'D:\BaiduSyncdisk\毕业论文\英文论文的图\三图\图3_5', dpi=1500, bbox_inches='tight')
plt.show()

趋势统计步骤5_2：20年时间窗口，不同气候区AI_r的趋势


In [4]:
print("Trend Statistics Step 5_2: 20-year time window, AI_E trends in different climate zones")
x = generate_year_sequence(1992, 2013)
y_dry = AI_extreme_20_025_dry_point
y_semi_dry = AI_extreme_20_025_semi_dry_point
y_semi_wet = AI_extreme_20_025_semi_wet_point
y_wet = AI_extreme_20_025_wet_point


# Calculate linear regression
def fit_line(x, y):
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    y_fit = slope * x + intercept
    return slope, intercept, r_value, y_fit, p_value


slope_dry, intercept_dry, r_dry, y2_dry, p_dry = fit_line(x, y_dry)
slope_semi_dry, intercept_semi_dry, r_semi_dry, y2_semi_dry, p_semi_dry = fit_line(x, y_semi_dry)
slope_semi_wet, intercept_semi_wet, r_semi_wet, y2_semi_wet, p_semi_wet = fit_line(x, y_semi_wet)
slope_wet, intercept_wet, r_wet, y2_wet, p_wet = fit_line(x, y_wet)

# Create single plot
plt.figure(figsize=(11, 6))

# Define colors and labels
colors = ['red', 'orange', 'green', 'blue']
labels = ['Arid', 'Semi-arid', 'Semi-humid', 'Humid']

# Plot original data points + trend lines
plt.plot(x, y_dry, 'o', color=colors[0], markersize=8, label=labels[0])
plt.plot(x, y2_dry, '--', color=colors[0], linewidth=2)

plt.plot(x, y_semi_dry, 's', color=colors[1], markersize=8, label=labels[1])
plt.plot(x, y2_semi_dry, '--', color=colors[1], linewidth=2)

plt.plot(x, y_semi_wet, '^', color=colors[2], markersize=8, label=labels[2])
plt.plot(x, y2_semi_wet, '--', color=colors[2], linewidth=2)

plt.plot(x, y_wet, 'D', color=colors[3], markersize=8, label=labels[3])
plt.plot(x, y2_wet, '--', color=colors[3], linewidth=2)

# Add equation and p-value


def format_equation(slope, intercept, p_value):
    sign = '+' if intercept >= 0 else '-'
    return f'y = {slope:.3f}x {sign} {abs(intercept):.2f}, $p$={p_value:.2f}'


plt.text(0.02, 0.95, format_equation(slope_dry, intercept_dry, p_dry),
         transform=plt.gca().transAxes, color=colors[0], fontsize=14)
plt.text(0.02, 0.88, format_equation(slope_semi_dry, intercept_semi_dry, p_semi_dry),
         transform=plt.gca().transAxes, color=colors[1], fontsize=14)
plt.text(0.3, 0.95, format_equation(slope_semi_wet, intercept_semi_wet, p_semi_wet),
         transform=plt.gca().transAxes, color=colors[2], fontsize=14)
plt.text(0.3, 0.88, format_equation(slope_wet, intercept_wet, p_wet),
         transform=plt.gca().transAxes, color=colors[3], fontsize=14)
for spine in plt.gca().spines.values():
    spine.set_linewidth(2.8)  # Set border line width, unit is points, can be adjusted to your desired value

# Beautify the plot
plt.xlabel('Year', fontsize=27)
plt.ylabel('AI$_{\mathrm{E}}$ ', fontsize=27)
# plt.title('Trends of AI$_{\mathrm{S}}$ in Different Climate Zones', fontsize=14)
plt.tick_params(axis='both', labelsize=27)  # Set tick label font size
plt.legend(loc='lower right', fontsize=16)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig(r'D:\BaiduSyncdisk\毕业论文\英文论文的图\三图\图3_6', dpi=1500, bbox_inches='tight')
plt.show()

趋势统计步骤5_2：20年时间窗口，不同气候区AI_e的趋势
